<a href="https://colab.research.google.com/github/mrfriman666/mrfriman666/blob/main/Nissan_Logger_v5_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 📦 Ячейка 1/5: Окружение (~10 мин)
import os

print("=" * 60)
print("📥 Системные пакеты + Java 17")
print("=" * 60)
!apt-get update -qq
!apt-get install -y -qq curl git unzip xz-utils zip libglu1-mesa openjdk-17-jdk-headless ninja-build cmake > /dev/null

# Java 17 (Java 21 несовместима с compileSdk 36)
!update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java 2>&1 | tail -2

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ['PATH']
!java -version 2>&1 | head -1

print("\n📥 Flutter SDK")
!git clone https://github.com/flutter/flutter.git -b stable --depth 1 /content/flutter 2>/dev/null

os.environ['PATH'] = '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:' + os.environ['PATH']
os.environ['PUB_CACHE'] = '/content/.pub-cache'
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/ninja'

!flutter config --no-analytics --no-cli-animations 2>/dev/null
!flutter --disable-telemetry 2>/dev/null

print("\n📥 Android SDK + NDK 28")
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmdline-tools.zip
!mkdir -p /content/android-sdk/cmdline-tools
!unzip -q /tmp/cmdline-tools.zip -d /content/android-sdk/cmdline-tools
!mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest 2>/dev/null || true

os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['PATH'] = '/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:' + os.environ['PATH']

!yes | sdkmanager --licenses > /dev/null 2>&1
!sdkmanager "platform-tools" "platforms;android-36" "build-tools;36.0.0" "ndk;28.2.13676358" > /dev/null 2>&1

!flutter config --android-sdk /content/android-sdk 2>/dev/null
!flutter precache --android 2>/dev/null

print("\n✅ Готово!")
!flutter --version

📥 Системные пакеты + Java 17
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/java to provide /usr/bin/java (java) in manual mode
openjdk version "17.0.19" 2026-04-21

📥 Flutter SDK
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  227M  100  227M    0     0  70.6M      0  0:00:03  0:00:03 --:--:-- 70.5M
Analytics reporting disabled.
Setting "cli-animations" value to "false".

You may need to restart any open editors for them to read new settings.

📥 Android SDK + NDK 28
Setting "android-sdk" value to "/content/android-sdk".

You may need to restart any open editors for them to read new settings.
[1/12] Material Fonts                                              450ms
[2/1

In [2]:
# @title 🏗️ Ячейка 2/5: Проект + Android + Модели (~30 сек)
import os

os.chdir('/content')
!rm -rf /content/nissan_logger_pro_v5
!flutter create --org com.nissanlogger --project-name nissan_logger_pro_v5 nissan_logger_pro_v5
os.chdir('/content/nissan_logger_pro_v5')

for folder in ['models', 'services', 'screens', 'widgets']:
    os.makedirs(f'/content/nissan_logger_pro_v5/lib/{folder}', exist_ok=True)

# ============ pubspec.yaml ============
with open('pubspec.yaml', 'w') as f:
    f.write('''name: nissan_logger_pro_v5
description: Nissan X-Trail T30 QR20DE Tuning Logger v5
version: 5.0.0+1
publish_to: 'none'

environment:
  sdk: '>=3.0.0 <4.0.0'
  flutter: ">=3.10.0"

dependencies:
  flutter:
    sdk: flutter
  cupertino_icons: ^1.0.6
  fl_chart: 0.68.0
  flutter_bluetooth_serial: 0.4.0
  permission_handler: 11.3.1
  path_provider: 2.1.4
  path: ^1.9.0
  csv: 6.0.0
  shared_preferences: 2.3.2
  file_picker: 8.1.2
  share_plus: 10.0.2
  intl: ^0.19.0
  uuid: 4.5.0
  vibration: 2.0.0
  math_expressions: 2.5.0

dev_dependencies:
  flutter_test:
    sdk: flutter
  flutter_lints: ^4.0.0

flutter:
  uses-material-design: true
''')
print("✅ pubspec.yaml")

# ============ AndroidManifest.xml ============
with open('android/app/src/main/AndroidManifest.xml', 'w') as f:
    f.write('''<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-permission android:name="android.permission.BLUETOOTH" />
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" />
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" android:usesPermissionFlags="neverForLocation" />
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT" />
    <uses-permission android:name="android.permission.ACCESS_FINE_LOCATION" />
    <uses-permission android:name="android.permission.ACCESS_COARSE_LOCATION" />
    <uses-permission android:name="android.permission.WRITE_EXTERNAL_STORAGE" android:maxSdkVersion="28" />
    <uses-permission android:name="android.permission.READ_EXTERNAL_STORAGE" />
    <uses-permission android:name="android.permission.VIBRATE" />
    <uses-permission android:name="android.permission.WAKE_LOCK" />
    <uses-permission android:name="android.permission.INTERNET" />

    <application android:label="Nissan Logger Pro v5" android:name="${applicationName}"
        android:icon="@mipmap/ic_launcher" android:usesCleartextTraffic="true"
        android:requestLegacyExternalStorage="true" android:allowBackup="true">
        <activity android:name=".MainActivity" android:exported="true" android:launchMode="singleTop"
            android:theme="@style/LaunchTheme"
            android:configChanges="orientation|keyboardHidden|keyboard|screenSize|smallestScreenSize|locale|layoutDirection|fontScale|screenLayout|density|uiMode"
            android:hardwareAccelerated="true" android:windowSoftInputMode="adjustResize">
            <meta-data android:name="io.flutter.embedding.android.NormalTheme" android:resource="@style/NormalTheme" />
            <intent-filter>
                <action android:name="android.intent.action.MAIN"/>
                <category android:name="android.intent.category.LAUNCHER"/>
            </intent-filter>
        </activity>
        <meta-data android:name="flutterEmbedding" android:value="2" />
    </application>
</manifest>
''')
print("✅ AndroidManifest.xml")

# ============ MainActivity.kt ============
main_activity_dir = 'android/app/src/main/kotlin/com/nissanlogger/pro_v5'
os.makedirs(main_activity_dir, exist_ok=True)
!rm -rf android/app/src/main/java

with open(f'{main_activity_dir}/MainActivity.kt', 'w') as f:
    f.write('''package com.nissanlogger.pro_v5
import android.os.Bundle
import android.view.WindowManager
import io.flutter.embedding.android.FlutterActivity
class MainActivity: FlutterActivity() {
    override fun onCreate(savedInstanceState: Bundle?) {
        super.onCreate(savedInstanceState)
        window.addFlags(WindowManager.LayoutParams.FLAG_KEEP_SCREEN_ON)
    }
}
''')
print("✅ MainActivity.kt")

# ============ Gradle ============
with open('android/gradle/wrapper/gradle-wrapper.properties', 'w') as f:
    f.write('''distributionBase=GRADLE_USER_HOME
distributionPath=wrapper/dists
zipStoreBase=GRADLE_USER_HOME
zipStorePath=wrapper/dists
distributionUrl=https\\://services.gradle.org/distributions/gradle-8.10-all.zip
''')

with open('android/settings.gradle.kts', 'w') as f:
    f.write('''pluginManagement {
    val flutterSdkPath = run {
        val properties = java.util.Properties()
        file("local.properties").inputStream().use { properties.load(it) }
        val flutterSdkPath = properties.getProperty("flutter.sdk")
        require(flutterSdkPath != null) { "flutter.sdk not set in local.properties" }
        flutterSdkPath
    }
    includeBuild("$flutterSdkPath/packages/flutter_tools/gradle")
    repositories { google(); mavenCentral(); gradlePluginPortal() }
}
plugins {
    id("dev.flutter.flutter-plugin-loader") version "1.0.0"
    id("com.android.application") version "8.6.0" apply false
    id("org.jetbrains.kotlin.android") version "1.9.24" apply false
}
include(":app")
''')

with open('android/build.gradle.kts', 'w') as f:
    f.write('''allprojects {
    repositories { google(); mavenCentral() }
    configurations.all {
        resolutionStrategy {
            force("androidx.core:core:1.13.1")
            force("androidx.core:core-ktx:1.13.1")
            force("androidx.appcompat:appcompat:1.7.0")
            force("androidx.annotation:annotation:1.8.2")
        }
    }
}
val newBuildDir: Directory = rootProject.layout.buildDirectory.dir("../../build").get()
rootProject.layout.buildDirectory.value(newBuildDir)
subprojects {
    val newSubprojectBuildDir: Directory = newBuildDir.dir(project.name)
    project.layout.buildDirectory.value(newSubprojectBuildDir)
}
subprojects { project.evaluationDependsOn(":app") }
tasks.register<Delete>("clean") { delete(rootProject.layout.buildDirectory) }
''')

with open('android/app/build.gradle.kts', 'w') as f:
    f.write('''plugins {
    id("com.android.application")
    id("kotlin-android")
    id("dev.flutter.flutter-gradle-plugin")
}
android {
    namespace = "com.nissanlogger.pro_v5"
    compileSdk = 36
    ndkVersion = "28.2.13676358"
    compileOptions {
        sourceCompatibility = JavaVersion.VERSION_17
        targetCompatibility = JavaVersion.VERSION_17
    }
    kotlinOptions { jvmTarget = JavaVersion.VERSION_17.toString() }
    defaultConfig {
        applicationId = "com.nissanlogger.pro_v5"
        minSdk = 21
        targetSdk = 34
        versionCode = 5
        versionName = "5.0.0"
        multiDexEnabled = true
    }
    buildTypes {
        release {
            signingConfig = signingConfigs.getByName("debug")
            isMinifyEnabled = false
            isShrinkResources = false
        }
    }
}
dependencies {
    implementation("androidx.core:core:1.13.1")
    implementation("androidx.core:core-ktx:1.13.1")
    implementation("androidx.appcompat:appcompat:1.7.0")
    implementation("androidx.multidex:multidex:2.0.1")
}
flutter { source = "../.." }
''')

with open('android/gradle.properties', 'w') as f:
    f.write('''org.gradle.jvmargs=-Xmx4G -XX:+UseParallelGC -XX:MaxMetaspaceSize=2G
android.useAndroidX=true
android.enableJetifier=true
android.nonTransitiveRClass=false
kotlin.code.style=official
org.gradle.parallel=true
org.gradle.caching=false
org.gradle.configuration-cache=false
kotlin.jvm.target.validation.mode=warning
android.suppressUnsupportedCompileSdk=36
''')
print("✅ Gradle конфиги")

# ============ lib/constants.dart ============
with open('lib/constants.dart', 'w') as f:
    f.write('''class AppConstants {
  static const String appVersion = '5.0.0';
  static const String appName = 'Nissan Logger Pro v5';
  static const double engineDisplacement = 2.0;
  static const double knockRetardWarning = 1.0;
  static const double knockRetardDanger = 3.0;
  static const double afrLeanWarning = 15.5;
  static const double afrLeanDanger = 16.5;
  static const double afrRichWarning = 11.5;
  static const int coolantTempWarning = 100;
  static const int coolantTempDanger = 110;
  static const double fuelTrimWarning = 12.0;
  static const double fuelTrimDanger = 20.0;
  static const int autoLogRpmThreshold = 1500;
  static const int autoLogSpeedThreshold = 5;
  static const int autoLogIdleTimeoutSec = 30;
  static const int defaultPollingInterval = 50;
  static const double gasolineDensity = 745.0;
  static const double stoichiometricAFR = 14.7;
}
''')

# ============ lib/models/obd_data.dart ============
with open('lib/models/obd_data.dart', 'w') as f:
    f.write('''import '../constants.dart';

class OBDData {
  final DateTime timestamp;
  final int rpm;
  final int speed;
  final double engineLoad;
  final int coolantTemp;
  final int intakeTemp;
  final double maf;
  final double throttlePos;
  final double ignitionTiming;
  final double shortFuelTrim;
  final double longFuelTrim;
  final double o2Voltage;
  final double afr;
  final double vtcTargetAngle;
  final double vtcActualAngle;
  final double knockRetard;
  final int knockCount;
  final double actualIgnition;
  final double injectorDuty;
  final double injectorPulseWidth;
  final double requestedTorque;
  final double actualTorque;
  final double oilTemp;
  final double afrTarget;
  final double lambda;
  final double manifoldPressure;
  final double acceleratorPedal;
  final double throttleActual;
  final double batteryVoltage;
  final double engineDisplacement;
  final double tripFuelL;

  OBDData({
    required this.timestamp,
    this.rpm = 0, this.speed = 0, this.engineLoad = 0,
    this.coolantTemp = 0, this.intakeTemp = 0, this.maf = 0,
    this.throttlePos = 0, this.ignitionTiming = 0,
    this.shortFuelTrim = 0, this.longFuelTrim = 0,
    this.o2Voltage = 0, this.afr = 14.7,
    this.vtcTargetAngle = 0, this.vtcActualAngle = 0,
    this.knockRetard = 0, this.knockCount = 0,
    this.actualIgnition = 0, this.injectorDuty = 0,
    this.injectorPulseWidth = 0,
    this.requestedTorque = 0, this.actualTorque = 0,
    this.oilTemp = 0, this.afrTarget = 14.7,
    this.lambda = 1.0, this.manifoldPressure = 0,
    this.acceleratorPedal = 0, this.throttleActual = 0,
    this.batteryVoltage = 0,
    this.engineDisplacement = 2.0,
    this.tripFuelL = 0,
  });

  double get calculatedHP {
    if (maf <= 0 || rpm <= 0) return 0;
    return maf * 0.8;
  }
  double get calculatedTorque {
    if (calculatedHP <= 0 || rpm <= 0) return 0;
    return (calculatedHP * 7127) / rpm;
  }
  double get volumetricEfficiency {
    if (rpm <= 0 || maf <= 0) return 0;
    double theoretical = rpm * engineDisplacement * 1.184 / 120;
    if (theoretical <= 0) return 0;
    return (maf / theoretical * 100).clamp(0, 150);
  }
  double get fuelMassFlow {
    if (maf <= 0 || afr <= 0) return 0;
    return maf / afr;
  }
  double get fuelFlowLph => fuelMassFlow * 3600 / AppConstants.gasolineDensity;
  double get fuelL100km {
    if (speed < 5) return 0;
    return fuelFlowLph / speed * 100;
  }
  double get totalFuelTrim => shortFuelTrim + longFuelTrim;
  double get vtcError => (vtcTargetAngle - vtcActualAngle).abs();
  String get engineMode {
    if (rpm < 100) return 'STOP';
    if (rpm < 900 && throttlePos < 5) return 'IDLE';
    if (throttlePos > 80) return 'WOT';
    if (throttlePos < 10 && speed > 0) return 'COAST';
    return 'CRUISE';
  }

  List<dynamic> toCsvRow() {
    return [
      timestamp.millisecondsSinceEpoch,
      rpm, speed, engineLoad.toStringAsFixed(2),
      coolantTemp, intakeTemp,
      maf.toStringAsFixed(3), throttlePos.toStringAsFixed(2),
      ignitionTiming.toStringAsFixed(2),
      shortFuelTrim.toStringAsFixed(2), longFuelTrim.toStringAsFixed(2),
      o2Voltage.toStringAsFixed(4), afr.toStringAsFixed(3),
      vtcTargetAngle.toStringAsFixed(2), vtcActualAngle.toStringAsFixed(2),
      knockRetard.toStringAsFixed(2), knockCount,
      actualIgnition.toStringAsFixed(2), injectorDuty.toStringAsFixed(2),
      requestedTorque.toStringAsFixed(2), actualTorque.toStringAsFixed(2),
      oilTemp.toStringAsFixed(1),
      afrTarget.toStringAsFixed(3), lambda.toStringAsFixed(4),
      manifoldPressure.toStringAsFixed(2),
      acceleratorPedal.toStringAsFixed(2), throttleActual.toStringAsFixed(2),
      calculatedHP.toStringAsFixed(2), calculatedTorque.toStringAsFixed(2),
      batteryVoltage.toStringAsFixed(2),
      volumetricEfficiency.toStringAsFixed(1),
      fuelFlowLph.toStringAsFixed(3),
      fuelL100km.toStringAsFixed(2),
      tripFuelL.toStringAsFixed(3),
    ];
  }

  static List<String> csvHeaders() {
    return [
      'Timestamp_ms', 'RPM', 'Speed_kmh', 'EngineLoad_pct',
      'CoolantTemp_C', 'IntakeTemp_C', 'MAF_gs', 'ThrottlePos_pct',
      'IgnitionTiming_deg', 'STFT_pct', 'LTFT_pct', 'O2Voltage_V', 'AFR',
      'VTC_Target_deg', 'VTC_Actual_deg', 'KnockRetard_deg', 'KnockCount',
      'ActualIgnition_deg', 'InjectorDuty_pct',
      'RequestedTorque_Nm', 'ActualTorque_Nm', 'OilTemp_C',
      'AFR_Target', 'Lambda', 'ManifoldPressure_kPa',
      'AcceleratorPedal_pct', 'ThrottleActual_pct',
      'EstimatedHP', 'EstimatedTorque_Nm',
      'BatteryVoltage_V', 'VE_pct',
      'FuelFlow_Lph', 'FuelConsumption_L100km', 'TripFuel_L',
    ];
  }
}
''')

# ============ lib/models/tuning_map.dart ============
with open('lib/models/tuning_map.dart', 'w') as f:
    f.write('''class TuningMap {
  final String name;
  final String address;
  final int rows;
  final int cols;
  final List<double> rpmAxis;
  final List<double> loadAxis;
  List<List<double>> data;
  final String units;
  final double minValue;
  final double maxValue;

  TuningMap({
    required this.name, required this.address,
    required this.rows, required this.cols,
    required this.rpmAxis, required this.loadAxis,
    required this.data, required this.units,
    this.minValue = -100, this.maxValue = 400,
  });

  double getValue(double rpm, double load) {
    int rpmIdx = _findClosestIndex(rpmAxis, rpm);
    int loadIdx = _findClosestIndex(loadAxis, load);
    return data[rpmIdx][loadIdx];
  }
  void setValue(double rpm, double load, double value) {
    int rpmIdx = _findClosestIndex(rpmAxis, rpm);
    int loadIdx = _findClosestIndex(loadAxis, load);
    data[rpmIdx][loadIdx] = value;
  }
  int _findClosestIndex(List<double> axis, double value) {
    int idx = 0;
    double minDiff = double.infinity;
    for (int i = 0; i < axis.length; i++) {
      double diff = (axis[i] - value).abs();
      if (diff < minDiff) { minDiff = diff; idx = i; }
    }
    return idx;
  }
  TuningMap copy() {
    return TuningMap(
      name: name, address: address, rows: rows, cols: cols,
      rpmAxis: List.from(rpmAxis), loadAxis: List.from(loadAxis),
      data: data.map((row) => List<double>.from(row)).toList(),
      units: units, minValue: minValue, maxValue: maxValue,
    );
  }
  double get avgValue {
    double sum = 0; int count = 0;
    for (var row in data) { for (var v in row) { sum += v; count++; } }
    return count > 0 ? sum / count : 0;
  }
  Map<String, dynamic> toJson() => {
    'name': name, 'address': address, 'rows': rows, 'cols': cols,
    'rpmAxis': rpmAxis, 'loadAxis': loadAxis, 'data': data,
    'units': units, 'minValue': minValue, 'maxValue': maxValue,
  };
}
''')

# ============ lib/models/analysis_result.dart ============
with open('lib/models/analysis_result.dart', 'w') as f:
    f.write('''class MapCell {
  final int rpmIndex;
  final int loadIndex;
  final double rpm;
  final double load;
  final double currentValue;
  final double suggestedValue;
  final double confidence;
  final int sampleCount;
  final String reason;
  MapCell({
    required this.rpmIndex, required this.loadIndex,
    required this.rpm, required this.load,
    required this.currentValue, required this.suggestedValue,
    required this.confidence, required this.sampleCount,
    required this.reason,
  });
  double get delta => suggestedValue - currentValue;
  double get deltaPercent => currentValue != 0 ? (delta / currentValue) * 100 : 0;
}

class AnalysisResult {
  final String mapName;
  final DateTime analyzedAt;
  final int totalSamples;
  final List<MapCell> changes;
  final String summary;
  AnalysisResult({
    required this.mapName, required this.analyzedAt,
    required this.totalSamples, required this.changes,
    required this.summary,
  });
}
''')

# ============ lib/models/dtc_code.dart ============
with open('lib/models/dtc_code.dart', 'w') as f:
    f.write('''class DTCCode {
  final String code;
  final String description;
  final DTCType type;
  final bool isPending;
  DTCCode({
    required this.code, required this.description,
    required this.type, this.isPending = false,
  });
}
enum DTCType { powertrain, chassis, body, network }
''')

# ============ lib/models/alert.dart (со snapshot!) ============
with open('lib/models/alert.dart', 'w') as f:
    f.write('''enum AlertLevel { info, warning, danger }

class AlertSnapshot {
  final int rpm;
  final int speed;
  final double engineLoad;
  final int coolantTemp;
  final int intakeTemp;
  final double maf;
  final double throttlePos;
  final double afr;
  final double knockRetard;
  final double shortFuelTrim;
  final double longFuelTrim;
  final double ignitionTiming;
  final double vtcActualAngle;

  AlertSnapshot({
    required this.rpm, required this.speed, required this.engineLoad,
    required this.coolantTemp, required this.intakeTemp,
    required this.maf, required this.throttlePos, required this.afr,
    required this.knockRetard,
    required this.shortFuelTrim, required this.longFuelTrim,
    required this.ignitionTiming, required this.vtcActualAngle,
  });
}

class Alert {
  final String message;
  final AlertLevel level;
  final DateTime timestamp;
  final String? category;
  final AlertSnapshot? snapshot;
  final String? explanation;

  Alert({
    required this.message, required this.level, required this.timestamp,
    this.category, this.snapshot, this.explanation,
  });
}
''')

# ============ lib/models/vehicle_profile.dart ============
with open('lib/models/vehicle_profile.dart', 'w') as f:
    f.write('''import 'dart:convert';
class VehicleProfile {
  final String id;
  final String name;
  final String make;
  final String model;
  final String year;
  final String engine;
  final double displacement;
  final String ecuFirmware;
  final DateTime createdAt;
  VehicleProfile({
    required this.id, required this.name, required this.make,
    required this.model, required this.year, required this.engine,
    this.displacement = 2.0, this.ecuFirmware = '',
    required this.createdAt,
  });
  Map<String, dynamic> toJson() => {
    'id': id, 'name': name, 'make': make, 'model': model,
    'year': year, 'engine': engine, 'displacement': displacement,
    'ecuFirmware': ecuFirmware, 'createdAt': createdAt.toIso8601String(),
  };
  factory VehicleProfile.fromJson(Map<String, dynamic> json) {
    return VehicleProfile(
      id: json['id'], name: json['name'], make: json['make'],
      model: json['model'], year: json['year'], engine: json['engine'],
      displacement: (json['displacement'] ?? 2.0).toDouble(),
      ecuFirmware: json['ecuFirmware'] ?? '',
      createdAt: DateTime.parse(json['createdAt']),
    );
  }
  String toJsonString() => jsonEncode(toJson());
  factory VehicleProfile.fromJsonString(String s) => VehicleProfile.fromJson(jsonDecode(s));
}
''')

# ============ lib/models/custom_map_def.dart ============
with open('lib/models/custom_map_def.dart', 'w') as f:
    f.write('''import 'dart:convert';
class CustomMapDef {
  final String id;
  final String name;
  final int address;
  final int rows;
  final int cols;
  final bool isUInt16;
  final String formula;
  final String units;
  final List<double>? xAxis;
  final List<double>? yAxis;
  final DateTime createdAt;
  CustomMapDef({
    required this.id, required this.name, required this.address,
    required this.rows, required this.cols, this.isUInt16 = true,
    required this.formula, this.units = '',
    this.xAxis, this.yAxis, required this.createdAt,
  });
  int get bytesPerCell => isUInt16 ? 2 : 1;
  int get totalBytes => rows * cols * bytesPerCell;
  String get addressHex => '0x' + address.toRadixString(16).toUpperCase().padLeft(4, '0');
  Map<String, dynamic> toJson() => {
    'id': id, 'name': name, 'address': address,
    'rows': rows, 'cols': cols, 'isUInt16': isUInt16,
    'formula': formula, 'units': units,
    'xAxis': xAxis, 'yAxis': yAxis,
    'createdAt': createdAt.toIso8601String(),
  };
  factory CustomMapDef.fromJson(Map<String, dynamic> json) {
    return CustomMapDef(
      id: json['id'], name: json['name'], address: json['address'],
      rows: json['rows'], cols: json['cols'],
      isUInt16: json['isUInt16'] ?? true,
      formula: json['formula'], units: json['units'] ?? '',
      xAxis: json['xAxis'] != null
          ? List<double>.from(json['xAxis'].map((e) => (e as num).toDouble()))
          : null,
      yAxis: json['yAxis'] != null
          ? List<double>.from(json['yAxis'].map((e) => (e as num).toDouble()))
          : null,
      createdAt: DateTime.parse(json['createdAt']),
    );
  }
  String toJsonString() => jsonEncode(toJson());
  factory CustomMapDef.fromJsonString(String s) => CustomMapDef.fromJson(jsonDecode(s));
}

class EcuMapReadResult {
  final CustomMapDef def;
  final List<List<double>> data;
  final DateTime readAt;
  final String rawHex;
  EcuMapReadResult({
    required this.def, required this.data,
    required this.readAt, this.rawHex = '',
  });
}
''')

print("✅ Все модели созданы")
print()
print("=" * 60)
print("✅ Ячейка 2/5 готова!")
print("=" * 60)
!ls lib/models/

Creating project nissan_logger_pro_v5...
Resolving dependencies in `nissan_logger_pro_v5`...
Got dependencies in `nissan_logger_pro_v5`.
Wrote 131 files.

All done!
You can find general documentation for Flutter at: https://docs.flutter.dev/
Detailed API documentation is available at: https://api.flutter.dev/
If you prefer video documentation, consider: https://www.youtube.com/c/flutterdev

In order to run your application, type:

  $ cd nissan_logger_pro_v5
  $ flutter run

Your application code is in nissan_logger_pro_v5/lib/main.dart.

✅ pubspec.yaml
✅ AndroidManifest.xml
✅ MainActivity.kt
✅ Gradle конфиги
✅ Все модели созданы

✅ Ячейка 2/5 готова!
alert.dart	      dtc_code.dart    vehicle_profile.dart
analysis_result.dart  obd_data.dart
custom_map_def.dart   tuning_map.dart


In [11]:
# @title 🔧 Ячейка 3/5: Все сервисы (со всеми фиксами v5)
import os
os.chdir('/content/nissan_logger_pro_v5')

# ============ nissan_pid_library.dart ============
with open('lib/services/nissan_pid_library.dart', 'w') as f:
    f.write('''class NissanPidDef {
  final String cmd;
  final String answer;
  final String name;
  final String desc;
  final String unit;
  final int bytesCount;
  final double Function(List<int>) formula;
  final double minVal;
  final double maxVal;
  final int priority;
  final String category;

  NissanPidDef({
    required this.cmd, required this.answer,
    required this.name, required this.desc,
    required this.unit, required this.bytesCount,
    required this.formula,
    this.minVal = 0, this.maxVal = 255,
    this.priority = 3, this.category = 'other',
  });
}

class NissanPidLibrary {
  static final List<NissanPidDef> all = [
    NissanPidDef(cmd: '2212010401', answer: '621201', name: 'RPM',
        desc: 'Обороты', unit: 'RPM', bytesCount: 2,
        priority: 1, category: 'engine', minVal: 0, maxVal: 8000,
        formula: (b) => (b[0] * 256 + b[1]) * 12.5),
    NissanPidDef(cmd: '22110A0401', answer: '62110A', name: 'TIMING',
        desc: 'УОЗ факт', unit: '°BTDC', bytesCount: 1,
        priority: 1, category: 'ignition', minVal: -20, maxVal: 60,
        formula: (b) => (110 - b[0]).toDouble()),
    NissanPidDef(cmd: '22112D0401', answer: '62112D', name: 'KNOCK',
        desc: 'Корр.УОЗ', unit: '°', bytesCount: 1,
        priority: 1, category: 'ignition', minVal: -30, maxVal: 30,
        formula: (b) { int v = b[0]; if (v >= 128) v -= 256; return v.toDouble(); }),
    NissanPidDef(cmd: '22111E0401', answer: '62111E', name: 'TPS',
        desc: 'Дроссель', unit: '%', bytesCount: 1,
        priority: 1, category: 'throttle', minVal: 0, maxVal: 100,
        formula: (b) => b[0] * 0.35),
    NissanPidDef(cmd: '2212090401', answer: '621209', name: 'MAF',
        desc: 'MAF', unit: 'g/s', bytesCount: 2,
        priority: 1, category: 'air', minVal: 0, maxVal: 500,
        formula: (b) => (b[0] * 256 + b[1]) * 0.01),
    NissanPidDef(cmd: '2211010401', answer: '621101', name: 'ECT',
        desc: 'ОЖ', unit: '°C', bytesCount: 1,
        priority: 1, category: 'temp', minVal: -30, maxVal: 130,
        formula: (b) => (b[0] - 50).toDouble()),
    NissanPidDef(cmd: '2211170401', answer: '621117', name: 'LOAD',
        desc: 'Нагрузка', unit: '%', bytesCount: 1,
        priority: 1, category: 'engine', minVal: 0, maxVal: 100,
        formula: (b) => b[0] * 100.0 / 256.0),
    NissanPidDef(cmd: '2211020401', answer: '621102', name: 'SPEED',
        desc: 'Скорость', unit: 'км/ч', bytesCount: 1,
        priority: 1, category: 'engine', minVal: 0, maxVal: 200,
        formula: (b) => b[0] * 2.0),
    NissanPidDef(cmd: '2211350401', answer: '621135', name: 'VTC_ACTUAL',
        desc: 'VTC факт B1', unit: '°CA', bytesCount: 1,
        priority: 1, category: 'vtc', minVal: -10, maxVal: 50,
        formula: (b) => b[0] * 0.5 - 64),
    NissanPidDef(cmd: '2211230401', answer: '621123', name: 'STFT',
        desc: 'STFT B1', unit: '%', bytesCount: 1,
        priority: 1, category: 'fuel', minVal: -100, maxVal: 100,
        formula: (b) => (b[0] - 100).toDouble()),
    NissanPidDef(cmd: '2211250401', answer: '621125', name: 'LTFT',
        desc: 'LTFT B1', unit: '%', bytesCount: 1,
        priority: 1, category: 'fuel', minVal: -100, maxVal: 100,
        formula: (b) => (b[0] - 100).toDouble()),
    NissanPidDef(cmd: '2212060401', answer: '621206', name: 'INJ_B1',
        desc: 'Впрыск B1', unit: 'ms', bytesCount: 2,
        priority: 1, category: 'fuel', minVal: 0, maxVal: 30,
        formula: (b) => (b[0] * 256 + b[1]) * 0.01),
    NissanPidDef(cmd: '2211180401', answer: '621118', name: 'O2_B1S1',
        desc: 'O2 B1S1', unit: 'V', bytesCount: 1,
        priority: 1, category: 'fuel', minVal: 0, maxVal: 1,
        formula: (b) => b[0] * 0.01),
    NissanPidDef(cmd: '22117C0401', answer: '62117C', name: 'PEDAL',
        desc: 'Педаль газа', unit: '%', bytesCount: 1,
        priority: 2, category: 'throttle', formula: (b) => b[0] * 0.5),
    NissanPidDef(cmd: '2211030401', answer: '621103', name: 'BATT',
        desc: 'Напряжение', unit: 'V', bytesCount: 1,
        priority: 2, category: 'electric', minVal: 8, maxVal: 16,
        formula: (b) => b[0] * 0.08),
    NissanPidDef(cmd: '2211060401', answer: '621106', name: 'IAT',
        desc: 'Впуск', unit: '°C', bytesCount: 1,
        priority: 2, category: 'temp', minVal: -30, maxVal: 100,
        formula: (b) => (b[0] - 50).toDouble()),
    NissanPidDef(cmd: '22112A0401', answer: '62112A', name: 'MAP_V',
        desc: 'MAP', unit: 'V', bytesCount: 1,
        priority: 2, category: 'air', formula: (b) => b[0] * 0.02),
    NissanPidDef(cmd: '22110B0401', answer: '62110B', name: 'IACV',
        desc: 'Клапан ХХ', unit: '%', bytesCount: 1,
        priority: 2, category: 'idle', formula: (b) => b[0] * 0.5),
    NissanPidDef(cmd: '22110D0401', answer: '62110D', name: 'IDLE_BASE',
        desc: 'Базовые ХХ', unit: 'RPM', bytesCount: 1,
        priority: 2, category: 'idle', maxVal: 3200,
        formula: (b) => b[0] * 12.5),
    NissanPidDef(cmd: '2212080401', answer: '621208', name: 'INJ_BASE',
        desc: 'Впрыск баз', unit: 'ms', bytesCount: 2,
        priority: 2, category: 'fuel',
        formula: (b) => (b[0] * 256 + b[1]) / 2048.0),
    NissanPidDef(cmd: '2211380401', answer: '621138', name: 'VTC_SOL',
        desc: 'VTC Sol B1', unit: '%', bytesCount: 1,
        priority: 2, category: 'vtc',
        formula: (b) => b[0] * 100.0 / 256.0),
    NissanPidDef(cmd: '2211240401', answer: '621124', name: 'STFT_B2',
        desc: 'STFT B2', unit: '%', bytesCount: 1,
        priority: 2, category: 'fuel', minVal: -100, maxVal: 100,
        formula: (b) => (b[0] - 100).toDouble()),
    NissanPidDef(cmd: '2211260401', answer: '621126', name: 'LTFT_B2',
        desc: 'LTFT B2', unit: '%', bytesCount: 1,
        priority: 2, category: 'fuel', minVal: -100, maxVal: 100,
        formula: (b) => (b[0] - 100).toDouble()),
    NissanPidDef(cmd: '22111F0401', answer: '62111F', name: 'OIL_TEMP',
        desc: 'Темп. масла', unit: '°C', bytesCount: 1,
        priority: 3, category: 'temp', formula: (b) => (b[0] - 50).toDouble()),
    NissanPidDef(cmd: '2211040401', answer: '621104', name: 'FUEL_TEMP',
        desc: 'Темп. топлива', unit: '°C', bytesCount: 1,
        priority: 3, category: 'temp', formula: (b) => (b[0] - 50).toDouble()),
    NissanPidDef(cmd: '22114B0401', answer: '62114B', name: 'RAD_TEMP',
        desc: 'Темп. радиатора', unit: '°C', bytesCount: 1,
        priority: 3, category: 'temp', formula: (b) => (b[0] - 50).toDouble()),
    NissanPidDef(cmd: '2211190401', answer: '621119', name: 'O2_B2S1',
        desc: 'O2 B2S1', unit: 'V', bytesCount: 1,
        priority: 3, category: 'fuel', formula: (b) => b[0] * 0.01),
    NissanPidDef(cmd: '22111A0401', answer: '62111A', name: 'O2_B1S2',
        desc: 'O2 B1S2', unit: 'V', bytesCount: 1,
        priority: 3, category: 'fuel', formula: (b) => b[0] * 0.01),
    NissanPidDef(cmd: '22111B0401', answer: '62111B', name: 'O2_B2S2',
        desc: 'O2 B2S2', unit: 'V', bytesCount: 1,
        priority: 3, category: 'fuel', formula: (b) => b[0] * 0.01),
    NissanPidDef(cmd: '2212250401', answer: '621225', name: 'AF_B1S1',
        desc: 'A/F B1S1', unit: 'V', bytesCount: 2,
        priority: 3, category: 'fuel',
        formula: (b) => (b[0] * 256 + b[1]) * 0.005),
    NissanPidDef(cmd: '22122D0401', answer: '62122D', name: 'VTC_DUTY_IN_B1',
        desc: 'VTC Duty B1', unit: '%', bytesCount: 2,
        priority: 2, category: 'vtc',
        formula: (b) => (b[0] * 256 + b[1]) * 3200.0 / 32768.0),
    NissanPidDef(cmd: '2211140401', answer: '621114', name: 'FUEL_LVL',
        desc: 'Уровень топлива', unit: 'V', bytesCount: 1,
        priority: 3, category: 'fuel', formula: (b) => b[0] * 0.04),
    NissanPidDef(cmd: '2212570401', answer: '621257', name: 'POWER_RQ',
        desc: 'Мощность запр.', unit: 'kW', bytesCount: 2,
        priority: 2, category: 'engine',
        formula: (b) => (b[0] * 256 + b[1]) * 0.03125),
    NissanPidDef(cmd: '2212280401', answer: '621228', name: 'TORQUE',
        desc: 'Момент', unit: 'Nm', bytesCount: 2,
        priority: 2, category: 'engine',
        formula: (b) {
          int val = b[0] * 256 + b[1];
          if (val >= 32768) val -= 65536;
          return val / 4.0;
        }),
    NissanPidDef(cmd: '2211290401', answer: '621129', name: 'BARO',
        desc: 'Атм. давление', unit: 'V', bytesCount: 1,
        priority: 3, category: 'air', formula: (b) => b[0] * 0.02),
    NissanPidDef(cmd: '2212040401', answer: '621204', name: 'MAF_V',
        desc: 'MAF V', unit: 'V', bytesCount: 2,
        priority: 3, category: 'air',
        formula: (b) => (b[0] * 256 + b[1]) * 0.005),
    NissanPidDef(cmd: '22120D0401', answer: '62120D', name: 'ACCEL1',
        desc: 'Педаль S1 V', unit: 'V', bytesCount: 2,
        priority: 3, category: 'throttle',
        formula: (b) => (b[0] * 256 + b[1]) * 0.005),
    NissanPidDef(cmd: '2211900401', answer: '621190', name: 'ALT_SPEED',
        desc: 'Об. генератора', unit: 'RPM', bytesCount: 1,
        priority: 3, category: 'electric', formula: (b) => b[0] * 0.75),
    NissanPidDef(cmd: '2211510401', answer: '621151', name: 'BAT_SOC',
        desc: 'Заряд АКБ', unit: '%', bytesCount: 1,
        priority: 3, category: 'electric', formula: (b) => b[0].toDouble()),
  ];

  static List<NissanPidDef> byPriority(int p) => all.where((x) => x.priority == p).toList();
  static NissanPidDef? byName(String name) {
    try { return all.firstWhere((p) => p.name == name); } catch (e) { return null; }
  }
  static List<NissanPidDef> byCategory(String c) => all.where((x) => x.category == c).toList();
  static List<String> get categories => all.map((p) => p.category).toSet().toList()..sort();
}
''')
print("✅ nissan_pid_library.dart")

# ============ settings_service.dart ============
with open('lib/services/settings_service.dart', 'w') as f:
    f.write('''import 'package:shared_preferences/shared_preferences.dart';

class SettingsService {
  static const _kPollingInterval = 'polling_interval';
  static const _kLastBtDevice = 'last_bt_device';
  static const _kAutoConnect = 'auto_connect';
  static const _kAutoLog = 'auto_log';
  static const _kSoundEnabled = 'sound_enabled';
  static const _kVibrationEnabled = 'vibration_enabled';
  static const _kAlertsEnabled = 'alerts_enabled';
  static const _kSelectedGraphProfile = 'selected_graph_profile';
  static const _kSelectedLogParams = 'selected_log_params';
  static const _kEngineDisplacement = 'engine_displacement';
  static const _kCachedPidList = 'cached_pids';
  static const _kCachedEcuId = 'cached_ecu_id';
  static const _kMafMultiplier = 'maf_multiplier';
  static const _kSpeedMultiplier = 'speed_multiplier';
  static const _kTripFuelL = 'trip_fuel_l';
  static const _kMemReadCommand = 'mem_read_command';
  static const _kCustomMaps = 'custom_maps';

  static SharedPreferences? _prefs;
  static Future<void> init() async {
    _prefs ??= await SharedPreferences.getInstance();
  }

  static int get pollingInterval => _prefs?.getInt(_kPollingInterval) ?? 50;
  static Future<void> setPollingInterval(int v) async {
    await init(); await _prefs!.setInt(_kPollingInterval, v);
  }
  static String? get lastBtDevice => _prefs?.getString(_kLastBtDevice);
  static Future<void> setLastBtDevice(String? addr) async {
    await init();
    if (addr == null) { await _prefs!.remove(_kLastBtDevice); }
    else { await _prefs!.setString(_kLastBtDevice, addr); }
  }
  static bool get autoConnect => _prefs?.getBool(_kAutoConnect) ?? false;
  static Future<void> setAutoConnect(bool v) async {
    await init(); await _prefs!.setBool(_kAutoConnect, v);
  }
  static bool get autoLog => _prefs?.getBool(_kAutoLog) ?? false;
  static Future<void> setAutoLog(bool v) async {
    await init(); await _prefs!.setBool(_kAutoLog, v);
  }
  static bool get soundEnabled => _prefs?.getBool(_kSoundEnabled) ?? true;
  static Future<void> setSoundEnabled(bool v) async {
    await init(); await _prefs!.setBool(_kSoundEnabled, v);
  }
  static bool get vibrationEnabled => _prefs?.getBool(_kVibrationEnabled) ?? true;
  static Future<void> setVibrationEnabled(bool v) async {
    await init(); await _prefs!.setBool(_kVibrationEnabled, v);
  }
  static bool get alertsEnabled => _prefs?.getBool(_kAlertsEnabled) ?? true;
  static Future<void> setAlertsEnabled(bool v) async {
    await init(); await _prefs!.setBool(_kAlertsEnabled, v);
  }
  static String get selectedGraphProfile =>
      _prefs?.getString(_kSelectedGraphProfile) ?? 'Настройка зажигания';
  static Future<void> setSelectedGraphProfile(String v) async {
    await init(); await _prefs!.setString(_kSelectedGraphProfile, v);
  }
  static List<String> get selectedLogParams =>
      _prefs?.getStringList(_kSelectedLogParams) ?? ['RPM'];
  static Future<void> setSelectedLogParams(List<String> v) async {
    await init(); await _prefs!.setStringList(_kSelectedLogParams, v);
  }
  static double get engineDisplacement => _prefs?.getDouble(_kEngineDisplacement) ?? 2.0;
  static Future<void> setEngineDisplacement(double v) async {
    await init(); await _prefs!.setDouble(_kEngineDisplacement, v);
  }
  static List<String> get cachedPidList => _prefs?.getStringList(_kCachedPidList) ?? [];
  static Future<void> setCachedPidList(List<String> pids) async {
    await init(); await _prefs!.setStringList(_kCachedPidList, pids);
  }
  static String? get cachedEcuId => _prefs?.getString(_kCachedEcuId);
  static Future<void> setCachedEcuId(String? id) async {
    await init();
    if (id == null) { await _prefs!.remove(_kCachedEcuId); }
    else { await _prefs!.setString(_kCachedEcuId, id); }
  }
  static Future<void> clearPidCache() async {
    await init();
    await _prefs!.remove(_kCachedPidList);
    await _prefs!.remove(_kCachedEcuId);
  }
  static double get mafMultiplier => _prefs?.getDouble(_kMafMultiplier) ?? 0.1;
  static Future<void> setMafMultiplier(double v) async {
    await init(); await _prefs!.setDouble(_kMafMultiplier, v);
  }
  static double get speedMultiplier => _prefs?.getDouble(_kSpeedMultiplier) ?? 1.05;
  static Future<void> setSpeedMultiplier(double v) async {
    await init(); await _prefs!.setDouble(_kSpeedMultiplier, v);
  }
  static double get tripFuelL => _prefs?.getDouble(_kTripFuelL) ?? 0.0;
  static Future<void> setTripFuelL(double v) async {
    await init(); await _prefs!.setDouble(_kTripFuelL, v);
  }
  static Future<void> resetTripFuel() async {
    await init(); await _prefs!.setDouble(_kTripFuelL, 0.0);
  }
  static String? get memReadCommand => _prefs?.getString(_kMemReadCommand);
  static Future<void> setMemReadCommand(String? cmd) async {
    await init();
    if (cmd == null) { await _prefs!.remove(_kMemReadCommand); }
    else { await _prefs!.setString(_kMemReadCommand, cmd); }
  }
  static List<String> get customMaps => _prefs?.getStringList(_kCustomMaps) ?? [];
  static Future<void> setCustomMaps(List<String> maps) async {
    await init(); await _prefs!.setStringList(_kCustomMaps, maps);
  }
}
''')
print("✅ settings_service.dart")

# ============ obd_service.dart (с pause polling + MAF/Speed multipliers) ============
with open('lib/services/obd_service.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:typed_data';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';
import '../models/obd_data.dart';
import 'nissan_pid_library.dart';
import 'settings_service.dart';

class OBDService {
  BluetoothConnection? _connection;
  StringBuffer _responseBuffer = StringBuffer();
  final StreamController<OBDData> _dataController = StreamController<OBDData>.broadcast();
  final StreamController<String> _logController = StreamController<String>.broadcast();

  bool _isPolling = false;
  int pollingInterval = 50;
  String _protocolInfo = '';
  bool _isInitialized = false;
  bool _ecuResponds = false;
  String _ecuIdString = '';
  int _pollCounter = 0;
  int _lastPollDurationMs = 0;
  double _pollFps = 0.0;
  String? _lastConnectedAddress;

  bool _pollingSuspended = false;
  bool get pollingSuspended => _pollingSuspended;

  double _tripFuelL = 0;
  DateTime? _lastFuelTime;

  bool _autoReconnect = true;
  int _reconnectAttempts = 0;
  static const int MAX_RECONNECT_ATTEMPTS = 3;
  Timer? _reconnectTimer;

  bool _commandInProgress = false;
  Completer<String>? _responseCompleter;
  StreamSubscription? _inputSubscription;

  final Map<String, double> _nissanValues = {};
  Map<String, double> get nissanValues => Map.unmodifiable(_nissanValues);

  final Map<String, List<int>> _rawNissanData = {};
  Map<String, List<int>> get rawNissanData => Map.unmodifiable(_rawNissanData);

  final Map<String, String> _workingPids = {};
  Map<String, String> get workingPids => Map.unmodifiable(_workingPids);

  List<NissanPidDef> _activePids = [];
  List<NissanPidDef> _fastPids = [];
  List<NissanPidDef> _mediumPids = [];
  List<NissanPidDef> _slowPids = [];
  List<NissanPidDef> get activePids => _activePids;

  int get pollFps => _pollFps.toInt();
  int get lastPollMs => _lastPollDurationMs;
  double get tripFuelL => _tripFuelL;

  Stream<OBDData> get dataStream => _dataController.stream;
  Stream<String> get logStream => _logController.stream;
  bool get isConnected => _connection?.isConnected ?? false;
  bool get isInitialized => _isInitialized;
  bool get ecuResponds => _ecuResponds;
  String get protocolInfo => _protocolInfo;
  String get ecuId => _ecuIdString;

  Future<void> resetTripFuel() async {
    _tripFuelL = 0;
    _lastFuelTime = null;
    await SettingsService.resetTripFuel();
  }

  Future<void> loadTripFuel() async {
    _tripFuelL = SettingsService.tripFuelL;
  }

  void _log(String message) {
    print('[OBD] ' + message);
    _logController.add(message);
  }

  Future<List<BluetoothDevice>> getBondedDevices() async {
    try { return await FlutterBluetoothSerial.instance.getBondedDevices(); }
    catch (e) { return []; }
  }

  Future<BluetoothState> getBluetoothState() async {
    return await FlutterBluetoothSerial.instance.state;
  }

  Future<bool?> requestEnable() async {
    return await FlutterBluetoothSerial.instance.requestEnable();
  }

  Future<bool> connect(String address) async {
    try {
      _log('=== BT ' + address + ' ===');
      _isInitialized = false;
      _ecuResponds = false;
      _lastConnectedAddress = address;
      _reconnectAttempts = 0;

      _connection = await BluetoothConnection.toAddress(address);
      _log('BT OK');

      _inputSubscription = _connection!.input!.listen(
        _onDataReceived, onDone: _onDisconnected,
        onError: (e) => _log('BT Error: ' + e.toString()),
      );

      await Future.delayed(const Duration(milliseconds: 1500));
      _responseBuffer.clear();
      _commandInProgress = false;
      _responseCompleter = null;

      _connection!.output.add(Uint8List.fromList([13, 13, 13]));
      await _connection!.output.allSent;
      await Future.delayed(const Duration(milliseconds: 500));
      _responseBuffer.clear();

      String r = await sendCommand('ATZ', timeout: 5000);
      _log('ATZ: [' + r + ']');
      await Future.delayed(const Duration(milliseconds: 1500));

      if (r.toUpperCase().contains('ELM')) {
        _isInitialized = true;
        _log('ELM OK');
        await SettingsService.setLastBtDevice(address);
        return true;
      }
      return false;
    } catch (e) {
      _log('ERR: ' + e.toString());
      return false;
    }
  }

  Future<bool> initECU({bool useCache = true}) async {
    if (!isConnected) return false;
    _log('=== INIT ECU ===');
    _ecuResponds = false;
    _rawNissanData.clear();
    _workingPids.clear();
    _activePids.clear();
    _fastPids.clear();
    _mediumPids.clear();
    _slowPids.clear();

    await sendCommand('ATZ', timeout: 4000);
    await Future.delayed(const Duration(milliseconds: 1000));
    await sendCommand('ATE0', timeout: 1500);
    await sendCommand('ATL0', timeout: 1500);
    await sendCommand('ATS0', timeout: 1500);
    await sendCommand('ATH0', timeout: 1500);
    await sendCommand('ATAL', timeout: 1500);
    await sendCommand('ATSW00', timeout: 1500);
    await sendCommand('ATST19', timeout: 1500);
    await sendCommand('ATAT2', timeout: 1500);
    await sendCommand('ATIB10', timeout: 1500);
    await sendCommand('ATSP5', timeout: 1500);
    await sendCommand('ATSH8110FC', timeout: 1500);
    await sendCommand('ATFI', timeout: 3000);
    await Future.delayed(const Duration(milliseconds: 200));

    String r = await sendCommand('2211000401', timeout: 8000);
    _log('BUS: [' + r + ']');
    String rClean = r.replaceAll(' ', '').toUpperCase();

    if (!rClean.contains('6211')) {
      _log('BUS FAIL');
      return false;
    }
    _log('!!! ЭБУ ОТВЕЧАЕТ !!!');
    _ecuResponds = true;

    r = await sendCommand('1A81', timeout: 3000);
    rClean = r.replaceAll(' ', '').toUpperCase();
    if (rClean.contains('5A')) {
      int idx = rClean.indexOf('5A');
      _ecuIdString = _hexToAscii(rClean.substring(idx + 2));
      _log('ECU: ' + _ecuIdString);
    }

    if (useCache) {
      final cachedEcu = SettingsService.cachedEcuId;
      final cachedPids = SettingsService.cachedPidList;
      if (cachedEcu == _ecuIdString && cachedPids.isNotEmpty) {
        _log('CACHE ' + cachedPids.length.toString());
        for (var pidName in cachedPids) {
          final pid = NissanPidLibrary.byName(pidName);
          if (pid != null) _activePids.add(pid);
        }
        _fastPids = _activePids.where((p) => p.priority == 1).toList();
        _mediumPids = _activePids.where((p) => p.priority == 2).toList();
        _slowPids = _activePids.where((p) => p.priority == 3).toList();
        _protocolInfo = 'Nissan ' + _ecuIdString + ' (кеш)';
        Future.delayed(const Duration(milliseconds: 300), () => startPolling());
        return true;
      }
    }

    _log('SCAN ' + NissanPidLibrary.all.length.toString());
    int working = 0;
    for (var pid in NissanPidLibrary.all) {
      r = await sendCommand(pid.cmd, timeout: 600);
      rClean = r.replaceAll(' ', '').toUpperCase();
      if (rClean.contains(pid.answer)) {
        List<int> bytes = _extractDataBytes(r, pid.answer);
        if (bytes.length >= pid.bytesCount) {
          _activePids.add(pid);
          _workingPids[pid.cmd] = r;
          working++;
        }
      }
      await Future.delayed(const Duration(milliseconds: 20));
    }
    _fastPids = _activePids.where((p) => p.priority == 1).toList();
    _mediumPids = _activePids.where((p) => p.priority == 2).toList();
    _slowPids = _activePids.where((p) => p.priority == 3).toList();
    _log('Найдено ' + working.toString());

    if (_activePids.isEmpty) return false;

    await SettingsService.setCachedEcuId(_ecuIdString);
    await SettingsService.setCachedPidList(_activePids.map((p) => p.name).toList());

    _protocolInfo = 'Nissan ' + _ecuIdString + ' (' + working.toString() + ')';
    Future.delayed(const Duration(milliseconds: 300), () => startPolling());
    return true;
  }

  String _hexToAscii(String hex) {
    StringBuffer sb = StringBuffer();
    for (int i = 0; i < hex.length - 1; i += 2) {
      try {
        int b = int.parse(hex.substring(i, i + 2), radix: 16);
        if (b >= 0x20 && b <= 0x7E) sb.write(String.fromCharCode(b));
      } catch (e) { break; }
    }
    return sb.toString();
  }

  Future<String> sendCommand(String cmd, {int timeout = 1000, bool suspendPolling = true}) async {
    if (_connection == null || !_connection!.isConnected) return '';

    final needSuspend = suspendPolling && _isPolling;
    if (needSuspend) {
      _pollingSuspended = true;
      int wait = 0;
      while (_commandInProgress && wait < 200) {
        await Future.delayed(const Duration(milliseconds: 10));
        wait++;
      }
    }

    int waitCount = 0;
    while (_commandInProgress && waitCount < 50) {
      await Future.delayed(const Duration(milliseconds: 5));
      waitCount++;
    }
    if (_commandInProgress) {
      _commandInProgress = false;
      _responseCompleter = null;
    }

    _responseBuffer.clear();
    _commandInProgress = true;
    _responseCompleter = Completer<String>();

    try {
      final commandBytes = <int>[];
      commandBytes.addAll(cmd.codeUnits);
      commandBytes.add(13);
      _connection!.output.add(Uint8List.fromList(commandBytes));
      await _connection!.output.allSent;

      String response = '';
      try {
        response = await _responseCompleter!.future.timeout(Duration(milliseconds: timeout));
      } catch (e) { response = _responseBuffer.toString(); }

      _commandInProgress = false;
      _responseCompleter = null;

      if (needSuspend) {
        await Future.delayed(const Duration(milliseconds: 50));
        _pollingSuspended = false;
      }
      return _cleanResponse(response);
    } catch (e) {
      _commandInProgress = false;
      _responseCompleter = null;
      if (needSuspend) _pollingSuspended = false;
      return '';
    }
  }

  String _cleanResponse(String response) {
    String cleaned = response.replaceAll('>', '')
        .replaceAll('\r', ' ').replaceAll('\n', ' ');
    while (cleaned.contains('  ')) {
      cleaned = cleaned.replaceAll('  ', ' ');
    }
    return cleaned.trim();
  }

  void _onDataReceived(Uint8List data) {
    final received = String.fromCharCodes(data);
    _responseBuffer.write(received);
    if (received.contains('>') && _responseCompleter != null &&
        !_responseCompleter!.isCompleted) {
      _responseCompleter!.complete(_responseBuffer.toString());
    }
  }

  void _onDisconnected() {
    _log('BT DC');
    stopPolling();
    _isInitialized = false;
    _ecuResponds = false;
    _commandInProgress = false;
    _responseCompleter = null;
    _connection = null;
    if (_autoReconnect && _lastConnectedAddress != null &&
        _reconnectAttempts < MAX_RECONNECT_ATTEMPTS) {
      _tryReconnect();
    }
  }

  void _tryReconnect() {
    _reconnectAttempts++;
    _log('Reconnect ' + _reconnectAttempts.toString());
    _reconnectTimer?.cancel();
    _reconnectTimer = Timer(Duration(seconds: 3 * _reconnectAttempts), () async {
      if (_lastConnectedAddress != null) {
        final ok = await connect(_lastConnectedAddress!);
        if (ok) {
          await initECU(useCache: true);
        } else if (_reconnectAttempts < MAX_RECONNECT_ATTEMPTS) {
          _tryReconnect();
        }
      }
    });
  }

  void startPolling() {
    if (_isPolling) return;
    if (!_ecuResponds) return;
    if (_activePids.isEmpty) return;
    _isPolling = true;
    _log('POLL START');
    _pollFast();
  }

  void stopPolling() { _isPolling = false; }

  int _mediumIdx = 0;
  int _slowIdx = 0;

  Future<void> _pollFast() async {
    Stopwatch fpsTimer = Stopwatch()..start();
    int fpsCounter = 0;
    while (_isPolling && isConnected && _ecuResponds) {
      try {
        while (_pollingSuspended && _isPolling) {
          await Future.delayed(const Duration(milliseconds: 20));
        }
        if (!_isPolling) break;

        _pollCounter++;
        Stopwatch sw = Stopwatch()..start();

        for (var pd in _fastPids) {
          if (!_isPolling || _pollingSuspended) break;
          try {
            final r = await sendCommand(pd.cmd, timeout: 300, suspendPolling: false);
            final bytes = _extractDataBytes(r, pd.answer);
            if (bytes.length >= pd.bytesCount) {
              _nissanValues[pd.name] = pd.formula(bytes);
              _rawNissanData[pd.cmd] = bytes;
            }
          } catch (e) {}
        }

        if (_pollCounter % 3 == 0 && _mediumPids.isNotEmpty && !_pollingSuspended) {
          for (int i = 0; i < 2 && _isPolling && !_pollingSuspended; i++) {
            var pd = _mediumPids[_mediumIdx % _mediumPids.length];
            _mediumIdx++;
            try {
              final r = await sendCommand(pd.cmd, timeout: 300, suspendPolling: false);
              final bytes = _extractDataBytes(r, pd.answer);
              if (bytes.length >= pd.bytesCount) {
                _nissanValues[pd.name] = pd.formula(bytes);
                _rawNissanData[pd.cmd] = bytes;
              }
            } catch (e) {}
          }
        }

        if (_pollCounter % 10 == 0 && _slowPids.isNotEmpty && !_pollingSuspended) {
          var pd = _slowPids[_slowIdx % _slowPids.length];
          _slowIdx++;
          try {
            final r = await sendCommand(pd.cmd, timeout: 300, suspendPolling: false);
            final bytes = _extractDataBytes(r, pd.answer);
            if (bytes.length >= pd.bytesCount) {
              _nissanValues[pd.name] = pd.formula(bytes);
              _rawNissanData[pd.cmd] = bytes;
            }
          } catch (e) {}
        }

        sw.stop();
        _lastPollDurationMs = sw.elapsedMilliseconds;
        fpsCounter++;
        if (fpsTimer.elapsedMilliseconds >= 1000) {
          _pollFps = fpsCounter * 1000.0 / fpsTimer.elapsedMilliseconds;
          fpsCounter = 0;
          fpsTimer.reset();
        }
        _publishData();
      } catch (e) {}
      if (pollingInterval > 0) {
        await Future.delayed(Duration(milliseconds: pollingInterval));
      }
    }
  }

  double _calcAfr() {
    double o2 = _nissanValues['O2_B1S1'] ?? 0;
    double stft = _nissanValues['STFT'] ?? 0;
    double lambda;
    if (o2 > 0.85) lambda = 0.87;
    else if (o2 > 0.75) lambda = 0.92;
    else if (o2 > 0.6) lambda = 0.97;
    else if (o2 > 0.45) lambda = 1.00;
    else if (o2 > 0.3) lambda = 1.03;
    else if (o2 > 0.15) lambda = 1.05;
    else lambda = 1.10;
    lambda *= (1 + stft / 100.0 * 0.3);
    return (lambda * 14.7).clamp(10.0, 20.0);
  }

  void _updateTripFuel(double fuelFlowLph) {
    final now = DateTime.now();
    if (_lastFuelTime != null && fuelFlowLph > 0) {
      final dt = now.difference(_lastFuelTime!).inMilliseconds / 1000.0;
      final litresConsumed = (fuelFlowLph / 3600.0) * dt;
      _tripFuelL += litresConsumed;
      if (_pollCounter % 100 == 0) {
        SettingsService.setTripFuelL(_tripFuelL);
      }
    }
    _lastFuelTime = now;
  }

  void _publishData() {
    double _v(String name) => _nissanValues[name] ?? 0;
    double afr = _calcAfr();
    double throttle = _v('TPS');

    double rawMaf = _v('MAF');
    double mafMult = SettingsService.mafMultiplier;
    double mafCorrected = rawMaf * (mafMult / 0.01);

    double rawSpeed = _v('SPEED');
    int correctedSpeed = (rawSpeed * SettingsService.speedMultiplier).toInt().clamp(0, 300);

    double displacement = SettingsService.engineDisplacement;

    var data = OBDData(
      timestamp: DateTime.now(),
      rpm: _v('RPM').toInt().clamp(0, 9999),
      speed: correctedSpeed,
      engineLoad: _v('LOAD').clamp(0, 100),
      coolantTemp: _v('ECT').toInt().clamp(-40, 200),
      intakeTemp: _v('IAT').toInt().clamp(-40, 100),
      maf: mafCorrected,
      throttlePos: throttle.clamp(0, 100),
      ignitionTiming: _v('TIMING'),
      actualIgnition: _v('TIMING'),
      vtcActualAngle: _v('VTC_ACTUAL'),
      vtcTargetAngle: 0,
      knockRetard: _v('KNOCK').abs(),
      shortFuelTrim: _v('STFT').clamp(-100, 100),
      longFuelTrim: _v('LTFT').clamp(-100, 100),
      o2Voltage: _v('O2_B1S1'),
      afr: afr, oilTemp: 0,
      injectorPulseWidth: _v('INJ_B1'),
      injectorDuty: (_v('INJ_B1') / 20.0 * 100).clamp(0, 100),
      manifoldPressure: _v('MAP_V') * 40,
      acceleratorPedal: _v('PEDAL'),
      throttleActual: throttle,
      actualTorque: 0, requestedTorque: 0,
      batteryVoltage: _v('BATT'),
      engineDisplacement: displacement,
      tripFuelL: _tripFuelL,
    );
    _updateTripFuel(data.fuelFlowLph);
    _dataController.add(data);
  }

  List<int> _extractDataBytes(String response, String prefix) {
    response = response.replaceAll(' ', '').replaceAll('BUSINIT:OK', '').toUpperCase();
    int idx = response.indexOf(prefix);
    if (idx == -1) return [];
    String data = response.substring(idx + prefix.length);
    List<int> bytes = [];
    for (int i = 0; i < data.length - 1; i += 2) {
      try {
        String hex = data.substring(i, i + 2);
        if (!RegExp(r'^[0-9A-F]+$').hasMatch(hex)) break;
        bytes.add(int.parse(hex, radix: 16));
      } catch (e) { break; }
    }
    return bytes;
  }

  Future<void> disconnect() async {
    _autoReconnect = false;
    _reconnectTimer?.cancel();
    stopPolling();
    _isInitialized = false;
    _ecuResponds = false;
    _commandInProgress = false;
    _responseCompleter = null;
    await _inputSubscription?.cancel();
    _inputSubscription = null;
    await _connection?.close();
    _connection = null;
    _autoReconnect = true;
    await SettingsService.setTripFuelL(_tripFuelL);
  }

  void dispose() {
    disconnect();
    _dataController.close();
    _logController.close();
  }
}
''')
print("✅ obd_service.dart (с pause polling + MAF/Speed multipliers)")

# ============ logger_service.dart ============
with open('lib/services/logger_service.dart', 'w') as f:
    f.write('''import 'dart:io';
import 'package:csv/csv.dart';
import 'package:path_provider/path_provider.dart';
import 'package:intl/intl.dart';
import '../models/obd_data.dart';
import '../constants.dart';
import 'settings_service.dart';

class LoggerService {
  final List<OBDData> _logBuffer = [];
  bool _isLogging = false;
  bool _isAutoLogging = false;
  String? _currentLogPath;
  DateTime? _lastActivityTime;

  bool get isLogging => _isLogging;
  bool get isAutoLogging => _isAutoLogging;
  int get bufferSize => _logBuffer.length;
  String? get currentLogPath => _currentLogPath;

  Future<void> startLogging({bool auto = false}) async {
    _logBuffer.clear();
    _isLogging = true;
    _isAutoLogging = auto;
    _lastActivityTime = DateTime.now();
    final now = DateTime.now();
    final prefix = auto ? 'auto_' : '';
    final fileName = 'nissan_' + prefix + 'log_' +
                     DateFormat('yyyyMMdd_HHmmss').format(now) + '.csv';
    final directory = await getApplicationDocumentsDirectory();
    _currentLogPath = directory.path + '/' + fileName;
  }

  void addData(OBDData data) {
    if (_isLogging) {
      _logBuffer.add(data);
      if (_logBuffer.length >= 50) _flushToFile();
    }
    if (SettingsService.autoLog) _handleAutoLog(data);
  }

  void _handleAutoLog(OBDData data) {
    final isMoving = data.rpm > AppConstants.autoLogRpmThreshold ||
                     data.speed > AppConstants.autoLogSpeedThreshold;
    if (isMoving) {
      _lastActivityTime = DateTime.now();
      if (!_isLogging) startLogging(auto: true);
    } else if (_isLogging && _isAutoLogging && _lastActivityTime != null) {
      final idleSec = DateTime.now().difference(_lastActivityTime!).inSeconds;
      if (idleSec >= AppConstants.autoLogIdleTimeoutSec) stopLogging();
    }
  }

  Future<String?> stopLogging() async {
    if (!_isLogging) return null;
    _isLogging = false;
    _isAutoLogging = false;
    await _flushToFile();
    return _currentLogPath;
  }

  Future<void> _flushToFile() async {
    if (_currentLogPath == null || _logBuffer.isEmpty) return;
    final file = File(_currentLogPath!);
    final exists = await file.exists();
    List<List<dynamic>> rows = [];
    if (!exists) rows.add(OBDData.csvHeaders());
    for (var data in _logBuffer) rows.add(data.toCsvRow());
    String csvData = const ListToCsvConverter().convert(rows);
    if (exists) {
      await file.writeAsString('\\n' + csvData, mode: FileMode.append);
    } else {
      await file.writeAsString(csvData);
    }
    _logBuffer.clear();
  }

  Future<List<FileSystemEntity>> getSavedLogs() async {
    final directory = await getApplicationDocumentsDirectory();
    final files = directory.listSync().where((f) => f.path.endsWith('.csv')).toList();
    files.sort((a, b) => b.path.compareTo(a.path));
    return files;
  }

  Future<void> deleteLog(String path) async {
    final file = File(path);
    if (await file.exists()) await file.delete();
  }
}
''')
print("✅ logger_service.dart")

# ============ dtc_database.dart ============
with open('lib/services/dtc_database.dart', 'w') as f:
    f.write('''class DTCDatabase {
  static final Map<String, String> codes = {
    'P0100': 'Неисправность цепи датчика расхода воздуха',
    'P0101': 'Сигнал MAF вне допустимого диапазона',
    'P0102': 'Низкий уровень сигнала MAF',
    'P0103': 'Высокий уровень сигнала MAF',
    'P0110': 'Неисправность датчика температуры впуска',
    'P0112': 'Низкий уровень датчика темп. впуска',
    'P0113': 'Высокий уровень датчика темп. впуска',
    'P0115': 'Неисправность датчика температуры ОЖ',
    'P0117': 'Низкий уровень датчика ОЖ',
    'P0118': 'Высокий уровень датчика ОЖ',
    'P0120': 'Неисправность датчика дросселя A',
    'P0122': 'Низкий сигнал дросселя A',
    'P0123': 'Высокий сигнал дросселя A',
    'P0130': 'Датчик O2 B1S1 неисправен',
    'P0131': 'Низкий сигнал O2 B1S1',
    'P0132': 'Высокий сигнал O2 B1S1',
    'P0133': 'Медленный отклик O2 B1S1',
    'P0134': 'Нет активности O2 B1S1',
    'P0135': 'Нагреватель O2 B1S1',
    'P0136': 'Датчик O2 B1S2',
    'P0141': 'Нагреватель O2 B1S2',
    'P0171': 'Слишком бедная смесь B1',
    'P0172': 'Слишком богатая смесь B1',
    'P0201': 'Форсунка 1',
    'P0202': 'Форсунка 2',
    'P0203': 'Форсунка 3',
    'P0204': 'Форсунка 4',
    'P0300': 'Множественные пропуски зажигания',
    'P0301': 'Пропуски цилиндр 1',
    'P0302': 'Пропуски цилиндр 2',
    'P0303': 'Пропуски цилиндр 3',
    'P0304': 'Пропуски цилиндр 4',
    'P0325': 'Датчик детонации',
    'P0335': 'ДПКВ',
    'P0340': 'Датчик распредвала',
    'P0420': 'Катализатор B1 ниже нормы',
    'P0440': 'EVAP',
    'P0500': 'Датчик скорости',
    'P0505': 'Регулятор ХХ',
    'P0605': 'ROM ЭБУ',
    'P0700': 'АКПП',
    'P1111': 'VVT клапан',
    'P1128': 'Блокировка ETCS',
    'P1130': 'A/F датчик диапазон',
    'P1345': 'VVT датчик',
    'P1614': 'NATS иммобилайзер',
    'P1652': 'IACV',
    'P1656': 'OCV VVT B1',
  };

  static String getDescription(String code) {
    return codes[code] ?? 'Неизвестная ошибка (' + code + ')';
  }
}
''')
print("✅ dtc_database.dart")

# ============ dtc_service.dart ============
with open('lib/services/dtc_service.dart', 'w') as f:
    f.write('''import 'obd_service.dart';
import '../models/dtc_code.dart';
import 'dtc_database.dart';

class DTCService {
  final OBDService _obd;
  DTCService(this._obd);

  Future<List<DTCCode>> readStoredDTC() async {
    final r = await _obd.sendCommand('03');
    return _parseDTC(r, false);
  }
  Future<List<DTCCode>> readPendingDTC() async {
    final r = await _obd.sendCommand('07');
    return _parseDTC(r, true);
  }
  Future<bool> clearDTC() async {
    final r = await _obd.sendCommand('04');
    return r.contains('44') || r.contains('OK');
  }

  List<DTCCode> _parseDTC(String r, bool pending) {
    List<DTCCode> codes = [];
    String prefix = pending ? '47' : '43';
    String clean = r.replaceAll(' ', '').toUpperCase();
    int idx = clean.indexOf(prefix);
    if (idx == -1) return codes;
    String data = clean.substring(idx + 2);
    if (data.length < 2) return codes;
    int count = int.parse(data.substring(0, 2), radix: 16);
    data = data.substring(2);
    for (int i = 0; i < count && data.length >= 4; i++) {
      String raw = data.substring(0, 4);
      data = data.substring(4);
      String code = _decode(raw);
      if (code != '0000') {
        codes.add(DTCCode(code: code,
          description: DTCDatabase.getDescription(code),
          type: _type(code), isPending: pending));
      }
    }
    return codes;
  }

  String _decode(String raw) {
    if (raw.length != 4) return '0000';
    int b1 = int.parse(raw.substring(0, 2), radix: 16);
    int b2 = int.parse(raw.substring(2, 4), radix: 16);
    String p;
    switch ((b1 >> 6) & 3) {
      case 0: p = 'P'; break;
      case 1: p = 'C'; break;
      case 2: p = 'B'; break;
      case 3: p = 'U'; break;
      default: p = 'P';
    }
    return p + ((b1 >> 4) & 3).toString() +
           (b1 & 0x0F).toRadixString(16).toUpperCase() +
           b2.toRadixString(16).padLeft(2, '0').toUpperCase();
  }

  DTCType _type(String code) {
    if (code.startsWith('P')) return DTCType.powertrain;
    if (code.startsWith('C')) return DTCType.chassis;
    if (code.startsWith('B')) return DTCType.body;
    return DTCType.network;
  }
}
''')
print("✅ dtc_service.dart")

# ============ alert_service.dart (со snapshot + explanation) ============
with open('lib/services/alert_service.dart', 'w') as f:
    f.write(r'''import 'package:flutter/services.dart';
import 'package:vibration/vibration.dart';
import '../models/obd_data.dart';
import '../models/alert.dart';
import '../constants.dart';
import 'settings_service.dart';

class AlertService {
  DateTime? _lastAlertTime;
  final List<Alert> _recentAlerts = [];
  final List<Alert> _allAlerts = [];

  List<Alert> get recentAlerts => List.unmodifiable(_recentAlerts);
  List<Alert> get allAlerts => List.unmodifiable(_allAlerts);

  List<Alert> checkData(OBDData data) {
    if (!SettingsService.alertsEnabled) return [];
    List<Alert> alerts = [];

    final snap = AlertSnapshot(
      rpm: data.rpm, speed: data.speed, engineLoad: data.engineLoad,
      coolantTemp: data.coolantTemp, intakeTemp: data.intakeTemp,
      maf: data.maf, throttlePos: data.throttlePos, afr: data.afr,
      knockRetard: data.knockRetard,
      shortFuelTrim: data.shortFuelTrim, longFuelTrim: data.longFuelTrim,
      ignitionTiming: data.actualIgnition, vtcActualAngle: data.vtcActualAngle,
    );

    if (data.knockRetard >= AppConstants.knockRetardDanger) {
      alerts.add(Alert(
        message: 'СИЛЬНАЯ ДЕТОНАЦИЯ ' + data.knockRetard.toStringAsFixed(1) + '°',
        level: AlertLevel.danger, timestamp: DateTime.now(),
        category: 'knock', snapshot: snap,
        explanation: 'ЭБУ снижает УОЗ на ' + data.knockRetard.toStringAsFixed(1) + '°.\n' +
            'Возможные причины:\n' +
            '• Некачественное топливо (низкий октан)\n' +
            '• Перегрев (ОЖ ' + data.coolantTemp.toString() + '°C)\n' +
            '• Обеднённая смесь (AFR ' + data.afr.toStringAsFixed(2) + ')\n' +
            '• Излишний УОЗ в карте (' + data.actualIgnition.toStringAsFixed(1) + '°)\n' +
            '• Высокая нагрузка ' + data.engineLoad.toStringAsFixed(0) + '%',
      ));
    } else if (data.knockRetard >= AppConstants.knockRetardWarning) {
      alerts.add(Alert(
        message: 'Детонация ' + data.knockRetard.toStringAsFixed(1) + '°',
        level: AlertLevel.warning, timestamp: DateTime.now(),
        category: 'knock', snapshot: snap,
        explanation: 'Лёгкая детонация.\nRPM: ' + data.rpm.toString() +
            ', Нагрузка: ' + data.engineLoad.toStringAsFixed(0) + '%',
      ));
    }

    if (data.coolantTemp >= AppConstants.coolantTempDanger) {
      alerts.add(Alert(
        message: 'ПЕРЕГРЕВ! ОЖ = ' + data.coolantTemp.toString() + '°C',
        level: AlertLevel.danger, timestamp: DateTime.now(),
        category: 'temp', snapshot: snap,
        explanation: 'КРИТИЧЕСКАЯ температура!\n' +
            '• Отказ термостата\n• Помпа\n• Вентилятор\n' +
            '• Утечка ОЖ / воздушная пробка\n\n' +
            'ЗАГЛУШИ ДВИГАТЕЛЬ!',
      ));
    } else if (data.coolantTemp >= AppConstants.coolantTempWarning) {
      alerts.add(Alert(
        message: 'ОЖ высокая: ' + data.coolantTemp.toString() + '°C',
        level: AlertLevel.warning, timestamp: DateTime.now(),
        category: 'temp', snapshot: snap,
        explanation: 'Норма 85-95°C.\nПроверь вентилятор, уровень ОЖ, термостат.',
      ));
    }

    if (data.engineLoad > 70 && data.afr > AppConstants.afrLeanDanger) {
      alerts.add(Alert(
        message: 'ОЧЕНЬ БЕДНАЯ! AFR=' + data.afr.toStringAsFixed(2),
        level: AlertLevel.danger, timestamp: DateTime.now(),
        category: 'afr', snapshot: snap,
        explanation: 'Смесь очень бедная при нагрузке ' + data.engineLoad.toStringAsFixed(0) + '%.\n' +
            'ОПАСНО - риск прогара клапанов!\n' +
            '• Подсос воздуха\n' +
            '• Забиты форсунки (' + data.injectorPulseWidth.toStringAsFixed(2) + 'ms)\n' +
            '• Слабое давление топлива\n' +
            '• Неисправность MAF (' + data.maf.toStringAsFixed(2) + ' g/s)\n' +
            '• STFT ' + data.shortFuelTrim.toStringAsFixed(1) +
              '%, LTFT ' + data.longFuelTrim.toStringAsFixed(1) + '%',
      ));
    }

    double totalTrim = data.totalFuelTrim.abs();
    if (totalTrim >= AppConstants.fuelTrimDanger) {
      final trimDir = data.totalFuelTrim > 0 ? 'ЛЬЁТ (бедно)' : 'РЕЖЕТ (богато)';
      alerts.add(Alert(
        message: 'Коррекции ' + data.totalFuelTrim.toStringAsFixed(1) + '%',
        level: AlertLevel.danger, timestamp: DateTime.now(),
        category: 'fuel', snapshot: snap,
        explanation: 'ЭБУ ' + trimDir + '.\n' +
            'STFT: ' + data.shortFuelTrim.toStringAsFixed(1) + '%, ' +
            'LTFT: ' + data.longFuelTrim.toStringAsFixed(1) + '%\n\n' +
            'При бедной (+):\n• Подсос воздуха\n• Забиты форсунки\n' +
            'При богатой (-):\n• Льют форсунки\n• Врёт MAF',
      ));
    }

    if (alerts.isNotEmpty) {
      _triggerAlerts(alerts);
      _recentAlerts.insertAll(0, alerts);
      _allAlerts.addAll(alerts);
      if (_recentAlerts.length > 50) _recentAlerts.removeRange(50, _recentAlerts.length);
      if (_allAlerts.length > 500) _allAlerts.removeRange(0, _allAlerts.length - 500);
    }
    return alerts;
  }

  void _triggerAlerts(List<Alert> alerts) {
    if (_lastAlertTime != null) {
      if (DateTime.now().difference(_lastAlertTime!).inSeconds < 3) return;
    }
    _lastAlertTime = DateTime.now();
    bool danger = alerts.any((a) => a.level == AlertLevel.danger);
    if (SettingsService.vibrationEnabled) _vibrate(danger);
    if (SettingsService.soundEnabled) _playSound(danger);
  }

  Future<void> _vibrate(bool danger) async {
    try {
      bool has = await Vibration.hasVibrator() ?? false;
      if (has) {
        if (danger) Vibration.vibrate(pattern: [0, 500, 200, 500, 200, 500]);
        else Vibration.vibrate(duration: 300);
      }
    } catch (e) { HapticFeedback.heavyImpact(); }
  }

  Future<void> _playSound(bool danger) async {
    try {
      SystemSound.play(danger ? SystemSoundType.alert : SystemSoundType.click);
    } catch (e) {}
  }

  void clearAlerts() {
    _recentAlerts.clear();
    _allAlerts.clear();
  }

  void dispose() {}
}
''')
print("✅ alert_service.dart (со snapshot + explanation)")

# ============ analyzer_service.dart ============
with open('lib/services/analyzer_service.dart', 'w') as f:
    f.write('''import 'dart:io';
import 'dart:math';
import 'package:csv/csv.dart';
import '../models/obd_data.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';

class AnalyzerService {
  static const int MIN_SAMPLES = 2;
  static const double MIN_CONFIDENCE = 0.4;

  Future<AnalysisResult> analyzeSparkMap(List<OBDData> logData, TuningMap map) async {
    List<MapCell> changes = [];
    final valid = logData.where((d) => d.rpm > 400 && d.rpm < 7500 && d.engineLoad > 0).toList();
    if (valid.length < 10) {
      return AnalysisResult(mapName: 'Spark Advance', analyzedAt: DateTime.now(),
        totalSamples: logData.length, changes: [],
        summary: 'Мало данных: ' + valid.length.toString());
    }
    Map<String, List<OBDData>> cellData = _groupByCells(valid, map);
    for (var entry in cellData.entries) {
      final samples = entry.value;
      if (samples.length < MIN_SAMPLES) continue;
      final parts = entry.key.split(',');
      int rpmIdx = int.parse(parts[0]);
      int loadIdx = int.parse(parts[1]);
      double currentValue = map.data[rpmIdx][loadIdx];
      double avgTiming = _average(samples.map((d) => d.actualIgnition));
      double avgKnock = _average(samples.map((d) => d.knockRetard));
      double avgAFR = _average(samples.map((d) => d.afr));
      double suggested = currentValue;
      String reason = '';
      double confidence = 0;
      if (avgKnock > 2.0) {
        suggested = currentValue - min(avgKnock, 3.0);
        reason = 'Детонация ' + avgKnock.toStringAsFixed(1) + '°';
        confidence = 0.9;
      } else if (avgKnock > 0.5) {
        suggested = currentValue - 1;
        reason = 'Лёгкая детонация';
        confidence = 0.7;
      } else if (avgTiming != 0 && (avgTiming - currentValue).abs() > 2) {
        suggested = avgTiming;
        reason = 'ЭБУ ставит ' + avgTiming.toStringAsFixed(1) + '°';
        confidence = 0.6;
      } else if (avgKnock < 0.1 && avgAFR > 13.0 && avgAFR < 14.5) {
        suggested = currentValue + 1.0;
        reason = 'Стабильно, +1° УОЗ';
        confidence = 0.5;
      }
      suggested = suggested.clamp(-5, 45);
      if ((suggested - currentValue).abs() >= 0.5 && confidence >= MIN_CONFIDENCE) {
        changes.add(MapCell(rpmIndex: rpmIdx, loadIndex: loadIdx,
          rpm: map.rpmAxis[rpmIdx], load: map.loadAxis[loadIdx],
          currentValue: currentValue, suggestedValue: suggested,
          confidence: confidence, sampleCount: samples.length, reason: reason));
      }
    }
    return AnalysisResult(mapName: 'Spark Advance', analyzedAt: DateTime.now(),
      totalSamples: logData.length, changes: changes,
      summary: 'Правок: ' + changes.length.toString());
  }

  Future<AnalysisResult> analyzeFuelMap(List<OBDData> logData, TuningMap map) async {
    return AnalysisResult(mapName: 'Fuel Map', analyzedAt: DateTime.now(),
      totalSamples: logData.length, changes: [], summary: 'TODO');
  }
  Future<AnalysisResult> analyzeVTCMap(List<OBDData> logData, TuningMap map) async {
    return AnalysisResult(mapName: 'VTC', analyzedAt: DateTime.now(),
      totalSamples: logData.length, changes: [], summary: 'TODO');
  }
  Future<AnalysisResult> analyzeTorqueMap(List<OBDData> logData, TuningMap map) async {
    return AnalysisResult(mapName: 'Torque', analyzedAt: DateTime.now(),
      totalSamples: logData.length, changes: [], summary: 'Не измеряется');
  }

  Map<String, List<OBDData>> _groupByCells(List<OBDData> data, TuningMap map) {
    Map<String, List<OBDData>> result = {};
    for (var d in data) {
      String key = _getCellKey(d.rpm.toDouble(), d.engineLoad, map);
      result.putIfAbsent(key, () => []).add(d);
    }
    return result;
  }

  String _getCellKey(double rpm, double load, TuningMap map) {
    int rpmIdx = _findClosestIndex(map.rpmAxis, rpm);
    int loadIdx = _findClosestIndex(map.loadAxis, load);
    return rpmIdx.toString() + ',' + loadIdx.toString();
  }

  int _findClosestIndex(List<double> axis, double value) {
    int idx = 0;
    double minDiff = double.infinity;
    for (int i = 0; i < axis.length; i++) {
      double diff = (axis[i] - value).abs();
      if (diff < minDiff) { minDiff = diff; idx = i; }
    }
    return idx;
  }

  double _average(Iterable<num> values) {
    if (values.isEmpty) return 0;
    return values.reduce((a, b) => a + b) / values.length;
  }

  Future<List<OBDData>> loadLogFromCSV(String path) async {
    final file = File(path);
    final content = await file.readAsString();
    final rows = const CsvToListConverter().convert(content);
    if (rows.isEmpty) return [];
    List<OBDData> data = [];
    for (int i = 1; i < rows.length; i++) {
      try {
        final row = rows[i];
        data.add(OBDData(
          timestamp: DateTime.fromMillisecondsSinceEpoch(row[0] as int),
          rpm: row[1] as int, speed: row[2] as int,
          engineLoad: double.tryParse(row[3].toString()) ?? 0,
          coolantTemp: row[4] as int, intakeTemp: row[5] as int,
          maf: double.tryParse(row[6].toString()) ?? 0,
          throttlePos: double.tryParse(row[7].toString()) ?? 0,
          ignitionTiming: double.tryParse(row[8].toString()) ?? 0,
          shortFuelTrim: double.tryParse(row[9].toString()) ?? 0,
          longFuelTrim: double.tryParse(row[10].toString()) ?? 0,
          o2Voltage: double.tryParse(row[11].toString()) ?? 0,
          afr: double.tryParse(row[12].toString()) ?? 14.7,
          vtcActualAngle: double.tryParse(row[14].toString()) ?? 0,
          knockRetard: double.tryParse(row[15].toString()) ?? 0,
          actualIgnition: double.tryParse(row[17].toString()) ?? 0,
        ));
      } catch (e) { continue; }
    }
    return data;
  }
}
''')
print("✅ analyzer_service.dart")

# ============ tuning_service.dart ============
with open('lib/services/tuning_service.dart', 'w') as f:
    f.write('''import '../models/tuning_map.dart';
import 'map_storage_service.dart';

class TuningService {
  Future<TuningMap> getSparkAdvanceMap() async {
    final d = _defaultSpark();
    return await _apply(d);
  }
  TuningMap _defaultSpark() {
    return TuningMap(
      name: 'Spark Advance WOT', address: '0x06EBC',
      rows: 16, cols: 16,
      rpmAxis: [400, 800, 1200, 1600, 2000, 2400, 2800, 3200, 3600, 4000, 4400, 4800, 5200, 5600, 6000, 6400],
      loadAxis: [6, 13, 19, 25, 31, 38, 44, 50, 56, 63, 69, 75, 81, 88, 94, 100],
      units: 'deg', minValue: -5, maxValue: 45,
      data: [
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 21.25, 21.25, 25.3],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 21.25, 21.25, 25.3],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 21.25, 21.25, 25.3],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 21.25, 21.25, 25.3],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 21.25, 21.25, 25.3],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 21.25, 21.25, 25.3],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 21.25, 21.25, 25.3],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 21.25, 21.25, 25.3],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 21.25, 21.25, 25.3],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.96, 16.64, 23.04, 30.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.96, 16.64, 23.04, 30.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.95, 8.96, 16.64, 23.04, 30.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 21.25, 30.08, 30.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 16.64, 21.13, 26.13, 35.59, 35.5],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 14.09, 18.56, 26.37, 28.43, 30.08, 35.5],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.96, 14.21, 24.32, 30.08, 32.77, 35.59, 35.5],
      ],
    );
  }
  Future<TuningMap> getFuelMap() async {
    final d = TuningMap(
      name: 'Fresh Air Rate / VE', address: '0x0A754',
      rows: 16, cols: 15,
      rpmAxis: [400, 800, 1200, 1600, 2000, 2400, 2800, 3200, 3600, 4000, 4400, 4800, 5200, 5600, 6000, 6400],
      loadAxis: [10, 20, 30, 40, 50, 60, 70, 75, 80, 85, 90, 92, 94, 96, 100],
      units: '%', minValue: 15, maxValue: 250,
      data: [
        [64.00, 61.91, 59.82, 51.45, 51.45, 47.27, 41.00, 34.73, 32.64, 29.29, 25.11, 21.77, 18.00, 14.24, 6.71],
        [91.26, 74.35, 68.18, 59.82, 59.82, 55.63, 49.36, 43.09, 41.00, 37.66, 33.47, 30.13, 26.37, 22.60, 15.08],
        [107.12, 86.50, 75.95, 68.87, 68.18, 64.00, 57.73, 51.45, 49.36, 46.02, 41.84, 39.49, 34.73, 30.96, 23.44],
        [121.54, 103.94, 90.59, 81.58, 77.93, 73.04, 66.23, 59.82, 57.73, 54.38, 49.86, 46.86, 43.09, 39.33, 31.80],
        [129.99, 117.54, 104.00, 93.43, 89.17, 82.53, 75.81, 67.86, 65.75, 62.74, 58.56, 55.21, 51.45, 47.69, 40.16],
        [135.66, 128.89, 114.98, 107.49, 101.17, 92.79, 86.56, 78.45, 75.35, 71.80, 66.97, 63.53, 59.82, 56.05, 52.71],
        [140.20, 134.15, 122.14, 115.11, 108.75, 100.35, 93.13, 88.07, 83.96, 78.95, 74.55, 69.98, 65.65, 61.07, 61.07],
        [143.44, 147.12, 151.11, 143.73, 135.47, 125.71, 117.25, 109.65, 102.46, 97.72, 91.96, 86.91, 80.84, 76.80, 72.43],
        [147.62, 151.80, 160.11, 160.88, 156.85, 145.47, 135.90, 124.60, 120.53, 119.43, 114.84, 107.54, 101.61, 95.66, 84.73],
        [151.80, 155.98, 166.95, 171.76, 176.71, 176.71, 172.63, 158.13, 154.66, 143.61, 131.48, 124.60, 117.60, 112.65, 97.95],
        [155.98, 160.17, 174.85, 183.95, 188.13, 186.97, 182.62, 175.94, 165.69, 161.12, 149.06, 140.58, 131.89, 123.63, 114.47],
        [160.17, 164.35, 181.21, 190.75, 197.15, 199.95, 198.86, 187.59, 181.50, 179.13, 167.96, 157.77, 146.63, 138.34, 125.50],
        [164.35, 168.53, 185.25, 198.47, 201.98, 201.98, 195.30, 195.60, 185.31, 186.81, 180.72, 170.89, 163.82, 151.98, 138.27],
        [168.53, 172.71, 189.43, 201.98, 206.16, 206.16, 201.98, 197.83, 193.62, 184.41, 181.20, 170.00, 164.48, 156.16, 146.26],
        [172.71, 176.89, 193.62, 206.16, 210.34, 206.16, 201.98, 193.62, 197.80, 189.43, 185.25, 175.83, 165.99, 149.62, 149.62],
        [214.52, 218.70, 235.43, 247.97, 252.15, 247.97, 247.97, 235.43, 239.61, 231.25, 227.07, 214.52, 206.16, 189.43, 189.43],
      ],
    );
    return await _apply(d);
  }
  Future<TuningMap> getEngineTorqueMap() async {
    final d = TuningMap(
      name: 'Engine Torque', address: '0x07C3C',
      rows: 16, cols: 16,
      rpmAxis: [400, 800, 1200, 1600, 2000, 2400, 2800, 3200, 3600, 4000, 4400, 4800, 5200, 5600, 6000, 6400],
      loadAxis: [6, 13, 19, 25, 31, 38, 44, 50, 56, 63, 69, 75, 81, 88, 94, 100],
      units: 'Nm', minValue: -100, maxValue: 200,
      data: [
        [-30.76, -8.11, 14.45, 37.11, 57.62, 81.25, 101.17, 115.92, 115.92, 115.92, 115.92, 115.92, 115.92, 115.92, 115.92, 115.92],
        [-30.76, -8.11, 14.45, 37.11, 57.62, 81.25, 101.17, 115.92, 115.92, 115.92, 115.92, 115.92, 115.92, 115.92, 115.92, 115.92],
        [-31.93, -11.23, 11.72, 36.82, 61.13, 83.30, 104.39, 124.38, 142.38, 144.63, 144.63, 144.63, 144.63, 144.63, 144.63, 144.63],
        [-32.81, -11.23, 11.43, 36.62, 62.21, 85.16, 108.69, 129.49, 150.39, 150.39, 150.39, 150.39, 150.39, 150.39, 150.39, 150.39],
        [-33.89, -11.62, 12.50, 38.57, 64.55, 87.99, 109.67, 133.01, 154.20, 154.20, 154.20, 154.20, 154.20, 154.20, 154.20, 154.20],
        [-35.84, -13.57, 9.18, 35.25, 61.72, 86.43, 109.77, 134.11, 154.20, 166.60, 166.60, 166.60, 166.60, 166.60, 166.60, 166.60],
        [-37.50, -15.23, 9.08, 36.23, 61.72, 86.52, 111.43, 138.19, 158.01, 167.58, 167.58, 167.58, 167.58, 167.58, 167.58, 167.58],
        [-39.75, -15.14, 10.06, 36.52, 59.86, 84.28, 106.25, 131.93, 165.72, 165.72, 165.72, 165.72, 165.72, 165.72, 165.72, 165.72],
        [-42.87, -15.62, 10.16, 34.57, 58.69, 84.67, 113.87, 135.55, 161.86, 168.55, 168.55, 168.55, 168.55, 168.55, 168.55, 168.55],
        [-46.78, -18.75, 8.69, 34.77, 61.52, 87.40, 110.64, 135.42, 152.05, 178.12, 178.12, 178.12, 178.12, 178.12, 178.12, 178.12],
        [-50.10, -20.90, 7.42, 32.42, 59.47, 85.64, 106.98, 131.35, 155.18, 176.17, 176.17, 176.17, 176.17, 176.17, 176.17, 176.17],
        [-55.18, -23.63, 7.03, 30.86, 55.76, 81.64, 106.25, 129.10, 147.56, 170.21, 175.29, 175.29, 175.29, 175.29, 175.29, 175.29],
        [-60.55, -29.30, 1.95, 28.22, 52.54, 77.15, 100.29, 119.14, 134.99, 151.95, 171.39, 171.39, 171.39, 171.39, 171.39, 171.39],
        [-64.16, -31.84, 0.59, 25.98, 50.59, 75.88, 97.27, 116.89, 133.40, 156.84, 158.01, 158.01, 158.01, 158.01, 158.01, 158.01],
        [-66.70, -33.89, -0.98, 23.83, 48.14, 72.75, 95.61, 114.26, 132.13, 152.25, 152.25, 152.25, 152.25, 152.25, 152.25, 152.25],
        [-64.26, -32.52, -0.88, 23.83, 47.14, 72.75, 95.61, 114.26, 132.13, 152.25, 152.25, 152.25, 152.25, 152.25, 152.25, 152.25],
      ],
    );
    return await _apply(d);
  }
  Future<TuningMap> getVTCMap() async {
    final d = TuningMap(
      name: 'VTC Intake', address: '0x06BF1',
      rows: 8, cols: 8,
      rpmAxis: [800, 1600, 2400, 3200, 4000, 4800, 5600, 6400],
      loadAxis: [0, 15, 30, 45, 60, 75, 90, 100],
      units: 'deg', minValue: 0, maxValue: 40,
      data: [
        [0.0, 15.0, 20.0, 25.0, 30.0, 10.0, 0.0, 35.0],
        [0.0, 20.0, 30.0, 35.0, 35.0, 20.0, 10.0, 25.0],
        [0.0, 25.0, 35.0, 35.0, 35.0, 20.0, 10.0, 25.0],
        [0.0, 10.0, 15.0, 20.0, 20.0, 20.0, 20.0, 20.0],
        [0.0, 10.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0],
        [0.0, 10.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0],
        [0.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
      ],
    );
    return await _apply(d);
  }

  Future<TuningMap> _apply(TuningMap def) async {
    final saved = await MapStorageService.loadMapData(def.address);
    if (saved == null) return def;
    if (saved.length != def.rows) return def;
    if (saved.isNotEmpty && saved[0].length != def.cols) return def;
    return TuningMap(
      name: def.name, address: def.address,
      rows: def.rows, cols: def.cols,
      rpmAxis: def.rpmAxis, loadAxis: def.loadAxis,
      data: saved, units: def.units,
      minValue: def.minValue, maxValue: def.maxValue,
    );
  }
}
''')
print("✅ tuning_service.dart")

# ============ map_storage_service.dart ============
with open('lib/services/map_storage_service.dart', 'w') as f:
    f.write('''import 'dart:convert';
import 'package:shared_preferences/shared_preferences.dart';
import '../models/tuning_map.dart';

class MapStorageService {
  static const String _kPrefix = 'map_edit_';
  static const String _kListKey = 'map_edit_list';
  static SharedPreferences? _prefs;

  static Future<void> _init() async {
    _prefs ??= await SharedPreferences.getInstance();
  }

  static Future<void> saveMap(TuningMap map) async {
    await _init();
    final key = _kPrefix + map.address;
    final jsonStr = jsonEncode({
      'name': map.name, 'address': map.address,
      'rows': map.rows, 'cols': map.cols, 'data': map.data,
      'updatedAt': DateTime.now().toIso8601String(),
    });
    await _prefs!.setString(key, jsonStr);
    final list = _prefs!.getStringList(_kListKey) ?? [];
    if (!list.contains(map.address)) {
      list.add(map.address);
      await _prefs!.setStringList(_kListKey, list);
    }
  }

  static Future<List<List<double>>?> loadMapData(String address) async {
    await _init();
    final jsonStr = _prefs!.getString(_kPrefix + address);
    if (jsonStr == null) return null;
    try {
      final json = jsonDecode(jsonStr) as Map<String, dynamic>;
      final rawData = json['data'] as List;
      return rawData.map<List<double>>((row) {
        return (row as List).map<double>((v) => (v as num).toDouble()).toList();
      }).toList();
    } catch (e) { return null; }
  }

  static Future<bool> hasEdits(String address) async {
    await _init();
    return _prefs!.containsKey(_kPrefix + address);
  }

  static Future<DateTime?> getUpdatedAt(String address) async {
    await _init();
    final jsonStr = _prefs!.getString(_kPrefix + address);
    if (jsonStr == null) return null;
    try {
      final json = jsonDecode(jsonStr) as Map<String, dynamic>;
      return DateTime.tryParse(json['updatedAt'] ?? '');
    } catch (e) { return null; }
  }

  static Future<void> resetMap(String address) async {
    await _init();
    await _prefs!.remove(_kPrefix + address);
    final list = _prefs!.getStringList(_kListKey) ?? [];
    list.remove(address);
    await _prefs!.setStringList(_kListKey, list);
  }

  static Future<void> resetAll() async {
    await _init();
    final list = _prefs!.getStringList(_kListKey) ?? [];
    for (final addr in list) { await _prefs!.remove(_kPrefix + addr); }
    await _prefs!.remove(_kListKey);
  }
}
''')
print("✅ map_storage_service.dart")

# ============ formula_evaluator.dart ============
with open('lib/services/formula_evaluator.dart', 'w') as f:
    f.write('''import 'package:math_expressions/math_expressions.dart';

class FormulaEvaluator {
  final String formulaStr;
  Expression? _expression;

  FormulaEvaluator(this.formulaStr) { _parse(); }

  void _parse() {
    try {
      final parser = Parser();
      String cleaned = formulaStr.replaceAll('X', 'x').replaceAll(' ', '');
      _expression = parser.parse(cleaned);
    } catch (e) { _expression = null; }
  }

  bool get isValid => _expression != null;

  double evaluate(num rawValue) {
    if (_expression == null) return rawValue.toDouble();
    try {
      final cm = ContextModel();
      cm.bindVariable(Variable('x'), Number(rawValue));
      final result = _expression!.evaluate(EvaluationType.REAL, cm);
      if (result is num) return result.toDouble();
      return rawValue.toDouble();
    } catch (e) { return rawValue.toDouble(); }
  }

  static bool isFormulaValid(String formula) {
    try {
      final e = FormulaEvaluator(formula);
      if (!e.isValid) return false;
      final t = e.evaluate(100);
      return !t.isNaN && !t.isInfinite;
    } catch (e) { return false; }
  }
}

class NissanFormulas {
  static const Map<String, String> presets = {
    'Torque (X-32768)/10.24': '(X-32768)/10.24',
    'Force X-32768': 'X-32768',
    'VTC (X-128)/2': '(X-128)/2',
    'Spark (X-16384)/128': '(X-16384)/128',
    'VE X/256': 'X/256',
    'Lambda X/128': 'X/128',
    'RPM X*10': 'X*10',
    'Raw X': 'X',
  };
}
''')
print("✅ formula_evaluator.dart")

# ============ ecu_map_reader.dart ============
with open('lib/services/ecu_map_reader.dart', 'w') as f:
    f.write('''import 'dart:async';
import '../models/custom_map_def.dart';
import 'obd_service.dart';
import 'settings_service.dart';
import 'formula_evaluator.dart';

class MemReadCommand {
  final String id;
  final String description;
  final String Function(int address, int length) builder;
  final String responsePrefix;
  MemReadCommand({required this.id, required this.description,
    required this.builder, required this.responsePrefix});
}

class EcuMapReaderService {
  final OBDService _obd;
  EcuMapReaderService(this._obd);

  static final List<MemReadCommand> knownCommands = [
    MemReadCommand(id: '23_kwp', description: 'KWP2000 (23)',
      builder: (addr, len) {
        final a = addr.toRadixString(16).toUpperCase().padLeft(6, '0');
        final l = len.toRadixString(16).toUpperCase().padLeft(2, '0');
        return '23' + a + l;
      }, responsePrefix: '63'),
    MemReadCommand(id: '21_consult', description: 'Consult II (21)',
      builder: (addr, len) {
        final a = addr.toRadixString(16).toUpperCase().padLeft(4, '0');
        return '21' + a;
      }, responsePrefix: '61'),
    MemReadCommand(id: 'd2_nissan', description: 'Nissan D2',
      builder: (addr, len) {
        final a = addr.toRadixString(16).toUpperCase().padLeft(6, '0');
        final l = len.toRadixString(16).toUpperCase().padLeft(2, '0');
        return 'D2' + a + l;
      }, responsePrefix: '92'),
  ];

  Future<MemReadCommand?> autoDetectCommand({
    int testAddress = 0x7C3C, int testLength = 4,
    void Function(String)? onProgress,
  }) async {
    if (!_obd.isConnected || !_obd.ecuResponds) {
      onProgress?.call('ЭБУ не подключен'); return null;
    }
    for (final cmd in knownCommands) {
      final testCmd = cmd.builder(testAddress, testLength);
      onProgress?.call('→ ' + cmd.description + ': ' + testCmd);
      final response = await _obd.sendCommand(testCmd, timeout: 3000);
      final cleanResp = response.replaceAll(' ', '').toUpperCase();
      onProgress?.call('  ' + response);
      if (cleanResp.startsWith(cmd.responsePrefix) && !cleanResp.startsWith('7F')) {
        onProgress?.call('✅ ' + cmd.id);
        await SettingsService.setMemReadCommand(cmd.id);
        return cmd;
      }
      await Future.delayed(const Duration(milliseconds: 200));
    }
    onProgress?.call('❌ Не найдено');
    return null;
  }

  MemReadCommand? getSavedCommand() {
    final id = SettingsService.memReadCommand;
    if (id == null) return null;
    try { return knownCommands.firstWhere((c) => c.id == id); }
    catch (e) { return null; }
  }

  Future<String> testRawCommand(String rawHexCmd) async {
    if (!_obd.isConnected) return 'Нет подключения';
    return await _obd.sendCommand(rawHexCmd, timeout: 5000);
  }

  Future<EcuMapReadResult?> readMap({
    required CustomMapDef def, MemReadCommand? command,
    void Function(int done, int total)? onProgress,
  }) async {
    final cmd = command ?? getSavedCommand();
    if (cmd == null) return null;
    if (!_obd.isConnected || !_obd.ecuResponds) return null;

    final List<int> allBytes = [];
    int bytesRead = 0;
    int chunkSize = 32;
    while (bytesRead < def.totalBytes) {
      final chunk = (def.totalBytes - bytesRead > chunkSize) ? chunkSize : (def.totalBytes - bytesRead);
      final currentAddr = def.address + bytesRead;
      final readCmd = cmd.builder(currentAddr, chunk);
      final response = await _obd.sendCommand(readCmd, timeout: 5000);
      final clean = response.replaceAll(' ', '').toUpperCase();
      final prefixIdx = clean.indexOf(cmd.responsePrefix);
      if (prefixIdx == -1) return null;
      String dataHex = clean.substring(prefixIdx + cmd.responsePrefix.length);
      if (dataHex.length > chunk * 2 + 6) dataHex = dataHex.substring(6);
      for (int i = 0; i + 1 < dataHex.length && allBytes.length < bytesRead + chunk; i += 2) {
        try { allBytes.add(int.parse(dataHex.substring(i, i + 2), radix: 16)); }
        catch (e) { break; }
      }
      bytesRead = allBytes.length;
      onProgress?.call(bytesRead, def.totalBytes);
      await Future.delayed(const Duration(milliseconds: 100));
    }

    final ev = FormulaEvaluator(def.formula);
    final data = <List<double>>[];
    int idx = 0;
    for (int row = 0; row < def.rows; row++) {
      final rowData = <double>[];
      for (int col = 0; col < def.cols; col++) {
        int rawValue = 0;
        if (def.isUInt16) {
          if (idx + 1 >= allBytes.length) break;
          rawValue = (allBytes[idx] << 8) | allBytes[idx + 1];
          idx += 2;
        } else {
          if (idx >= allBytes.length) break;
          rawValue = allBytes[idx];
          idx += 1;
        }
        rowData.add(ev.evaluate(rawValue));
      }
      if (rowData.isNotEmpty) data.add(rowData);
    }
    return EcuMapReadResult(def: def, data: data, readAt: DateTime.now());
  }
}
''')
print("✅ ecu_map_reader.dart")

# ============ export_service.dart ============
with open('lib/services/export_service.dart', 'w') as f:
    f.write('''import 'dart:convert';
import 'dart:io';
import 'package:path_provider/path_provider.dart';
import 'package:intl/intl.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';

class ExportService {
  Future<String> exportToWinOLS(TuningMap map) async {
    final dir = await getApplicationDocumentsDirectory();
    final ts = DateFormat('yyyyMMdd_HHmmss').format(DateTime.now());
    final path = dir.path + '/' + map.name.replaceAll(' ', '_') + '_' + ts + '.ols';
    StringBuffer sb = StringBuffer();
    sb.writeln('# WinOLS');
    sb.writeln('MAP_NAME=' + map.name);
    sb.writeln('MAP_ADDRESS=' + map.address);
    sb.writeln('X_AXIS=' + map.loadAxis.join(','));
    sb.writeln('Y_AXIS=' + map.rpmAxis.join(','));
    for (int i = 0; i < map.data.length; i++) {
      sb.writeln('ROW_' + i.toString() + '=' + map.data[i].map((v) => v.toStringAsFixed(2)).join(','));
    }
    await File(path).writeAsString(sb.toString());
    return path;
  }

  Future<String> exportToEcuEdit(TuningMap map) async {
    final dir = await getApplicationDocumentsDirectory();
    final ts = DateFormat('yyyyMMdd_HHmmss').format(DateTime.now());
    final path = dir.path + '/' + map.name.replaceAll(' ', '_') + '_' + ts + '.txt';
    StringBuffer sb = StringBuffer();
    sb.writeln('[MAP]');
    sb.writeln('Name=' + map.name);
    sb.writeln('Address=' + map.address);
    sb.writeln('[DATA]');
    for (int i = 0; i < map.data.length; i++) {
      sb.write(map.rpmAxis[i].toStringAsFixed(0).padLeft(5));
      for (var v in map.data[i]) sb.write(v.toStringAsFixed(2).padLeft(8));
      sb.writeln();
    }
    await File(path).writeAsString(sb.toString());
    return path;
  }

  Future<String> exportToJson(AnalysisResult result, TuningMap orig, TuningMap upd) async {
    final dir = await getApplicationDocumentsDirectory();
    final ts = DateFormat('yyyyMMdd_HHmmss').format(DateTime.now());
    final path = dir.path + '/tuning_' + ts + '.json';
    final data = {
      'meta': {'app': 'v5', 'exported': DateTime.now().toIso8601String()},
      'original': orig.toJson(), 'updated': upd.toJson(),
    };
    await File(path).writeAsString(const JsonEncoder.withIndent('  ').convert(data));
    return path;
  }

  Future<String> exportHexPatch(TuningMap map) async {
    final dir = await getApplicationDocumentsDirectory();
    final ts = DateFormat('yyyyMMdd_HHmmss').format(DateTime.now());
    final path = dir.path + '/' + map.name.replaceAll(' ', '_') + '_' + ts + '.hex';
    StringBuffer sb = StringBuffer();
    int baseAddr = int.parse(map.address.replaceAll('0x', ''), radix: 16);
    for (int i = 0; i < map.data.length; i++) {
      for (int j = 0; j < map.data[i].length; j++) {
        int addr = baseAddr + (i * map.cols + j) * 2;
        sb.writeln(addr.toRadixString(16).padLeft(8, '0').toUpperCase() + ': ' + map.data[i][j].toStringAsFixed(2));
      }
    }
    await File(path).writeAsString(sb.toString());
    return path;
  }
}
''')
print("✅ export_service.dart")

# ============ profile_service.dart ============
with open('lib/services/profile_service.dart', 'w') as f:
    f.write('''import 'package:shared_preferences/shared_preferences.dart';
import 'package:uuid/uuid.dart';
import '../models/vehicle_profile.dart';

class ProfileService {
  static const String _kProfiles = 'vehicle_profiles';
  static const String _kActive = 'active_profile_id';
  final Uuid _uuid = const Uuid();

  Future<List<VehicleProfile>> getAll() async {
    final prefs = await SharedPreferences.getInstance();
    final list = prefs.getStringList(_kProfiles) ?? [];
    return list.map((s) => VehicleProfile.fromJsonString(s)).toList();
  }
  Future<VehicleProfile?> getActive() async {
    final prefs = await SharedPreferences.getInstance();
    final id = prefs.getString(_kActive);
    if (id == null) return null;
    final profiles = await getAll();
    try { return profiles.firstWhere((p) => p.id == id); }
    catch (e) { return null; }
  }
  Future<void> setActive(String id) async {
    final prefs = await SharedPreferences.getInstance();
    await prefs.setString(_kActive, id);
  }
  Future<VehicleProfile> create({
    required String name, required String make, required String model,
    required String year, required String engine, double displacement = 2.0,
    String ecuFirmware = '',
  }) async {
    final profile = VehicleProfile(
      id: _uuid.v4(), name: name, make: make, model: model,
      year: year, engine: engine, displacement: displacement,
      ecuFirmware: ecuFirmware, createdAt: DateTime.now(),
    );
    final profiles = await getAll();
    profiles.add(profile);
    await _save(profiles);
    if (profiles.length == 1) await setActive(profile.id);
    return profile;
  }
  Future<void> delete(String id) async {
    final profiles = await getAll();
    profiles.removeWhere((p) => p.id == id);
    await _save(profiles);
  }
  Future<void> _save(List<VehicleProfile> profiles) async {
    final prefs = await SharedPreferences.getInstance();
    await prefs.setStringList(_kProfiles, profiles.map((p) => p.toJsonString()).toList());
  }
}
''')
print("✅ profile_service.dart")

# ============ performance_service.dart ============
with open('lib/services/performance_service.dart', 'w') as f:
    f.write('''import 'dart:async';
import '../models/obd_data.dart';

class PerformanceService {
  bool _isRunning = false;
  bool _isWaiting = false;
  DateTime? _startTime;
  double _time0to60 = 0, _time0to100 = 0, _time400m = 0;
  double _maxSpeed = 0;
  int _maxRPM = 0;
  double _maxHP = 0, _distance = 0;
  DateTime? _lastTime;
  final StreamController _controller = StreamController.broadcast();
  Stream get runStream => _controller.stream;
  bool get isRunning => _isRunning;
  bool get isWaiting => _isWaiting;
  double get time0to60 => _time0to60;
  double get time0to100 => _time0to100;
  double get time400m => _time400m;
  double get maxSpeed => _maxSpeed;
  int get maxRPM => _maxRPM;
  double get maxHP => _maxHP;
  double get maxTorque => 0;
  void startWaiting() { _isWaiting = true; _reset(); }
  void stop() { _isRunning = false; _isWaiting = false; }
  void _reset() {
    _startTime = null;
    _time0to60 = 0; _time0to100 = 0; _time400m = 0;
    _maxSpeed = 0; _maxRPM = 0; _maxHP = 0;
    _distance = 0; _lastTime = null;
  }
  void processData(OBDData data) {
    if (!_isWaiting && !_isRunning) return;
    if (_isWaiting && data.speed >= 1 && data.throttlePos > 30) {
      _isRunning = true;
      _isWaiting = false;
      _startTime = DateTime.now();
    }
    if (!_isRunning) return;
    if (data.speed > _maxSpeed) _maxSpeed = data.speed.toDouble();
    if (data.rpm > _maxRPM) _maxRPM = data.rpm;
    if (data.calculatedHP > _maxHP) _maxHP = data.calculatedHP;
    final elapsed = DateTime.now().difference(_startTime!).inMilliseconds / 1000.0;
    if (_lastTime != null) {
      final dt = DateTime.now().difference(_lastTime!).inMilliseconds / 1000.0;
      _distance += (data.speed / 3.6) * dt;
    }
    _lastTime = DateTime.now();
    if (_time0to60 == 0 && data.speed >= 60) _time0to60 = elapsed;
    if (_time0to100 == 0 && data.speed >= 100) _time0to100 = elapsed;
    if (_time400m == 0 && _distance >= 400) _time400m = elapsed;
    _controller.add(null);
    if (_time400m > 0 && data.speed >= 200) stop();
  }
  void dispose() { _controller.close(); }
}
''')
print("✅ performance_service.dart")

print()
print("=" * 60)
print("ПРОВЕРКА:")
print("=" * 60)
!ls -la lib/services/

✅ nissan_pid_library.dart
✅ settings_service.dart
✅ obd_service.dart (с pause polling + MAF/Speed multipliers)
✅ logger_service.dart
✅ dtc_database.dart
✅ dtc_service.dart
✅ alert_service.dart (со snapshot + explanation)
✅ analyzer_service.dart
✅ tuning_service.dart
✅ map_storage_service.dart
✅ formula_evaluator.dart
✅ ecu_map_reader.dart
✅ export_service.dart
✅ profile_service.dart
✅ performance_service.dart

ПРОВЕРКА:
total 116
drwxr-xr-x 2 root root  4096 Aug 15 02:15 .
drwxr-xr-x 6 root root  4096 Aug 15 02:15 ..
-rw-r--r-- 1 root root  6520 Aug 15 02:29 alert_service.dart
-rw-r--r-- 1 root root  5784 Aug 15 02:29 analyzer_service.dart
-rw-r--r-- 1 root root  2721 Aug 15 02:29 dtc_database.dart
-rw-r--r-- 1 root root  2120 Aug 15 02:29 dtc_service.dart
-rw-r--r-- 1 root root  4907 Aug 15 02:29 ecu_map_reader.dart
-rw-r--r-- 1 root root  2979 Aug 15 02:29 export_service.dart
-rw-r--r-- 1 root root  1435 Aug 15 02:29 formula_evaluator.dart
-rw-r--r-- 1 root root  2958 Aug 15 02:29 lo

In [4]:
# @title 🎨 Ячейка 4/5: Виджеты + Экраны (ЧАСТЬ A)
import os
os.chdir('/content/nissan_logger_pro_v5')

# ============ main.dart ============
with open('lib/main.dart', 'w') as f:
    f.write('''import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import 'screens/home_screen.dart';
import 'services/settings_service.dart';

void main() async {
  WidgetsFlutterBinding.ensureInitialized();
  FlutterError.onError = (d) => FlutterError.presentError(d);
  await SettingsService.init();
  await SystemChrome.setPreferredOrientations([
    DeviceOrientation.portraitUp,
    DeviceOrientation.landscapeLeft,
    DeviceOrientation.landscapeRight,
  ]);
  runApp(const NissanLoggerApp());
}

class NissanLoggerApp extends StatelessWidget {
  const NissanLoggerApp({super.key});
  @override
  Widget build(BuildContext context) {
    return MaterialApp(
      title: 'Nissan Logger Pro v5',
      debugShowCheckedModeBanner: false,
      theme: ThemeData(
        brightness: Brightness.dark,
        primarySwatch: Colors.blue,
        scaffoldBackgroundColor: const Color(0xFF1A1A2E),
        cardColor: const Color(0xFF16213E),
        colorScheme: const ColorScheme.dark(
          primary: Color(0xFFE94560),
          secondary: Color(0xFF0F3460),
          surface: Color(0xFF16213E),
        ),
        elevatedButtonTheme: ElevatedButtonThemeData(
          style: ElevatedButton.styleFrom(foregroundColor: Colors.white),
        ),
        textButtonTheme: TextButtonThemeData(
          style: TextButton.styleFrom(foregroundColor: Colors.white),
        ),
      ),
      home: const HomeScreen(),
    );
  }
}
''')
print("✅ main.dart")

# ============ fps_indicator.dart ============
with open('lib/widgets/fps_indicator.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'package:flutter/material.dart';
import '../services/obd_service.dart';

class FpsIndicator extends StatefulWidget {
  final OBDService obdService;
  const FpsIndicator({super.key, required this.obdService});
  @override
  State<FpsIndicator> createState() => _FpsIndicatorState();
}

class _FpsIndicatorState extends State<FpsIndicator> {
  Timer? _t;
  @override
  void initState() {
    super.initState();
    _t = Timer.periodic(const Duration(seconds: 1), (_) { if (mounted) setState(() {}); });
  }
  @override
  void dispose() { _t?.cancel(); super.dispose(); }
  @override
  Widget build(BuildContext context) {
    final conn = widget.obdService.isConnected;
    final ecu = widget.obdService.ecuResponds;
    final fps = widget.obdService.pollFps;
    Color color; IconData icon; String text;
    if (!conn) { color = Colors.red; icon = Icons.bluetooth_disabled; text = 'OFFLINE'; }
    else if (!ecu) { color = Colors.orange; icon = Icons.bluetooth_connected; text = 'BT ONLY'; }
    else {
      color = fps >= 5 ? Colors.green : (fps >= 3 ? Colors.orange : Colors.red);
      icon = Icons.bluetooth_connected;
      text = fps.toString() + ' Hz';
    }
    return Padding(padding: const EdgeInsets.only(right: 8),
      child: Row(children: [
        Icon(icon, color: color, size: 18),
        const SizedBox(width: 4),
        Text(text, style: TextStyle(color: color, fontSize: 12, fontWeight: FontWeight.bold)),
      ]));
  }
}
''')
print("✅ fps_indicator.dart")

# ============ map_table_view.dart ============
with open('lib/widgets/map_table_view.dart', 'w') as f:
    f.write('''import 'package:flutter/material.dart';
import 'package:intl/intl.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';
import '../services/map_storage_service.dart';

class MapTableView extends StatefulWidget {
  final TuningMap originalMap;
  final TuningMap? updatedMap;
  final List<MapCell> changes;
  final bool isFullscreen;
  final VoidCallback? onFullscreenTap;
  final VoidCallback? onSaved;
  const MapTableView({
    super.key, required this.originalMap, this.updatedMap,
    required this.changes, this.isFullscreen = false,
    this.onFullscreenTap, this.onSaved,
  });
  @override
  State<MapTableView> createState() => _MapTableViewState();
}

class _MapTableViewState extends State<MapTableView> {
  int? _selRow;
  int? _selCol;
  MapCell? _selChange;
  double _cellSize = 52.0;
  Map<String, MapCell> _changesMap = {};
  bool _hasSavedEdits = false;
  DateTime? _savedAt;

  @override
  void initState() {
    super.initState();
    _rebuild();
    _cellSize = widget.isFullscreen ? 62.0 : 52.0;
    _check();
  }
  @override
  void didUpdateWidget(MapTableView old) { super.didUpdateWidget(old); _rebuild(); _check(); }

  void _rebuild() {
    _changesMap.clear();
    for (var c in widget.changes) {
      _changesMap[c.rpmIndex.toString() + '_' + c.loadIndex.toString()] = c;
    }
  }

  Future<void> _check() async {
    final has = await MapStorageService.hasEdits(widget.originalMap.address);
    final at = await MapStorageService.getUpdatedAt(widget.originalMap.address);
    if (mounted) setState(() { _hasSavedEdits = has; _savedAt = at; });
  }

  Future<void> _save() async {
    if (widget.updatedMap == null) { _snack('Нет обновлённой карты', Colors.orange); return; }
    final ok = await showDialog<bool>(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: const Text('Сохранить правки?'),
      content: Text('Правки станут дефолтом.\\nПравок: ' + widget.changes.length.toString(),
        style: const TextStyle(color: Colors.white70)),
      actions: [
        TextButton(onPressed: () => Navigator.pop(c, false),
          style: TextButton.styleFrom(foregroundColor: Colors.white),
          child: const Text('Отмена')),
        TextButton(onPressed: () => Navigator.pop(c, true),
          style: TextButton.styleFrom(foregroundColor: Colors.green),
          child: const Text('СОХРАНИТЬ')),
      ]));
    if (ok != true) return;
    await MapStorageService.saveMap(widget.updatedMap!);
    _snack('Сохранено', Colors.green);
    await _check();
    widget.onSaved?.call();
  }

  Future<void> _reset({bool all = false}) async {
    final ok = await showDialog<bool>(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: Text(all ? 'Сбросить ВСЕ карты?' : 'Сбросить эту?'),
      actions: [
        TextButton(onPressed: () => Navigator.pop(c, false),
          style: TextButton.styleFrom(foregroundColor: Colors.white),
          child: const Text('Отмена')),
        TextButton(onPressed: () => Navigator.pop(c, true),
          style: TextButton.styleFrom(foregroundColor: Colors.orange),
          child: const Text('СБРОСИТЬ')),
      ]));
    if (ok != true) return;
    if (all) await MapStorageService.resetAll();
    else await MapStorageService.resetMap(widget.originalMap.address);
    _snack('Сброшено', Colors.green);
    await _check();
    widget.onSaved?.call();
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(
      content: Text(m, style: const TextStyle(color: Colors.white)),
      backgroundColor: c));
  }

  Color _cellColor(int r, int l) {
    final change = _changesMap[r.toString() + '_' + l.toString()];
    if (change == null) {
      final val = widget.originalMap.data[r][l];
      final range = widget.originalMap.maxValue - widget.originalMap.minValue;
      final norm = ((val - widget.originalMap.minValue) / range).clamp(0.0, 1.0);
      return Color.lerp(const Color(0xFF1A2E3E), const Color(0xFF2A5578), norm)!;
    }
    final pct = change.deltaPercent.abs();
    if (pct < 3) return change.delta > 0 ? Colors.yellow.shade800 : Colors.lightBlue.shade800;
    if (pct < 8) return change.delta > 0 ? Colors.orange.shade800 : Colors.blue.shade800;
    return change.delta > 0 ? Colors.red.shade800 : Colors.blueAccent.shade700;
  }

  @override
  Widget build(BuildContext context) {
    final map = widget.originalMap;
    return Column(children: [
      Container(padding: const EdgeInsets.all(6), color: const Color(0xFF16213E),
        child: Column(children: [
          Row(children: [
            const Icon(Icons.grid_on, color: Colors.cyan, size: 16),
            const SizedBox(width: 4),
            Expanded(child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
              Text(map.name, style: const TextStyle(
                color: Colors.cyan, fontWeight: FontWeight.bold, fontSize: 12), overflow: TextOverflow.ellipsis),
              if (_hasSavedEdits) Row(children: [
                const Icon(Icons.bookmark, color: Colors.orange, size: 10),
                const SizedBox(width: 3),
                Text('С правками • ' + (_savedAt != null ? DateFormat('dd.MM HH:mm').format(_savedAt!) : ''),
                  style: const TextStyle(color: Colors.orange, fontSize: 9)),
              ]),
            ])),
            Text(map.rows.toString() + 'x' + map.cols.toString(),
              style: const TextStyle(color: Colors.white54, fontSize: 10)),
            const SizedBox(width: 6),
            if (!widget.isFullscreen && widget.onFullscreenTap != null)
              GestureDetector(onTap: widget.onFullscreenTap, child: Container(
                padding: const EdgeInsets.all(4),
                decoration: BoxDecoration(color: Colors.cyan.withOpacity(0.3), borderRadius: BorderRadius.circular(3)),
                child: const Icon(Icons.fullscreen, color: Colors.cyan, size: 18))),
            if (widget.isFullscreen) ...[
              GestureDetector(
                onTap: () => setState(() => _cellSize = (_cellSize - 8).clamp(32.0, 90.0)),
                child: const Padding(padding: EdgeInsets.all(4),
                  child: Icon(Icons.remove_circle_outline, color: Colors.white70, size: 20))),
              Text(_cellSize.toInt().toString(), style: const TextStyle(color: Colors.white54, fontSize: 10)),
              GestureDetector(
                onTap: () => setState(() => _cellSize = (_cellSize + 8).clamp(32.0, 90.0)),
                child: const Padding(padding: EdgeInsets.all(4),
                  child: Icon(Icons.add_circle_outline, color: Colors.white70, size: 20))),
            ],
          ]),
          if (widget.changes.isNotEmpty || _hasSavedEdits)
            Padding(padding: const EdgeInsets.only(top: 6), child: Row(children: [
              if (widget.changes.isNotEmpty && widget.updatedMap != null)
                Expanded(child: ElevatedButton.icon(
                  onPressed: _save,
                  icon: const Icon(Icons.save, size: 14),
                  label: const Text('Сохранить', style: TextStyle(fontSize: 11, fontWeight: FontWeight.bold)),
                  style: ElevatedButton.styleFrom(backgroundColor: Colors.green, foregroundColor: Colors.white,
                    padding: const EdgeInsets.symmetric(vertical: 8)))),
              if (widget.changes.isNotEmpty && _hasSavedEdits) const SizedBox(width: 6),
              if (_hasSavedEdits) Expanded(child: PopupMenuButton<String>(
                onSelected: (v) { if (v == 'one') _reset(); else _reset(all: true); },
                itemBuilder: (c) => [
                  const PopupMenuItem(value: 'one', child: Text('Сбросить эту')),
                  const PopupMenuItem(value: 'all', child: Text('Сбросить ВСЕ')),
                ],
                child: Container(padding: const EdgeInsets.symmetric(vertical: 8, horizontal: 12),
                  decoration: BoxDecoration(color: Colors.orange, borderRadius: BorderRadius.circular(4)),
                  child: Row(mainAxisAlignment: MainAxisAlignment.center, children: const [
                    Icon(Icons.restore, color: Colors.white, size: 16), SizedBox(width: 4),
                    Text('Сброс', style: TextStyle(color: Colors.white, fontSize: 11, fontWeight: FontWeight.bold)),
                  ])))),
            ])),
        ])),
      if (_selChange != null) Container(
        padding: const EdgeInsets.all(6), margin: const EdgeInsets.all(4),
        decoration: BoxDecoration(color: Colors.cyan.withOpacity(0.15),
          borderRadius: BorderRadius.circular(6),
          border: Border.all(color: Colors.cyan.withOpacity(0.5))),
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
          Row(children: [
            Text('RPM ' + _selChange!.rpm.toInt().toString() +
              ' | Load ' + _selChange!.load.toInt().toString() + '%',
              style: const TextStyle(color: Colors.cyan, fontWeight: FontWeight.bold, fontSize: 12)),
            const Spacer(),
            IconButton(icon: const Icon(Icons.close, size: 14),
              onPressed: () => setState(() { _selChange = null; _selRow = null; _selCol = null; }),
              padding: EdgeInsets.zero, constraints: const BoxConstraints()),
          ]),
          Text(_selChange!.currentValue.toStringAsFixed(2) + ' → ' + _selChange!.suggestedValue.toStringAsFixed(2),
            style: TextStyle(color: _selChange!.delta > 0 ? Colors.green : Colors.orange, fontSize: 13)),
          Text(_selChange!.reason, style: const TextStyle(color: Colors.white70, fontSize: 10)),
        ])),
      Expanded(child: InteractiveViewer(constrained: false, minScale: 0.5, maxScale: 3.0,
        boundaryMargin: const EdgeInsets.all(20), child: _buildTable(map))),
    ]);
  }

  Widget _buildTable(TuningMap map) {
    final axisSize = _cellSize;
    final headerH = 28.0;
    return Padding(padding: const EdgeInsets.all(4), child: Column(
      crossAxisAlignment: CrossAxisAlignment.start, children: [
        Row(children: [
          Container(width: axisSize, height: headerH,
            decoration: BoxDecoration(color: const Color(0xFF0F3460),
              border: Border.all(color: Colors.white24, width: 0.5)),
            child: const Center(child: Text('RPM/%', style: TextStyle(fontSize: 9, color: Colors.white70)))),
          ...List.generate(map.cols, (j) => Container(width: _cellSize, height: headerH,
            decoration: BoxDecoration(color: const Color(0xFF0F3460),
              border: Border.all(color: Colors.white24, width: 0.5)),
            child: Center(child: Text(map.loadAxis[j].toStringAsFixed(0),
              style: const TextStyle(fontSize: 10, color: Colors.white70))))),
        ]),
        ...List.generate(map.rows, (i) => Row(children: [
          Container(width: axisSize, height: _cellSize,
            decoration: BoxDecoration(color: const Color(0xFF0F3460),
              border: Border.all(color: Colors.white24, width: 0.5)),
            child: Center(child: Text(map.rpmAxis[i].toStringAsFixed(0),
              style: const TextStyle(fontSize: 10, color: Colors.white70)))),
          ...List.generate(map.cols, (j) => _buildCell(i, j, map)),
        ])),
      ]));
  }

  Widget _buildCell(int i, int j, TuningMap map) {
    final change = _changesMap[i.toString() + '_' + j.toString()];
    final hasChange = change != null;
    final isSel = _selRow == i && _selCol == j;
    final origVal = map.data[i][j];
    final updVal = widget.updatedMap?.data[i][j] ?? origVal;
    return GestureDetector(onTap: () => setState(() {
      _selRow = i; _selCol = j; _selChange = change;
    }),
    child: Container(width: _cellSize, height: _cellSize,
      decoration: BoxDecoration(color: _cellColor(i, j),
        border: Border.all(color: isSel ? Colors.cyan : hasChange ? Colors.white54 : Colors.white12,
          width: isSel ? 2 : (hasChange ? 1 : 0.5))),
      child: hasChange ? Column(mainAxisAlignment: MainAxisAlignment.center, children: [
        Text(origVal.toStringAsFixed(1),
          style: TextStyle(fontSize: 9, color: Colors.white.withOpacity(0.7),
            decoration: TextDecoration.lineThrough)),
        Text(updVal.toStringAsFixed(1),
          style: TextStyle(fontSize: 11, color: change.delta > 0 ? Colors.greenAccent : Colors.yellowAccent,
            fontWeight: FontWeight.bold)),
      ]) : Center(child: Text(origVal.toStringAsFixed(1),
        style: const TextStyle(fontSize: 10, color: Colors.white70)))));
  }
}

class MapFullscreenView extends StatelessWidget {
  final TuningMap originalMap;
  final TuningMap? updatedMap;
  final List<MapCell> changes;
  final VoidCallback? onSaved;
  const MapFullscreenView({super.key, required this.originalMap, this.updatedMap,
    required this.changes, this.onSaved});
  @override
  Widget build(BuildContext context) {
    return Scaffold(
      backgroundColor: const Color(0xFF1A1A2E),
      appBar: AppBar(title: Text(originalMap.name),
        backgroundColor: const Color(0xFF16213E),
        leading: IconButton(icon: const Icon(Icons.close), onPressed: () => Navigator.pop(context))),
      body: MapTableView(originalMap: originalMap, updatedMap: updatedMap,
        changes: changes, isFullscreen: true, onSaved: onSaved));
  }
}
''')
print("✅ map_table_view.dart")

# ============ home_screen.dart ============
with open('lib/screens/home_screen.dart', 'w') as f:
    f.write('''import 'package:flutter/material.dart';
import '../services/obd_service.dart';
import '../services/logger_service.dart';
import '../services/alert_service.dart';
import '../services/profile_service.dart';
import '../services/performance_service.dart';
import '../services/settings_service.dart';
import 'dashboard_screen.dart';
import 'graph_screen.dart';
import 'log_graph_screen.dart';
import 'logging_screen.dart';
import 'events_screen.dart';
import 'dtc_screen.dart';
import 'analyzer_screen.dart';
import 'ecu_read_screen.dart';
import 'service_screen.dart';
import 'performance_screen.dart';
import 'export_screen.dart';
import 'custom_pid_screen.dart';
import 'profile_screen.dart';
import 'settings_screen.dart';
import 'terminal_screen.dart';

class HomeScreen extends StatefulWidget {
  const HomeScreen({super.key});
  @override
  State<HomeScreen> createState() => _HomeScreenState();
}

class _NavItem {
  final IconData icon;
  final String label;
  final Widget screen;
  _NavItem(this.icon, this.label, this.screen);
}

class _HomeScreenState extends State<HomeScreen> {
  int _currentIndex = 0;
  final OBDService _obd = OBDService();
  final LoggerService _logger = LoggerService();
  final AlertService _alert = AlertService();
  final ProfileService _profile = ProfileService();
  final PerformanceService _perf = PerformanceService();
  late final List<_NavItem> _items;

  @override
  void initState() {
    super.initState();
    _obd.loadTripFuel();
    _obd.dataStream.listen((data) {
      _logger.addData(data);
      _alert.checkData(data);
      _perf.processData(data);
    });
    _tryAutoConnect();
    _items = [
      _NavItem(Icons.speed, 'Приборы', DashboardScreen(obdService: _obd, alertService: _alert)),
      _NavItem(Icons.show_chart, 'Графики', GraphScreen(obdService: _obd)),
      _NavItem(Icons.timeline, 'ЛогГраф', const LogGraphScreen()),
      _NavItem(Icons.fiber_manual_record, 'Лог', LoggingScreen(obdService: _obd, loggerService: _logger)),
      _NavItem(Icons.notifications_active, 'События', EventsScreen(alertService: _alert)),
      _NavItem(Icons.warning, 'DTC', DTCScreen(obdService: _obd)),
      _NavItem(Icons.analytics, 'Анализ', AnalyzerScreen(obdService: _obd)),
      _NavItem(Icons.memory, 'ЭБУ Карты', EcuReadScreen(obdService: _obd)),
      _NavItem(Icons.build, 'Сервис', ServiceScreen(obdService: _obd)),
      _NavItem(Icons.timer, 'Замер', PerformanceScreen(obdService: _obd, performanceService: _perf)),
      _NavItem(Icons.upload_file, 'Экспорт', const ExportScreen()),
      _NavItem(Icons.code, 'PID', CustomPIDScreen(obdService: _obd)),
      _NavItem(Icons.directions_car, 'Авто', ProfileScreen(profileService: _profile)),
      _NavItem(Icons.terminal, 'Терминал', TerminalScreen(obdService: _obd)),
      _NavItem(Icons.settings, 'Настройки', SettingsScreen(obdService: _obd, alertService: _alert)),
    ];
  }

  Future<void> _tryAutoConnect() async {
    await Future.delayed(const Duration(seconds: 2));
    if (SettingsService.autoConnect) {
      final lastAddr = SettingsService.lastBtDevice;
      if (lastAddr != null && !_obd.isConnected) {
        final ok = await _obd.connect(lastAddr);
        if (ok) await _obd.initECU(useCache: true);
      }
    }
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      body: _items[_currentIndex].screen,
      bottomNavigationBar: Container(height: 72,
        decoration: const BoxDecoration(color: Color(0xFF16213E),
          border: Border(top: BorderSide(color: Color(0xFF0F3460), width: 0.5))),
        child: SingleChildScrollView(scrollDirection: Axis.horizontal,
          child: Row(children: List.generate(_items.length, (i) {
            final item = _items[i];
            final isSel = i == _currentIndex;
            return InkWell(onTap: () => setState(() => _currentIndex = i),
              child: Container(width: 78, padding: const EdgeInsets.symmetric(vertical: 8),
                decoration: isSel ? const BoxDecoration(border: Border(
                  top: BorderSide(color: Color(0xFFE94560), width: 3))) : null,
                child: Column(mainAxisAlignment: MainAxisAlignment.center, children: [
                  Icon(item.icon, color: isSel ? const Color(0xFFE94560) : Colors.white54, size: 22),
                  const SizedBox(height: 4),
                  Text(item.label, style: TextStyle(
                    color: isSel ? const Color(0xFFE94560) : Colors.white54,
                    fontSize: 10, fontWeight: isSel ? FontWeight.bold : FontWeight.normal)),
                ])));
          })))));
  }

  @override
  void dispose() {
    _obd.dispose();
    _alert.dispose();
    _perf.dispose();
    super.dispose();
  }
}
''')
print("✅ home_screen.dart")

# ============ dashboard_screen.dart ============
with open('lib/screens/dashboard_screen.dart', 'w') as f:
    f.write('''import 'package:flutter/material.dart';
import '../models/obd_data.dart';
import '../models/alert.dart';
import '../services/obd_service.dart';
import '../services/alert_service.dart';
import '../widgets/fps_indicator.dart';

class DashboardScreen extends StatefulWidget {
  final OBDService obdService;
  final AlertService alertService;
  const DashboardScreen({super.key, required this.obdService, required this.alertService});
  @override
  State<DashboardScreen> createState() => _DashboardScreenState();
}

class _DashboardScreenState extends State<DashboardScreen> {
  OBDData _data = OBDData(timestamp: DateTime.now());
  List<Alert> _alerts = [];

  @override
  void initState() {
    super.initState();
    widget.obdService.dataStream.listen((data) {
      if (mounted) setState(() {
        _data = data;
        _alerts = widget.alertService.recentAlerts.take(3).toList();
      });
    });
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Приборная панель'),
        backgroundColor: const Color(0xFF16213E),
        actions: [FpsIndicator(obdService: widget.obdService)]),
      body: SingleChildScrollView(padding: const EdgeInsets.all(12), child: Column(
        crossAxisAlignment: CrossAxisAlignment.stretch, children: [
          if (_alerts.isNotEmpty) Card(
            color: _alerts.first.level == AlertLevel.danger
              ? Colors.red.withOpacity(0.3) : Colors.orange.withOpacity(0.3),
            child: Padding(padding: const EdgeInsets.all(10), child: Column(
              crossAxisAlignment: CrossAxisAlignment.start,
              children: _alerts.map((a) => Text(a.message,
                style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 12, color: Colors.white))).toList()))),
          if (_alerts.isNotEmpty) const SizedBox(height: 8),
          Row(children: [
            Expanded(child: _gauge('RPM', _data.rpm.toString(), _rpmColor(_data.rpm))),
            const SizedBox(width: 8),
            Expanded(child: _gauge('KM/H', _data.speed.toString(), Colors.blue)),
          ]),
          const SizedBox(height: 8),
          GridView.count(shrinkWrap: true, physics: const NeverScrollableScrollPhysics(),
            crossAxisCount: 3, childAspectRatio: 1.8, crossAxisSpacing: 6, mainAxisSpacing: 6, children: [
              _param('Зажигание', _data.actualIgnition.toStringAsFixed(1), '°', _timingColor(_data.actualIgnition)),
              _param('Knock', _data.knockRetard.toStringAsFixed(1), '°', _knockColor(_data.knockRetard)),
              _param('VTC', _data.vtcActualAngle.toStringAsFixed(1), '°', Colors.cyan),
              _param('Нагрузка', _data.engineLoad.toStringAsFixed(0), '%', Colors.orange),
              _param('Дроссель', _data.throttlePos.toStringAsFixed(0), '%', Colors.green),
              _param('MAF', _data.maf.toStringAsFixed(2), 'g/s', Colors.purple),
              _param('AFR', _data.afr.toStringAsFixed(2), '', _afrColor(_data.afr)),
              _param('ОЖ', _data.coolantTemp.toString(), '°C', _tempColor(_data.coolantTemp)),
              _param('Впуск', _data.intakeTemp.toString(), '°C', Colors.cyan),
              _param('Батарея', _data.batteryVoltage.toStringAsFixed(2), 'V', Colors.yellow),
              _param('Форсунки', _data.injectorPulseWidth.toStringAsFixed(2), 'ms', Colors.amber),
              _param(_data.speed >= 5 ? 'L/100км' : 'L/ч',
                _data.speed >= 5 && _data.fuelL100km > 0
                  ? _data.fuelL100km.toStringAsFixed(1)
                  : _data.fuelFlowLph.toStringAsFixed(1), '', Colors.pink),
            ]),
          const SizedBox(height: 8),
          Card(color: const Color(0xFF16213E),
            child: Padding(padding: const EdgeInsets.all(12), child: Column(children: [
              Row(mainAxisAlignment: MainAxisAlignment.center, children: const [
                Icon(Icons.local_gas_station, color: Colors.pink, size: 18),
                SizedBox(width: 6),
                Text('РАСХОД ТОПЛИВА', style: TextStyle(color: Colors.white70, fontSize: 12)),
              ]),
              const SizedBox(height: 8),
              Row(children: [
                Expanded(child: Column(children: [
                  const Text('Мгновенный', style: TextStyle(color: Colors.white54, fontSize: 11)),
                  Text(_data.fuelFlowLph.toStringAsFixed(2) + ' L/ч',
                    style: const TextStyle(color: Colors.pink, fontSize: 18, fontWeight: FontWeight.bold)),
                  if (_data.speed >= 5) Text(_data.fuelL100km.toStringAsFixed(1) + ' L/100км',
                    style: const TextStyle(color: Colors.pinkAccent, fontSize: 14)),
                ])),
                Container(width: 1, height: 50, color: Colors.white24),
                Expanded(child: Column(children: [
                  const Text('За поездку', style: TextStyle(color: Colors.white54, fontSize: 11)),
                  Text(_data.tripFuelL.toStringAsFixed(2) + ' L',
                    style: const TextStyle(color: Colors.orange, fontSize: 18, fontWeight: FontWeight.bold)),
                ])),
              ]),
            ]))),
          const SizedBox(height: 8),
          Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(12), child: Column(children: [
            const Text('ТОПЛИВНЫЕ КОРРЕКЦИИ', style: TextStyle(color: Colors.white70, fontSize: 12)),
            const SizedBox(height: 8),
            Row(children: [
              Expanded(child: _trim('STFT', _data.shortFuelTrim)),
              Expanded(child: _trim('LTFT', _data.longFuelTrim)),
            ]),
          ]))),
          const SizedBox(height: 8),
          Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(12), child: Row(children: [
            Expanded(child: Column(children: [
              const Text('Мощность', style: TextStyle(color: Colors.white70)),
              Text(_data.calculatedHP.toStringAsFixed(1) + ' л.с.',
                style: const TextStyle(fontSize: 22, color: Colors.yellow, fontWeight: FontWeight.bold)),
            ])),
            Expanded(child: Column(children: [
              const Text('Момент', style: TextStyle(color: Colors.white70)),
              Text(_data.calculatedTorque.toStringAsFixed(0) + ' Нм',
                style: const TextStyle(fontSize: 22, color: Colors.orange, fontWeight: FontWeight.bold)),
            ])),
            Expanded(child: Column(children: [
              const Text('VE', style: TextStyle(color: Colors.white70)),
              Text(_data.volumetricEfficiency.toStringAsFixed(0) + '%',
                style: const TextStyle(fontSize: 22, color: Colors.lightBlue, fontWeight: FontWeight.bold)),
            ])),
          ]))),
          const SizedBox(height: 8),
          Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(12), child: Column(children: [
            const Text('РЕЖИМ РАБОТЫ', style: TextStyle(color: Colors.white70, fontSize: 12)),
            Text(_data.engineMode, style: const TextStyle(fontSize: 18, color: Colors.cyan, fontWeight: FontWeight.bold)),
          ]))),
        ])));
  }

  Widget _gauge(String l, String v, Color c) {
    return Card(color: const Color(0xFF16213E),
      child: Padding(padding: const EdgeInsets.all(16), child: Column(children: [
        Text(l, style: const TextStyle(color: Colors.white70, fontSize: 14)),
        const SizedBox(height: 8),
        FittedBox(child: Text(v, style: TextStyle(color: c, fontSize: 42, fontWeight: FontWeight.bold))),
      ])));
  }
  Widget _param(String l, String v, String u, Color c) {
    return Card(color: const Color(0xFF16213E),
      child: Padding(padding: const EdgeInsets.all(6), child: Column(
        mainAxisAlignment: MainAxisAlignment.center, children: [
          Text(l, style: const TextStyle(color: Colors.white54, fontSize: 10)),
          FittedBox(child: Row(mainAxisSize: MainAxisSize.min,
            crossAxisAlignment: CrossAxisAlignment.baseline,
            textBaseline: TextBaseline.alphabetic, children: [
              Text(v, style: TextStyle(color: c, fontSize: 16, fontWeight: FontWeight.bold)),
              if (u.isNotEmpty) Text(' ' + u, style: TextStyle(color: c.withOpacity(0.7), fontSize: 10)),
            ])),
        ])));
  }
  Widget _trim(String l, double v) {
    Color c = v.abs() > 15 ? Colors.red : v.abs() > 10 ? Colors.orange : Colors.green;
    return Column(children: [
      Text(l, style: const TextStyle(color: Colors.white70)),
      Text(v.toStringAsFixed(1) + '%', style: TextStyle(color: c, fontSize: 20, fontWeight: FontWeight.bold)),
    ]);
  }
  Color _rpmColor(int r) => r > 6500 ? Colors.red : r > 5500 ? Colors.orange : Colors.green;
  Color _timingColor(double t) => t < 0 ? Colors.red : t > 40 ? Colors.orange : Colors.green;
  Color _knockColor(double k) => k > 3 ? Colors.red : k > 1 ? Colors.orange : Colors.green;
  Color _afrColor(double a) => a < 11 || a > 15 ? Colors.red : a < 12 || a > 14.5 ? Colors.orange : Colors.green;
  Color _tempColor(int t) => t > 105 ? Colors.red : t > 95 ? Colors.orange : t < 60 ? Colors.blue : Colors.green;
}
''')
print("✅ dashboard_screen.dart")

print()
print("Часть A завершена. Файлы:")
!ls lib/widgets/
!ls lib/screens/ | head -5
# @title 🎨 Ячейка 4/5: Экраны (ЧАСТЬ B) - настройки + графики + логи + DTC + все остальные
import os
os.chdir('/content/nissan_logger_pro_v5')

# ============ settings_screen.dart (ПОЛНЫЙ v4 с ИНИЦИАЛИЗАЦИЯ ЭБУ + все настройки) ============
with open('lib/screens/settings_screen.dart', 'w') as f:
    f.write('''import 'package:flutter/material.dart';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';
import 'package:permission_handler/permission_handler.dart';
import '../services/obd_service.dart';
import '../services/alert_service.dart';
import '../services/settings_service.dart';
import '../widgets/fps_indicator.dart';

class SettingsScreen extends StatefulWidget {
  final OBDService obdService;
  final AlertService alertService;
  const SettingsScreen({super.key, required this.obdService, required this.alertService});
  @override
  State<SettingsScreen> createState() => _SettingsScreenState();
}

class _SettingsScreenState extends State<SettingsScreen> {
  List<BluetoothDevice> _devices = [];
  bool _isScanning = false;
  bool _isConnecting = false;
  bool _isInitializing = false;
  String _btStatus = '...';
  int _pollingInterval = 50;
  bool _autoConnect = false;
  bool _autoLog = false;
  bool _alertsEnabled = true;
  bool _soundEnabled = true;
  bool _vibrationEnabled = true;
  double _engineDisplacement = 2.0;
  double _mafMultiplier = 0.1;
  double _speedMultiplier = 1.05;

  @override
  void initState() {
    super.initState();
    _loadSettings();
    _checkBluetooth();
  }

  void _loadSettings() {
    _pollingInterval = SettingsService.pollingInterval;
    _autoConnect = SettingsService.autoConnect;
    _autoLog = SettingsService.autoLog;
    _alertsEnabled = SettingsService.alertsEnabled;
    _soundEnabled = SettingsService.soundEnabled;
    _vibrationEnabled = SettingsService.vibrationEnabled;
    _engineDisplacement = SettingsService.engineDisplacement;
    _mafMultiplier = SettingsService.mafMultiplier;
    _speedMultiplier = SettingsService.speedMultiplier;
    widget.obdService.pollingInterval = _pollingInterval;
  }

  Future<void> _checkPermissions() async {
    await Permission.bluetoothScan.request();
    await Permission.bluetoothConnect.request();
    await Permission.location.request();
  }

  Future<void> _checkBluetooth() async {
    await _checkPermissions();
    try {
      final state = await widget.obdService.getBluetoothState();
      setState(() => _btStatus = state == BluetoothState.STATE_ON ? 'Включён' : 'Выключен');
      if (state == BluetoothState.STATE_ON) _loadDevices();
    } catch (e) {
      setState(() => _btStatus = 'Ошибка');
    }
  }

  Future<void> _enableBt() async {
    await widget.obdService.requestEnable();
    await _checkBluetooth();
  }

  Future<void> _loadDevices() async {
    setState(() => _isScanning = true);
    try {
      final devs = await widget.obdService.getBondedDevices();
      setState(() { _devices = devs; _isScanning = false; });
    } catch (e) { setState(() => _isScanning = false); }
  }

  Future<void> _connect(BluetoothDevice device) async {
    setState(() => _isConnecting = true);
    try {
      _snack('Подключение...', Colors.blue);
      final ok = await widget.obdService.connect(device.address);
      setState(() => _isConnecting = false);
      if (ok) _snack('BT подключён! Нажми ИНИЦИАЛИЗАЦИЯ', Colors.orange);
      else _snack('Не удалось', Colors.red);
    } catch (e) {
      setState(() => _isConnecting = false);
      _snack('Ошибка', Colors.red);
    }
  }

  Future<void> _initECU({bool useCache = true}) async {
    if (!widget.obdService.isConnected) {
      _snack('Сначала подключитесь', Colors.red);
      return;
    }
    setState(() => _isInitializing = true);
    _snack('Инициализация...', Colors.blue);
    try {
      final ok = await widget.obdService.initECU(useCache: useCache);
      setState(() => _isInitializing = false);
      if (ok) _snack('ЭБУ отвечает! ' + widget.obdService.activePids.length.toString() + ' PID', Colors.green);
      else _snack('ЭБУ не отвечает', Colors.red);
    } catch (e) {
      setState(() => _isInitializing = false);
      _snack('Ошибка', Colors.red);
    }
  }

  Future<void> _disconnect() async {
    await widget.obdService.disconnect();
    setState(() {});
    _snack('Отключено', Colors.orange);
  }

  Future<void> _clearCache() async {
    await SettingsService.clearPidCache();
    _snack('Кеш PID очищен', Colors.orange);
  }

  Future<void> _resetTripFuel() async {
    final ok = await showDialog<bool>(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: const Text('Сбросить счётчик топлива?'),
      content: Text('Накоплено ' + widget.obdService.tripFuelL.toStringAsFixed(2) + ' L',
        style: const TextStyle(color: Colors.white70)),
      actions: [
        TextButton(onPressed: () => Navigator.pop(c, false),
          style: TextButton.styleFrom(foregroundColor: Colors.white),
          child: const Text('Отмена')),
        TextButton(onPressed: () => Navigator.pop(c, true),
          style: TextButton.styleFrom(foregroundColor: Colors.red),
          child: const Text('СБРОС')),
      ]));
    if (ok == true) {
      await widget.obdService.resetTripFuel();
      _snack('Сброшено', Colors.green);
      setState(() {});
    }
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(
      content: Text(m, style: const TextStyle(color: Colors.white)),
      backgroundColor: c, duration: const Duration(seconds: 2)));
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Настройки'),
        backgroundColor: const Color(0xFF16213E),
        actions: [
          FpsIndicator(obdService: widget.obdService),
          IconButton(icon: const Icon(Icons.refresh), onPressed: _loadDevices),
        ]),
      body: ListView(padding: const EdgeInsets.all(12), children: [
        // BLUETOOTH
        Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(12),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            const Text('BLUETOOTH', style: TextStyle(color: Colors.white70, fontSize: 12, fontWeight: FontWeight.bold)),
            const SizedBox(height: 8),
            Row(children: [
              Icon(_btStatus == 'Включён' ? Icons.bluetooth : Icons.bluetooth_disabled,
                color: _btStatus == 'Включён' ? Colors.green : Colors.red),
              const SizedBox(width: 8),
              Text('Статус: ' + _btStatus, style: const TextStyle(fontSize: 15)),
            ]),
            if (_btStatus != 'Включён')
              ElevatedButton.icon(onPressed: _enableBt,
                icon: const Icon(Icons.bluetooth),
                label: const Text('Включить BT'),
                style: ElevatedButton.styleFrom(backgroundColor: Colors.blue, foregroundColor: Colors.white)),
          ]))),
        const SizedBox(height: 8),
        // ELM327 + ИНИЦИАЛИЗАЦИЯ
        Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(12),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            Row(children: [
              const Text('ELM327', style: TextStyle(color: Colors.white70, fontSize: 12, fontWeight: FontWeight.bold)),
              const Spacer(),
              if (widget.obdService.ecuResponds)
                Container(padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
                  decoration: BoxDecoration(color: Colors.green.withOpacity(0.3), borderRadius: BorderRadius.circular(4)),
                  child: const Text('ЭБУ ОК', style: TextStyle(color: Colors.green, fontSize: 10, fontWeight: FontWeight.bold))),
            ]),
            const SizedBox(height: 6),
            const Padding(padding: EdgeInsets.symmetric(vertical: 4),
              child: Text('ВАЖНО: заведи двигатель!',
                style: TextStyle(color: Colors.orange, fontSize: 11, fontWeight: FontWeight.bold),
                textAlign: TextAlign.center)),
            if (_isScanning) const Center(child: CircularProgressIndicator())
            else if (_devices.isEmpty) const Text('Нет устройств')
            else ..._devices.map((d) => _deviceTile(d)).toList(),
            const SizedBox(height: 8),
            if (widget.obdService.isConnected && !widget.obdService.ecuResponds)
              SizedBox(width: double.infinity, height: 55,
                child: ElevatedButton.icon(
                  onPressed: _isInitializing ? null : () => _initECU(useCache: true),
                  icon: _isInitializing
                    ? const SizedBox(width: 20, height: 20, child: CircularProgressIndicator(color: Colors.white, strokeWidth: 2))
                    : const Icon(Icons.settings_input_component),
                  label: Text(_isInitializing ? 'ИНИЦИАЛИЗАЦИЯ...' : 'ИНИЦИАЛИЗАЦИЯ ЭБУ',
                    style: const TextStyle(fontSize: 15, fontWeight: FontWeight.bold)),
                  style: ElevatedButton.styleFrom(backgroundColor: Colors.deepOrange, foregroundColor: Colors.white))),
            if (widget.obdService.ecuResponds)
              Padding(padding: const EdgeInsets.only(top: 6),
                child: Row(children: [
                  Expanded(child: Text('Протокол: ' + widget.obdService.protocolInfo +
                    '\\nPID: ' + widget.obdService.activePids.length.toString() +
                    '\\nFPS: ' + widget.obdService.pollFps.toString(),
                    style: const TextStyle(color: Colors.green, fontSize: 11))),
                  TextButton(onPressed: () => _initECU(useCache: false),
                    style: TextButton.styleFrom(foregroundColor: Colors.white),
                    child: const Text('Пересканировать', style: TextStyle(fontSize: 10))),
                ])),
            if (widget.obdService.isConnected)
              Padding(padding: const EdgeInsets.only(top: 8),
                child: SizedBox(width: double.infinity,
                  child: ElevatedButton.icon(onPressed: _disconnect,
                    icon: const Icon(Icons.bluetooth_disabled),
                    label: const Text('ОТКЛЮЧИТЬ', style: TextStyle(fontWeight: FontWeight.bold)),
                    style: ElevatedButton.styleFrom(backgroundColor: Colors.red, foregroundColor: Colors.white)))),
          ]))),
        const SizedBox(height: 8),
        // АВТОМАТИЗАЦИЯ
        Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(12),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            const Text('АВТОМАТИЗАЦИЯ', style: TextStyle(color: Colors.white70, fontSize: 12, fontWeight: FontWeight.bold)),
            SwitchListTile(contentPadding: EdgeInsets.zero, dense: true,
              title: const Text('Автоподключение при запуске'),
              value: _autoConnect,
              onChanged: (v) async {
                setState(() => _autoConnect = v);
                await SettingsService.setAutoConnect(v);
              }),
            SwitchListTile(contentPadding: EdgeInsets.zero, dense: true,
              title: const Text('Автозапуск лога при движении'),
              subtitle: const Text('RPM>1500 или скорость>5', style: TextStyle(fontSize: 11)),
              value: _autoLog,
              onChanged: (v) async {
                setState(() => _autoLog = v);
                await SettingsService.setAutoLog(v);
              }),
          ]))),
        const SizedBox(height: 8),
        // ИНТЕРВАЛ ОПРОСА
        Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(12),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            const Text('ИНТЕРВАЛ ОПРОСА', style: TextStyle(color: Colors.white70, fontSize: 12, fontWeight: FontWeight.bold)),
            Text('Пауза: ' + _pollingInterval.toString() + ' мс', style: const TextStyle(fontSize: 15)),
            Slider(value: _pollingInterval.toDouble(), min: 0, max: 500, divisions: 50,
              label: _pollingInterval.toString() + ' мс',
              onChanged: (v) async {
                setState(() {
                  _pollingInterval = v.toInt();
                  widget.obdService.pollingInterval = _pollingInterval;
                });
                await SettingsService.setPollingInterval(_pollingInterval);
              }),
          ]))),
        const SizedBox(height: 8),
        // ДВИГАТЕЛЬ + MAF
        Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(12),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            const Text('ДВИГАТЕЛЬ И РАСХОДОМЕР', style: TextStyle(color: Colors.white70, fontSize: 12, fontWeight: FontWeight.bold)),
            const SizedBox(height: 8),
            Text('Объём: ' + _engineDisplacement.toStringAsFixed(1) + ' л', style: const TextStyle(fontSize: 15)),
            Slider(value: _engineDisplacement, min: 1.0, max: 5.0, divisions: 40,
              onChanged: (v) async {
                setState(() => _engineDisplacement = v);
                await SettingsService.setEngineDisplacement(v);
              }),
            const Divider(color: Colors.white24),
            const Text('MAF МНОЖИТЕЛЬ',
              style: TextStyle(color: Colors.orange, fontSize: 12, fontWeight: FontWeight.bold)),
            Text('× ' + _mafMultiplier.toStringAsFixed(3), style: const TextStyle(fontSize: 15)),
            Slider(value: _mafMultiplier, min: 0.01, max: 5.0, divisions: 100,
              onChanged: (v) async {
                setState(() => _mafMultiplier = v);
                await SettingsService.setMafMultiplier(v);
              }),
            const Text('Настрой чтобы MAF на ХХ прогретого = 2-4 g/s',
              style: TextStyle(color: Colors.white54, fontSize: 10)),
            const SizedBox(height: 4),
            Row(children: [
              _mafPreset('0.1', 0.1), const SizedBox(width: 3),
              _mafPreset('0.5', 0.5), const SizedBox(width: 3),
              _mafPreset('1.0', 1.0), const SizedBox(width: 3),
              _mafPreset('2.0', 2.0), const SizedBox(width: 3),
              _mafPreset('5.0', 5.0),
            ]),
          ]))),
        const SizedBox(height: 8),
        // КАЛИБРОВКА СКОРОСТИ
        Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(12),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            Row(children: const [
              Icon(Icons.speed, color: Colors.blue, size: 18), SizedBox(width: 6),
              Text('КАЛИБРОВКА СКОРОСТИ', style: TextStyle(color: Colors.white70, fontSize: 12, fontWeight: FontWeight.bold)),
            ]),
            const SizedBox(height: 8),
            Text('× ' + _speedMultiplier.toStringAsFixed(3), style: const TextStyle(fontSize: 15)),
            Slider(value: _speedMultiplier, min: 0.80, max: 1.30, divisions: 50,
              onChanged: (v) async {
                setState(() => _speedMultiplier = v);
                await SettingsService.setSpeedMultiplier(v);
              }),
            const Text('Приборка обычно завышает на 3-7%',
              style: TextStyle(color: Colors.white54, fontSize: 10)),
            const SizedBox(height: 4),
            Row(children: [
              _speedPreset('1.00', 1.00), const SizedBox(width: 3),
              _speedPreset('1.03', 1.03), const SizedBox(width: 3),
              _speedPreset('1.05', 1.05), const SizedBox(width: 3),
              _speedPreset('1.07', 1.07), const SizedBox(width: 3),
              _speedPreset('1.10', 1.10),
            ]),
          ]))),
        const SizedBox(height: 8),
        // РАСХОД ТОПЛИВА
        Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(12),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            Row(children: const [
              Icon(Icons.local_gas_station, color: Colors.pink, size: 18), SizedBox(width: 6),
              Text('РАСХОД ТОПЛИВА', style: TextStyle(color: Colors.white70, fontSize: 12, fontWeight: FontWeight.bold)),
            ]),
            const SizedBox(height: 8),
            Text('Накоплено: ' + widget.obdService.tripFuelL.toStringAsFixed(3) + ' L',
              style: const TextStyle(fontSize: 14, color: Colors.pink)),
            const SizedBox(height: 8),
            SizedBox(width: double.infinity, child: ElevatedButton.icon(
              onPressed: _resetTripFuel,
              icon: const Icon(Icons.restart_alt),
              label: const Text('СБРОСИТЬ СЧЁТЧИК'),
              style: ElevatedButton.styleFrom(backgroundColor: Colors.orange, foregroundColor: Colors.white))),
          ]))),
        const SizedBox(height: 8),
        // УВЕДОМЛЕНИЯ
        Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(12),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            const Text('УВЕДОМЛЕНИЯ', style: TextStyle(color: Colors.white70, fontSize: 12, fontWeight: FontWeight.bold)),
            SwitchListTile(contentPadding: EdgeInsets.zero, dense: true,
              title: const Text('Все алерты'), value: _alertsEnabled,
              onChanged: (v) async { setState(() => _alertsEnabled = v); await SettingsService.setAlertsEnabled(v); }),
            SwitchListTile(contentPadding: EdgeInsets.zero, dense: true,
              title: const Text('Звук'), value: _soundEnabled,
              onChanged: _alertsEnabled ? (v) async {
                setState(() => _soundEnabled = v); await SettingsService.setSoundEnabled(v);
              } : null),
            SwitchListTile(contentPadding: EdgeInsets.zero, dense: true,
              title: const Text('Вибрация'), value: _vibrationEnabled,
              onChanged: _alertsEnabled ? (v) async {
                setState(() => _vibrationEnabled = v); await SettingsService.setVibrationEnabled(v);
              } : null),
          ]))),
        const SizedBox(height: 8),
        // КЕШ
        Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(12),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            const Text('КЕШ ПРИЛОЖЕНИЯ', style: TextStyle(color: Colors.white70, fontSize: 12, fontWeight: FontWeight.bold)),
            Text('ECU: ' + (SettingsService.cachedEcuId ?? 'не сохранён'), style: const TextStyle(fontSize: 12)),
            Text('PID: ' + SettingsService.cachedPidList.length.toString(), style: const TextStyle(fontSize: 12)),
            const SizedBox(height: 6),
            OutlinedButton.icon(onPressed: _clearCache,
              icon: const Icon(Icons.delete_outline),
              label: const Text('Очистить кеш PID'),
              style: OutlinedButton.styleFrom(foregroundColor: Colors.white)),
          ]))),
      ]));
  }

  Widget _mafPreset(String l, double v) {
    final active = (_mafMultiplier - v).abs() < 0.001;
    return Expanded(child: ElevatedButton(
      onPressed: () async {
        setState(() => _mafMultiplier = v);
        await SettingsService.setMafMultiplier(v);
      },
      style: ElevatedButton.styleFrom(
        backgroundColor: active ? Colors.orange : const Color(0xFF0F3460),
        foregroundColor: Colors.white,
        padding: const EdgeInsets.symmetric(vertical: 6)),
      child: Text(l, style: const TextStyle(fontSize: 12, fontWeight: FontWeight.bold))));
  }

  Widget _speedPreset(String l, double v) {
    final active = (_speedMultiplier - v).abs() < 0.001;
    return Expanded(child: ElevatedButton(
      onPressed: () async {
        setState(() => _speedMultiplier = v);
        await SettingsService.setSpeedMultiplier(v);
      },
      style: ElevatedButton.styleFrom(
        backgroundColor: active ? Colors.blue : const Color(0xFF0F3460),
        foregroundColor: Colors.white,
        padding: const EdgeInsets.symmetric(vertical: 6)),
      child: Text(l, style: const TextStyle(fontSize: 11, fontWeight: FontWeight.bold))));
  }

  Widget _deviceTile(BluetoothDevice d) {
    final isOBD = (d.name ?? '').toUpperCase().contains('OBD') ||
                  (d.name ?? '').toUpperCase().contains('ELM');
    final isConn = widget.obdService.isConnected;
    return Card(color: isOBD ? const Color(0xFF0F3460) : const Color(0xFF1A1A2E),
      child: ListTile(dense: true,
        leading: Icon(Icons.bluetooth, color: isOBD ? Colors.orange : Colors.white70),
        title: Text(d.name ?? 'Unknown', style: const TextStyle(color: Colors.white)),
        subtitle: Text(d.address, style: const TextStyle(fontSize: 10, color: Colors.white70)),
        trailing: _isConnecting
          ? const SizedBox(width: 24, height: 24, child: CircularProgressIndicator(strokeWidth: 2))
          : ElevatedButton(
              onPressed: isConn ? null : () => _connect(d),
              style: ElevatedButton.styleFrom(
                backgroundColor: const Color(0xFFE94560),
                foregroundColor: Colors.white,
                disabledBackgroundColor: Colors.grey.shade700,
                disabledForegroundColor: Colors.white70,
                padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 6)),
              child: Text(isConn ? 'OK' : 'CONNECT',
                style: const TextStyle(fontSize: 12, fontWeight: FontWeight.bold)))));
  }
}
''')
print("✅ settings_screen.dart (полный v4 + MAF/Speed + trip fuel)")

# ============ graph_screen.dart ============
with open('lib/screens/graph_screen.dart', 'w') as f:
    f.write('''import 'package:flutter/material.dart';
import 'package:fl_chart/fl_chart.dart';
import '../models/obd_data.dart';
import '../services/obd_service.dart';
import '../services/settings_service.dart';
import '../widgets/fps_indicator.dart';

class GraphConfig {
  final String title;
  final double Function(OBDData) getValue;
  final Color color;
  final double minY;
  final double maxY;
  final String unit;
  GraphConfig({required this.title, required this.getValue, required this.color,
    required this.minY, required this.maxY, required this.unit});
}

class GraphScreen extends StatefulWidget {
  final OBDService obdService;
  const GraphScreen({super.key, required this.obdService});
  @override
  State<GraphScreen> createState() => _GraphScreenState();
}

class _GraphScreenState extends State<GraphScreen> {
  final int _maxPoints = 100;
  List<OBDData> _history = [];
  String _profile = 'Настройка зажигания';
  late Map<String, List<GraphConfig>> _profiles;

  @override
  void initState() {
    super.initState();
    _profile = SettingsService.selectedGraphProfile;
    _profiles = {
      'Настройка зажигания': [
        GraphConfig(title: 'RPM', getValue: (d) => d.rpm.toDouble(), color: Colors.blue, minY: 0, maxY: 7000, unit: 'об/мин'),
        GraphConfig(title: 'Зажигание', getValue: (d) => d.actualIgnition, color: Colors.green, minY: -10, maxY: 45, unit: '°'),
        GraphConfig(title: 'Knock', getValue: (d) => d.knockRetard, color: Colors.red, minY: 0, maxY: 15, unit: '°'),
        GraphConfig(title: 'Нагрузка', getValue: (d) => d.engineLoad, color: Colors.orange, minY: 0, maxY: 100, unit: '%'),
      ],
      'Настройка топлива': [
        GraphConfig(title: 'AFR', getValue: (d) => d.afr, color: Colors.green, minY: 10, maxY: 17, unit: ''),
        GraphConfig(title: 'STFT', getValue: (d) => d.shortFuelTrim, color: Colors.orange, minY: -30, maxY: 30, unit: '%'),
        GraphConfig(title: 'LTFT', getValue: (d) => d.longFuelTrim, color: Colors.red, minY: -30, maxY: 30, unit: '%'),
        GraphConfig(title: 'O2', getValue: (d) => d.o2Voltage, color: Colors.yellow, minY: 0, maxY: 1, unit: 'V'),
      ],
      'Расход топлива': [
        GraphConfig(title: 'L/ч', getValue: (d) => d.fuelFlowLph, color: Colors.pink, minY: 0, maxY: 40, unit: 'L/ч'),
        GraphConfig(title: 'L/100км', getValue: (d) => d.fuelL100km, color: Colors.pinkAccent, minY: 0, maxY: 30, unit: ''),
        GraphConfig(title: 'Скорость', getValue: (d) => d.speed.toDouble(), color: Colors.cyan, minY: 0, maxY: 200, unit: 'км/ч'),
        GraphConfig(title: 'MAF', getValue: (d) => d.maf, color: Colors.purple, minY: 0, maxY: 100, unit: 'g/s'),
      ],
      'Общий мониторинг': [
        GraphConfig(title: 'RPM', getValue: (d) => d.rpm.toDouble(), color: Colors.blue, minY: 0, maxY: 7000, unit: 'об/мин'),
        GraphConfig(title: 'ОЖ', getValue: (d) => d.coolantTemp.toDouble(), color: Colors.red, minY: 0, maxY: 130, unit: '°C'),
        GraphConfig(title: 'Дроссель', getValue: (d) => d.throttlePos, color: Colors.green, minY: 0, maxY: 100, unit: '%'),
        GraphConfig(title: 'Скорость', getValue: (d) => d.speed.toDouble(), color: Colors.cyan, minY: 0, maxY: 200, unit: 'км/ч'),
      ],
    };
    widget.obdService.dataStream.listen(_onData);
  }

  void _onData(OBDData data) {
    if (mounted) setState(() {
      _history.add(data);
      if (_history.length > _maxPoints) _history.removeAt(0);
    });
  }

  @override
  Widget build(BuildContext context) {
    final graphs = _profiles[_profile] ?? _profiles.values.first;
    return Scaffold(
      appBar: AppBar(title: const Text('Графики'), backgroundColor: const Color(0xFF16213E),
        actions: [
          FpsIndicator(obdService: widget.obdService),
          IconButton(icon: const Icon(Icons.clear), onPressed: () => setState(() => _history.clear())),
        ]),
      body: Column(children: [
        Padding(padding: const EdgeInsets.all(8),
          child: DropdownButtonFormField<String>(value: _profile,
            dropdownColor: const Color(0xFF16213E),
            decoration: InputDecoration(labelText: 'Профиль',
              border: OutlineInputBorder(borderRadius: BorderRadius.circular(8)),
              filled: true, fillColor: const Color(0xFF16213E)),
            items: _profiles.keys.map((k) => DropdownMenuItem(value: k, child: Text(k))).toList(),
            onChanged: (v) async {
              if (v != null) {
                setState(() => _profile = v);
                await SettingsService.setSelectedGraphProfile(v);
              }
            })),
        Expanded(child: SingleChildScrollView(
          child: Column(children: graphs.map((c) => _graph(c)).toList()))),
      ]));
  }

  Widget _graph(GraphConfig cfg) {
    if (_history.isEmpty) return Card(color: const Color(0xFF16213E), margin: const EdgeInsets.all(6),
      child: Container(height: 130, alignment: Alignment.center,
        child: Text(cfg.title, style: TextStyle(color: cfg.color))));
    double curVal = cfg.getValue(_history.last);
    List<FlSpot> spots = [];
    for (int i = 0; i < _history.length; i++) {
      spots.add(FlSpot(i.toDouble(), cfg.getValue(_history[i])));
    }
    return Card(color: const Color(0xFF16213E), margin: const EdgeInsets.all(6),
      child: Padding(padding: const EdgeInsets.all(10), child: Column(
        crossAxisAlignment: CrossAxisAlignment.start, children: [
          Row(mainAxisAlignment: MainAxisAlignment.spaceBetween, children: [
            Text(cfg.title, style: TextStyle(color: cfg.color, fontSize: 14, fontWeight: FontWeight.bold)),
            Text(curVal.toStringAsFixed(2) + ' ' + cfg.unit,
              style: TextStyle(color: cfg.color, fontSize: 18, fontWeight: FontWeight.bold)),
          ]),
          const SizedBox(height: 6),
          SizedBox(height: 120, child: LineChart(LineChartData(
            gridData: const FlGridData(show: false),
            titlesData: const FlTitlesData(show: false),
            borderData: FlBorderData(show: true, border: Border.all(color: Colors.white.withOpacity(0.1))),
            minX: 0, maxX: _maxPoints.toDouble(),
            minY: cfg.minY, maxY: cfg.maxY,
            lineBarsData: [LineChartBarData(spots: spots, isCurved: true, color: cfg.color,
              barWidth: 2, dotData: const FlDotData(show: false),
              belowBarData: BarAreaData(show: true, color: cfg.color.withOpacity(0.15)))],
          ))),
        ])));
  }
}
''')
print("✅ graph_screen.dart")

# ============ logging_screen.dart ============
with open('lib/screens/logging_screen.dart', 'w') as f:
    f.write('''import 'dart:io';
import 'dart:async';
import 'package:flutter/material.dart';
import 'package:intl/intl.dart';
import 'package:share_plus/share_plus.dart';
import '../services/obd_service.dart';
import '../services/logger_service.dart';
import '../widgets/fps_indicator.dart';

class LoggingScreen extends StatefulWidget {
  final OBDService obdService;
  final LoggerService loggerService;
  const LoggingScreen({super.key, required this.obdService, required this.loggerService});
  @override
  State<LoggingScreen> createState() => _LoggingScreenState();
}

class _LoggingScreenState extends State<LoggingScreen> {
  List<FileSystemEntity> _logs = [];
  Timer? _t;

  @override
  void initState() {
    super.initState();
    _load();
    _t = Timer.periodic(const Duration(seconds: 1), (_) { if (mounted) setState(() {}); });
  }
  @override
  void dispose() { _t?.cancel(); super.dispose(); }

  Future<void> _load() async {
    final logs = await widget.loggerService.getSavedLogs();
    setState(() => _logs = logs);
  }

  Future<void> _toggle() async {
    if (widget.loggerService.isLogging) {
      await widget.loggerService.stopLogging();
      await _load();
    } else {
      if (!widget.obdService.isConnected) return;
      await widget.loggerService.startLogging();
    }
    setState(() {});
  }

  Future<void> _share(String p) async {
    try { await Share.shareXFiles([XFile(p)]); } catch (e) {}
  }
  Future<void> _delete(String p) async {
    await widget.loggerService.deleteLog(p);
    await _load();
  }
  String _size(int b) {
    if (b < 1024) return b.toString() + ' B';
    if (b < 1024 * 1024) return (b / 1024).toStringAsFixed(1) + ' KB';
    return (b / (1024 * 1024)).toStringAsFixed(1) + ' MB';
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Логирование'), backgroundColor: const Color(0xFF16213E),
        actions: [FpsIndicator(obdService: widget.obdService),
          IconButton(icon: const Icon(Icons.refresh), onPressed: _load)]),
      body: Column(children: [
        Card(color: widget.loggerService.isLogging
            ? (widget.loggerService.isAutoLogging ? Colors.blue.withOpacity(0.3) : Colors.red.withOpacity(0.3))
            : const Color(0xFF16213E),
          margin: const EdgeInsets.all(12),
          child: Padding(padding: const EdgeInsets.all(16), child: Column(children: [
            Text(widget.loggerService.isLogging
              ? (widget.loggerService.isAutoLogging ? 'АВТОЛОГ' : 'ЗАПИСЬ')
              : 'Не пишется',
              style: TextStyle(fontSize: 20, fontWeight: FontWeight.bold,
                color: widget.loggerService.isLogging
                  ? (widget.loggerService.isAutoLogging ? Colors.blue : Colors.red)
                  : Colors.white)),
            const SizedBox(height: 8),
            Text('Буфер: ' + widget.loggerService.bufferSize.toString(),
              style: const TextStyle(color: Colors.white70)),
            const SizedBox(height: 16),
            SizedBox(width: double.infinity, height: 60,
              child: ElevatedButton.icon(onPressed: _toggle,
                icon: Icon(widget.loggerService.isLogging ? Icons.stop : Icons.fiber_manual_record, size: 32),
                label: Text(widget.loggerService.isLogging ? 'STOP' : 'START',
                  style: const TextStyle(fontSize: 20, fontWeight: FontWeight.bold)),
                style: ElevatedButton.styleFrom(
                  backgroundColor: widget.loggerService.isLogging ? Colors.red : Colors.green,
                  foregroundColor: Colors.white))),
          ]))),
        Expanded(child: _logs.isEmpty
          ? const Center(child: Text('Нет логов', style: TextStyle(color: Colors.white54)))
          : ListView.builder(padding: const EdgeInsets.all(8), itemCount: _logs.length,
              itemBuilder: (c, i) {
                final f = _logs[i];
                final name = f.path.split('/').last;
                final stat = File(f.path).statSync();
                final isAuto = name.startsWith('nissan_auto_');
                return Card(color: const Color(0xFF16213E),
                  child: ListTile(
                    leading: CircleAvatar(backgroundColor: isAuto ? Colors.blue : const Color(0xFF0F3460),
                      child: Icon(isAuto ? Icons.auto_awesome : Icons.description, color: Colors.white, size: 18)),
                    title: Text(name, style: const TextStyle(fontSize: 13)),
                    subtitle: Text(DateFormat('dd.MM.yyyy HH:mm').format(stat.modified) + ' • ' + _size(stat.size),
                      style: const TextStyle(fontSize: 11)),
                    trailing: Row(mainAxisSize: MainAxisSize.min, children: [
                      IconButton(icon: const Icon(Icons.share, color: Colors.blue), onPressed: () => _share(f.path)),
                      IconButton(icon: const Icon(Icons.delete, color: Colors.red), onPressed: () => _delete(f.path)),
                    ])));
              })),
      ]));
  }
}
''')
print("✅ logging_screen.dart")

# ============ events_screen.dart ============
with open('lib/screens/events_screen.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'package:flutter/material.dart';
import 'package:intl/intl.dart';
import '../models/alert.dart';
import '../services/alert_service.dart';

class EventsScreen extends StatefulWidget {
  final AlertService alertService;
  const EventsScreen({super.key, required this.alertService});
  @override
  State<EventsScreen> createState() => _EventsScreenState();
}

class _EventsScreenState extends State<EventsScreen> {
  Timer? _timer;
  @override
  void initState() {
    super.initState();
    _timer = Timer.periodic(const Duration(seconds: 1), (_) { if (mounted) setState(() {}); });
  }
  @override
  void dispose() { _timer?.cancel(); super.dispose(); }

  Color _colorFor(AlertLevel l) {
    switch (l) {
      case AlertLevel.danger: return Colors.red;
      case AlertLevel.warning: return Colors.orange;
      case AlertLevel.info: return Colors.blue;
    }
  }
  IconData _iconFor(String? cat) {
    switch (cat) {
      case 'knock': return Icons.warning_amber;
      case 'temp': return Icons.thermostat;
      case 'afr': return Icons.local_gas_station;
      case 'fuel': return Icons.settings_ethernet;
      default: return Icons.info;
    }
  }

  void _showDetail(Alert alert) {
    final s = alert.snapshot;
    final color = _colorFor(alert.level);
    showDialog(context: context, builder: (c) => Dialog(
      backgroundColor: const Color(0xFF16213E),
      child: Container(constraints: const BoxConstraints(maxWidth: 500),
        padding: const EdgeInsets.all(16),
        child: SingleChildScrollView(child: Column(
          crossAxisAlignment: CrossAxisAlignment.start, mainAxisSize: MainAxisSize.min, children: [
            Row(children: [
              Icon(_iconFor(alert.category), color: color),
              const SizedBox(width: 8),
              Expanded(child: Text(alert.message,
                style: TextStyle(color: color, fontWeight: FontWeight.bold, fontSize: 15))),
              IconButton(icon: const Icon(Icons.close, color: Colors.white54),
                onPressed: () => Navigator.pop(c),
                padding: EdgeInsets.zero, constraints: const BoxConstraints()),
            ]),
            Text(DateFormat('dd.MM.yyyy HH:mm:ss').format(alert.timestamp),
              style: const TextStyle(color: Colors.white54, fontSize: 11)),
            const Divider(color: Colors.white24, height: 20),
            if (s != null) ...[
              const Text('СТОП-КАДР:',
                style: TextStyle(color: Colors.cyan, fontWeight: FontWeight.bold, fontSize: 12)),
              const SizedBox(height: 8),
              _row('RPM', s.rpm.toString(), 'об/мин'),
              _row('Скорость', s.speed.toString(), 'км/ч'),
              _row('Нагрузка', s.engineLoad.toStringAsFixed(1), '%'),
              _row('Дроссель', s.throttlePos.toStringAsFixed(1), '%'),
              _row('MAF', s.maf.toStringAsFixed(2), 'g/s'),
              _row('ОЖ', s.coolantTemp.toString(), '°C'),
              _row('Впуск', s.intakeTemp.toString(), '°C'),
              _row('AFR', s.afr.toStringAsFixed(2), ''),
              _row('УОЗ', s.ignitionTiming.toStringAsFixed(1), '°'),
              _row('Knock', s.knockRetard.toStringAsFixed(1), '°'),
              _row('VTC', s.vtcActualAngle.toStringAsFixed(1), '°'),
              _row('STFT', s.shortFuelTrim.toStringAsFixed(1), '%'),
              _row('LTFT', s.longFuelTrim.toStringAsFixed(1), '%'),
            ],
            if (alert.explanation != null) ...[
              const Divider(color: Colors.white24, height: 20),
              Row(children: const [
                Icon(Icons.lightbulb_outline, color: Colors.yellow, size: 16),
                SizedBox(width: 6),
                Text('ПРИЧИНА:', style: TextStyle(color: Colors.yellow, fontWeight: FontWeight.bold, fontSize: 12)),
              ]),
              const SizedBox(height: 8),
              Container(padding: const EdgeInsets.all(8),
                decoration: BoxDecoration(color: Colors.black26, borderRadius: BorderRadius.circular(4)),
                child: Text(alert.explanation!,
                  style: const TextStyle(color: Colors.white70, fontSize: 12, height: 1.4))),
            ],
          ])))));
  }

  Widget _row(String label, String value, String unit) {
    return Padding(padding: const EdgeInsets.symmetric(vertical: 2),
      child: Row(children: [
        SizedBox(width: 100, child: Text(label, style: const TextStyle(color: Colors.white54, fontSize: 12))),
        Expanded(child: Text(value + (unit.isNotEmpty ? ' ' + unit : ''),
          style: const TextStyle(color: Colors.white, fontSize: 13, fontWeight: FontWeight.bold))),
      ]));
  }

  @override
  Widget build(BuildContext context) {
    final events = widget.alertService.allAlerts.reversed.toList();
    return Scaffold(
      appBar: AppBar(title: const Text('События'), backgroundColor: const Color(0xFF16213E),
        actions: [IconButton(icon: const Icon(Icons.delete_sweep),
          onPressed: () { widget.alertService.clearAlerts(); setState(() {}); })]),
      body: events.isEmpty
        ? const Center(child: Text('Пока событий нет', style: TextStyle(color: Colors.white54)))
        : ListView.builder(padding: const EdgeInsets.all(8), itemCount: events.length,
            itemBuilder: (c, i) {
              final e = events[i];
              final color = _colorFor(e.level);
              return Card(color: const Color(0xFF16213E),
                child: ListTile(leading: Icon(_iconFor(e.category), color: color),
                  title: Text(e.message, style: TextStyle(color: color, fontWeight: FontWeight.bold, fontSize: 13)),
                  subtitle: Text(DateFormat('dd.MM HH:mm:ss').format(e.timestamp),
                    style: const TextStyle(color: Colors.white54, fontSize: 11)),
                  trailing: e.snapshot != null ? const Icon(Icons.info_outline, color: Colors.cyan, size: 18) : null,
                  dense: true,
                  onTap: e.snapshot != null ? () => _showDetail(e) : null));
            }));
  }
}
''')
print("✅ events_screen.dart")

# ============ dtc_screen.dart ============
with open('lib/screens/dtc_screen.dart', 'w') as f:
    f.write('''import 'package:flutter/material.dart';
import '../services/obd_service.dart';
import '../services/dtc_service.dart';
import '../models/dtc_code.dart';
import '../widgets/fps_indicator.dart';

class DTCScreen extends StatefulWidget {
  final OBDService obdService;
  const DTCScreen({super.key, required this.obdService});
  @override
  State<DTCScreen> createState() => _DTCScreenState();
}

class _DTCScreenState extends State<DTCScreen> {
  late DTCService _dtc;
  List<DTCCode> _stored = [];
  List<DTCCode> _pending = [];
  bool _loading = false;
  bool _scanned = false;

  @override
  void initState() { super.initState(); _dtc = DTCService(widget.obdService); }

  Future<void> _read() async {
    if (!widget.obdService.isConnected) return;
    setState(() => _loading = true);
    _stored = await _dtc.readStoredDTC();
    _pending = await _dtc.readPendingDTC();
    setState(() { _loading = false; _scanned = true; });
  }

  Future<void> _clear() async {
    final ok = await showDialog<bool>(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: const Text('Стереть все ошибки?'),
      actions: [
        TextButton(onPressed: () => Navigator.pop(c, false),
          style: TextButton.styleFrom(foregroundColor: Colors.white),
          child: const Text('Отмена')),
        TextButton(onPressed: () => Navigator.pop(c, true),
          style: TextButton.styleFrom(foregroundColor: Colors.red),
          child: const Text('СТЕРЕТЬ')),
      ]));
    if (ok == true) {
      await _dtc.clearDTC();
      setState(() { _stored = []; _pending = []; });
    }
  }

  @override
  Widget build(BuildContext context) {
    final total = _stored.length + _pending.length;
    return Scaffold(
      appBar: AppBar(title: const Text('Ошибки DTC'), backgroundColor: const Color(0xFF16213E),
        actions: [
          FpsIndicator(obdService: widget.obdService),
          IconButton(icon: const Icon(Icons.refresh), onPressed: _loading ? null : _read),
          IconButton(icon: const Icon(Icons.delete_forever, color: Colors.red),
            onPressed: _loading || total == 0 ? null : _clear),
        ]),
      body: _loading ? const Center(child: CircularProgressIndicator())
        : Column(children: [
            Card(color: total == 0 && _scanned ? Colors.green.withOpacity(0.2)
                : total > 0 ? Colors.orange.withOpacity(0.2) : const Color(0xFF16213E),
              margin: const EdgeInsets.all(12),
              child: Padding(padding: const EdgeInsets.all(16), child: Row(children: [
                Icon(total == 0 && _scanned ? Icons.check_circle : total > 0 ? Icons.warning : Icons.info,
                  color: total == 0 && _scanned ? Colors.green : total > 0 ? Colors.orange : Colors.blue, size: 40),
                const SizedBox(width: 16),
                Expanded(child: Text(total == 0 && _scanned ? 'Ошибок нет!'
                    : total > 0 ? 'Ошибок: ' + total.toString() : 'Нажмите СКАНИРОВАТЬ',
                  style: const TextStyle(fontSize: 18, fontWeight: FontWeight.bold))),
              ]))),
            if (!_scanned) Padding(padding: const EdgeInsets.symmetric(horizontal: 16),
              child: SizedBox(width: double.infinity, height: 50,
                child: ElevatedButton.icon(onPressed: _read,
                  icon: const Icon(Icons.search),
                  label: const Text('СКАНИРОВАТЬ', style: TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
                  style: ElevatedButton.styleFrom(
                    backgroundColor: const Color(0xFFE94560), foregroundColor: Colors.white)))),
            Expanded(child: ListView(padding: const EdgeInsets.all(8), children: [
              ..._stored.map((d) => Card(color: const Color(0xFF16213E),
                child: ListTile(
                  leading: CircleAvatar(backgroundColor: Colors.orange,
                    child: Text(d.code[0], style: const TextStyle(color: Colors.black, fontWeight: FontWeight.bold))),
                  title: Text(d.code, style: const TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
                  subtitle: Text(d.description, style: const TextStyle(color: Colors.white70, fontSize: 12))))),
              ..._pending.map((d) => Card(color: const Color(0xFF16213E),
                child: ListTile(
                  leading: CircleAvatar(backgroundColor: Colors.yellow,
                    child: Text(d.code[0], style: const TextStyle(color: Colors.black, fontWeight: FontWeight.bold))),
                  title: Text(d.code + ' (pending)', style: const TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
                  subtitle: Text(d.description, style: const TextStyle(color: Colors.white70, fontSize: 12))))),
            ])),
          ]));
  }
}
''')
print("✅ dtc_screen.dart")

# ============ Остальные экраны (в упрощённой но рабочей форме) ============

with open('lib/screens/log_graph_screen.dart', 'w') as f:
    f.write('''import 'dart:math';
import 'package:flutter/material.dart';
import 'package:fl_chart/fl_chart.dart';
import 'package:file_picker/file_picker.dart';
import '../models/obd_data.dart';
import '../services/analyzer_service.dart';

class LogGraphScreen extends StatefulWidget {
  const LogGraphScreen({super.key});
  @override
  State<LogGraphScreen> createState() => _LogGraphScreenState();
}

class _LogGraphScreenState extends State<LogGraphScreen> {
  final _a = AnalyzerService();
  List<OBDData>? _log;
  String? _name;
  bool _loading = false;

  Future<void> _load() async {
    try {
      final r = await FilePicker.platform.pickFiles(type: FileType.custom, allowedExtensions: ['csv']);
      if (r == null) return;
      setState(() => _loading = true);
      _log = await _a.loadLogFromCSV(r.files.single.path!);
      _name = r.files.single.name;
      setState(() => _loading = false);
    } catch (e) { setState(() => _loading = false); }
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Графики лога'), backgroundColor: const Color(0xFF16213E)),
      body: Column(children: [
        Padding(padding: const EdgeInsets.all(12),
          child: ElevatedButton.icon(onPressed: _loading ? null : _load,
            icon: const Icon(Icons.folder_open),
            label: const Text('Загрузить CSV'),
            style: ElevatedButton.styleFrom(
              minimumSize: const Size.fromHeight(50), foregroundColor: Colors.white))),
        if (_name != null) Text(_name! + ' — ' + (_log?.length ?? 0).toString() + ' записей',
          style: const TextStyle(color: Colors.green)),
        Expanded(child: _log == null || _log!.isEmpty
          ? const Center(child: Text('Загрузите CSV', style: TextStyle(color: Colors.white54)))
          : _buildGraph()),
      ]));
  }

  Widget _buildGraph() {
    final rpm = _log!.map((d) => d.rpm.toDouble()).toList();
    List<FlSpot> spots = [];
    for (int i = 0; i < rpm.length; i++) { spots.add(FlSpot(i.toDouble(), rpm[i])); }
    return Padding(padding: const EdgeInsets.all(8), child: LineChart(LineChartData(
      minX: 0, maxX: rpm.length.toDouble(), minY: 0, maxY: 7000,
      lineBarsData: [LineChartBarData(spots: spots, color: Colors.blue, barWidth: 1.5,
        dotData: const FlDotData(show: false))],
    )));
  }
}
''')
print("✅ log_graph_screen.dart")

with open('lib/screens/analyzer_screen.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'package:flutter/material.dart';
import 'package:file_picker/file_picker.dart';
import '../models/obd_data.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';
import '../services/analyzer_service.dart';
import '../services/tuning_service.dart';
import '../services/obd_service.dart';
import '../widgets/fps_indicator.dart';
import '../widgets/map_table_view.dart';

class AnalyzerScreen extends StatefulWidget {
  final OBDService? obdService;
  const AnalyzerScreen({super.key, this.obdService});
  @override
  State<AnalyzerScreen> createState() => _AnalyzerScreenState();
}

class _AnalyzerScreenState extends State<AnalyzerScreen> {
  final _a = AnalyzerService();
  final _t = TuningService();
  List<OBDData>? _log;
  AnalysisResult? _result;
  TuningMap? _orig;
  TuningMap? _upd;
  bool _busy = false;
  String _map = 'Spark';

  Future<void> _load() async {
    try {
      final r = await FilePicker.platform.pickFiles(type: FileType.custom, allowedExtensions: ['csv']);
      if (r != null) {
        setState(() => _busy = true);
        _log = await _a.loadLogFromCSV(r.files.single.path!);
        setState(() => _busy = false);
      }
    } catch (e) { setState(() => _busy = false); }
  }

  Future<void> _analyze() async {
    if (_log == null || _log!.isEmpty) return;
    setState(() => _busy = true);
    TuningMap m;
    AnalysisResult r;
    if (_map == 'Spark') { m = await _t.getSparkAdvanceMap(); r = await _a.analyzeSparkMap(_log!, m); }
    else if (_map == 'Fuel') { m = await _t.getFuelMap(); r = await _a.analyzeFuelMap(_log!, m); }
    else if (_map == 'VTC') { m = await _t.getVTCMap(); r = await _a.analyzeVTCMap(_log!, m); }
    else { m = await _t.getEngineTorqueMap(); r = await _a.analyzeTorqueMap(_log!, m); }
    TuningMap u = m.copy();
    for (var c in r.changes) { u.data[c.rpmIndex][c.loadIndex] = c.suggestedValue; }
    setState(() { _result = r; _orig = m; _upd = u; _busy = false; });
  }

  void _openMap() {
    if (_orig == null) return;
    Navigator.push(context, MaterialPageRoute(builder: (c) => MapFullscreenView(
      originalMap: _orig!, updatedMap: _upd, changes: _result?.changes ?? [])));
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Анализатор'), backgroundColor: const Color(0xFF16213E),
        actions: [if (widget.obdService != null) FpsIndicator(obdService: widget.obdService!)]),
      body: Padding(padding: const EdgeInsets.all(12), child: Column(children: [
        ElevatedButton.icon(onPressed: _busy ? null : _load,
          icon: const Icon(Icons.folder_open), label: const Text('Загрузить CSV'),
          style: ElevatedButton.styleFrom(minimumSize: const Size.fromHeight(45), foregroundColor: Colors.white)),
        const SizedBox(height: 8),
        if (_log != null) Text('Записей: ' + _log!.length.toString(),
          style: const TextStyle(color: Colors.green)),
        const SizedBox(height: 8),
        DropdownButtonFormField<String>(value: _map,
          dropdownColor: const Color(0xFF16213E),
          items: const [
            DropdownMenuItem(value: 'Spark', child: Text('Зажигание')),
            DropdownMenuItem(value: 'Fuel', child: Text('Топливо/VE')),
            DropdownMenuItem(value: 'VTC', child: Text('VTC')),
            DropdownMenuItem(value: 'Torque', child: Text('Момент')),
          ], onChanged: (v) => setState(() => _map = v!)),
        const SizedBox(height: 8),
        ElevatedButton.icon(onPressed: _busy || _log == null ? null : _analyze,
          icon: const Icon(Icons.analytics), label: const Text('АНАЛИЗ'),
          style: ElevatedButton.styleFrom(
            backgroundColor: Colors.green, foregroundColor: Colors.white,
            minimumSize: const Size.fromHeight(45))),
        const SizedBox(height: 8),
        if (_result != null) ...[
          Text('Правок: ' + _result!.changes.length.toString(),
            style: const TextStyle(color: Colors.orange, fontSize: 16)),
          const SizedBox(height: 8),
          ElevatedButton.icon(onPressed: _openMap,
            icon: const Icon(Icons.grid_on), label: const Text('ОТКРЫТЬ КАРТУ'),
            style: ElevatedButton.styleFrom(
              backgroundColor: Colors.cyan, foregroundColor: Colors.white,
              minimumSize: const Size.fromHeight(50))),
          const SizedBox(height: 8),
          Expanded(child: ListView.builder(itemCount: _result!.changes.length,
            itemBuilder: (c, i) {
              final ch = _result!.changes[i];
              return Card(color: const Color(0xFF0F3460),
                child: ListTile(dense: true,
                  title: Text('RPM ' + ch.rpm.toInt().toString() + ' | ' + ch.load.toInt().toString() + '%',
                    style: const TextStyle(fontSize: 11, fontWeight: FontWeight.bold)),
                  subtitle: Text(ch.currentValue.toStringAsFixed(1) + ' → ' + ch.suggestedValue.toStringAsFixed(1),
                    style: const TextStyle(fontSize: 11))));
            })),
        ],
      ])));
  }
}
''')
print("✅ analyzer_screen.dart")

print()
print("Часть B (первые 8 экранов) готова. Осталось часть C.")
# @title 🎨 Ячейка 4/5: Экраны (ЧАСТЬ C - финал)
import os
os.chdir('/content/nissan_logger_pro_v5')

# ============ ecu_read_screen.dart (с АВТО + UNLOCK + терминал) ============
with open('lib/screens/ecu_read_screen.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'package:flutter/material.dart';
import 'package:intl/intl.dart';
import '../models/custom_map_def.dart';
import '../models/tuning_map.dart';
import '../services/obd_service.dart';
import '../services/ecu_map_reader.dart';
import '../services/map_storage_service.dart';
import '../widgets/fps_indicator.dart';
import '../widgets/map_table_view.dart';

class EcuReadScreen extends StatefulWidget {
  final OBDService obdService;
  const EcuReadScreen({super.key, required this.obdService});
  @override
  State<EcuReadScreen> createState() => _EcuReadScreenState();
}

class _EcuReadScreenState extends State<EcuReadScreen>
    with SingleTickerProviderStateMixin {
  late TabController _tab;
  late EcuMapReaderService _reader;
  bool _isDetecting = false;
  MemReadCommand? _detectedCommand;
  final List<String> _detectLog = [];
  Map<String, double> _readProgress = {};
  Map<String, EcuMapReadResult> _readResults = {};
  final TextEditingController _termCmd = TextEditingController();
  final ScrollController _termScroll = ScrollController();
  final List<String> _termLog = [];
  bool _isUnlocking = false;
  bool _isUnlocked = false;

  static final List<CustomMapDef> _standardMaps = [
    CustomMapDef(id: 'std_torque', name: 'Engine Torque',
      address: 0x7C3C, rows: 16, cols: 16, isUInt16: true,
      formula: '(X-32768)/10.24', units: 'Nm', createdAt: DateTime(2025)),
    CustomMapDef(id: 'std_force', name: 'Powertrain Force',
      address: 0xA2BC, rows: 16, cols: 16, isUInt16: true,
      formula: 'X-32768', units: 'N', createdAt: DateTime(2025)),
    CustomMapDef(id: 'std_vtc', name: 'VTC Intake Cam',
      address: 0x6BF1, rows: 8, cols: 8, isUInt16: false,
      formula: '(X-128)/2', units: 'deg', createdAt: DateTime(2025)),
    CustomMapDef(id: 'std_spark', name: 'Spark Advance WOT',
      address: 0x6EBC, rows: 16, cols: 16, isUInt16: true,
      formula: '(X-16384)/128', units: 'deg', createdAt: DateTime(2025)),
    CustomMapDef(id: 'std_ve', name: 'Fresh Air Rate / VE',
      address: 0xA754, rows: 16, cols: 15, isUInt16: true,
      formula: 'X/256', units: '%', createdAt: DateTime(2025)),
    CustomMapDef(id: 'std_lambda', name: 'Enrichment Lambda',
      address: 0x6521, rows: 8, cols: 8, isUInt16: false,
      formula: 'X/128', units: 'Lambda', createdAt: DateTime(2025)),
    CustomMapDef(id: 'std_idle', name: 'Desired Idle Speed',
      address: 0x8A7B, rows: 1, cols: 16, isUInt16: false,
      formula: 'X*10', units: 'RPM', createdAt: DateTime(2025)),
  ];

  @override
  void initState() {
    super.initState();
    _tab = TabController(length: 3, vsync: this);
    _reader = EcuMapReaderService(widget.obdService);
    _detectedCommand = _reader.getSavedCommand();
  }
  @override
  void dispose() {
    _tab.dispose(); _termCmd.dispose(); _termScroll.dispose(); super.dispose();
  }

  Future<void> _autoDetect() async {
    if (!widget.obdService.isConnected || !widget.obdService.ecuResponds) {
      _snack('Подключитесь к ЭБУ!', Colors.red); return;
    }
    setState(() { _isDetecting = true; _detectLog.clear(); });
    final cmd = await _reader.autoDetectCommand(
      onProgress: (msg) => setState(() => _detectLog.add(msg)));
    setState(() { _isDetecting = false; _detectedCommand = cmd; });
    if (cmd != null) _snack('✅ ' + cmd.description, Colors.green);
    else _snack('❌ Не найдено', Colors.red);
  }

  Future<void> _tryUnlock() async {
    if (!widget.obdService.isConnected || !widget.obdService.ecuResponds) {
      _snack('Подключитесь!', Colors.red); return;
    }
    setState(() {
      _isUnlocking = true;
      _detectLog.clear();
      _detectLog.add('=== РАЗБЛОКИРОВКА ===');
    });

    final seedResp = await widget.obdService.sendCommand('2701', timeout: 3000);
    final seedClean = seedResp.replaceAll(' ', '').toUpperCase();
    if (!seedClean.startsWith('6701') || seedClean.length < 12) {
      _detectLog.add('❌ Seed не получен: ' + seedResp);
      setState(() { _isUnlocking = false; });
      _snack('Seed не получен', Colors.red); return;
    }
    final seedHex = seedClean.substring(4, 12);
    _detectLog.add('Seed: ' + seedHex);
    setState(() {});

    final seed = <int>[];
    for (int i = 0; i < 8; i += 2) {
      seed.add(int.parse(seedHex.substring(i, i + 2), radix: 16));
    }
    final seedInt = (seed[0] << 24) | (seed[1] << 16) | (seed[2] << 8) | seed[3];

    final algs = <MapEntry<String, int>>[
      MapEntry('NOT seed', ~seedInt & 0xFFFFFFFF),
      MapEntry('XOR FFFFFFFF', seedInt ^ 0xFFFFFFFF),
      MapEntry('seed + 12345678', (seedInt + 0x12345678) & 0xFFFFFFFF),
      MapEntry('seed - 12345678', (seedInt - 0x12345678) & 0xFFFFFFFF),
      MapEntry('rotate3 XOR', (((seedInt << 3) | (seedInt >> 29)) ^ seedInt) & 0xFFFFFFFF),
      MapEntry('rotate7 XOR', (((seedInt << 7) | (seedInt >> 25)) ^ seedInt) & 0xFFFFFFFF),
      MapEntry('seed + 1971', (seedInt + 0x1971) & 0xFFFFFFFF),
      MapEntry('XOR 5A5A5A5A', seedInt ^ 0x5A5A5A5A),
      MapEntry('XOR A5A5A5A5', seedInt ^ 0xA5A5A5A5),
      MapEntry('reversed bytes',
        ((seed[3] << 24) | (seed[2] << 16) | (seed[1] << 8) | seed[0])),
      MapEntry('Nissan classic',
        ((seedInt * 0x91) + 0x1971) & 0xFFFFFFFF),
    ];

    for (final algo in algs) {
      final keyHex = algo.value.toRadixString(16).padLeft(8, '0').toUpperCase();
      _detectLog.add('→ ' + algo.key + ': ' + keyHex);
      setState(() {});
      final resp = await widget.obdService.sendCommand('2702' + keyHex, timeout: 3000);
      final respClean = resp.replaceAll(' ', '').toUpperCase();
      if (respClean.startsWith('6702')) {
        _detectLog.add('✅✅✅ УСПЕХ!');
        setState(() { _isUnlocking = false; _isUnlocked = true; });
        _snack('РАЗБЛОКИРОВАНО!', Colors.green); return;
      } else if (respClean.startsWith('7F2736')) {
        _detectLog.add('⏸ Много попыток');
        setState(() { _isUnlocking = false; });
        _snack('Подожди 10 сек', Colors.orange); return;
      }
      setState(() {});
      await Future.delayed(const Duration(milliseconds: 300));
      await widget.obdService.sendCommand('2701', timeout: 2000);
    }
    _detectLog.add('❌ Не подошёл');
    setState(() { _isUnlocking = false; });
    _snack('Не подошёл', Colors.red);
  }

  Future<void> _readMap(CustomMapDef def) async {
    if (_detectedCommand == null) { _snack('Сначала АВТО', Colors.orange); return; }
    setState(() => _readProgress[def.addressHex] = 0.0);
    try {
      final result = await _reader.readMap(def: def, command: _detectedCommand,
        onProgress: (done, total) => setState(() => _readProgress[def.addressHex] = done / total * 100));
      if (result != null) {
        setState(() {
          _readResults[def.addressHex] = result;
          _readProgress.remove(def.addressHex);
        });
      } else {
        setState(() => _readProgress.remove(def.addressHex));
      }
    } catch (e) {
      setState(() => _readProgress.remove(def.addressHex));
    }
  }

  Future<void> _saveAsDefault(EcuMapReadResult r) async {
    final m = TuningMap(name: r.def.name, address: r.def.addressHex,
      rows: r.def.rows, cols: r.def.cols,
      rpmAxis: r.def.yAxis ?? List.generate(r.def.rows, (i) => i.toDouble()),
      loadAxis: r.def.xAxis ?? List.generate(r.def.cols, (i) => i.toDouble()),
      data: r.data, units: r.def.units);
    await MapStorageService.saveMap(m);
    _snack('Сохранено', Colors.green);
  }

  void _viewReadMap(EcuMapReadResult r) {
    final m = TuningMap(name: r.def.name + ' (из ЭБУ)', address: r.def.addressHex,
      rows: r.def.rows, cols: r.def.cols,
      rpmAxis: r.def.yAxis ?? List.generate(r.def.rows, (i) => i * 400.0),
      loadAxis: r.def.xAxis ?? List.generate(r.def.cols, (i) => i * 10.0),
      data: r.data, units: r.def.units);
    Navigator.push(context, MaterialPageRoute(
      builder: (c) => MapFullscreenView(originalMap: m, updatedMap: null, changes: const [])));
  }

  Future<void> _sendTermCmd() async {
    final cmd = _termCmd.text.trim().toUpperCase();
    if (cmd.isEmpty) return;
    if (!widget.obdService.isConnected) { _snack('Нет подключения', Colors.red); return; }
    setState(() => _termLog.add('>>> ' + cmd));
    final r = await _reader.testRawCommand(cmd);
    setState(() => _termLog.add('<<< ' + r));
    _termCmd.clear();
  }

  void _quick(String c) { _termCmd.text = c; _sendTermCmd(); }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(
      content: Text(m, style: const TextStyle(color: Colors.white)), backgroundColor: c));
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('ЭБУ Карты'), backgroundColor: const Color(0xFF16213E),
        actions: [FpsIndicator(obdService: widget.obdService)],
        bottom: TabBar(controller: _tab, tabs: const [
          Tab(icon: Icon(Icons.list), text: 'Карты'),
          Tab(icon: Icon(Icons.add_box), text: 'Свои'),
          Tab(icon: Icon(Icons.terminal), text: 'Терминал'),
        ])),
      body: Column(children: [
        _statusPanel(),
        Expanded(child: TabBarView(controller: _tab, children: [
          _mapsTab(),
          const Center(child: Text('Кастомные карты (в разработке)', style: TextStyle(color: Colors.white54))),
          _termTab(),
        ])),
      ]));
  }

  Widget _statusPanel() {
    final conn = widget.obdService.ecuResponds;
    return Container(padding: const EdgeInsets.all(8), color: const Color(0xFF16213E),
      child: Column(children: [
        Row(children: [
          Icon(conn ? Icons.check_circle : Icons.error,
            color: conn ? Colors.green : Colors.red, size: 16),
          const SizedBox(width: 4),
          Expanded(child: Text(conn ? 'ЭБУ подключен' : 'ЭБУ не отвечает',
            style: TextStyle(color: conn ? Colors.green : Colors.red, fontSize: 12))),
          if (_detectedCommand != null) Container(
            padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
            decoration: BoxDecoration(color: Colors.green.withOpacity(0.3),
              borderRadius: BorderRadius.circular(4)),
            child: Text(_detectedCommand!.id, style: const TextStyle(
              color: Colors.green, fontSize: 9, fontWeight: FontWeight.bold))),
          if (_isUnlocked) Container(margin: const EdgeInsets.only(left: 4),
            padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
            decoration: BoxDecoration(color: Colors.green, borderRadius: BorderRadius.circular(4)),
            child: const Text('UNLOCKED', style: TextStyle(
              color: Colors.white, fontSize: 9, fontWeight: FontWeight.bold))),
        ]),
        const SizedBox(height: 6),
        Row(children: [
          Expanded(child: ElevatedButton.icon(
            onPressed: (_isDetecting || _isUnlocking || !conn) ? null : _autoDetect,
            icon: _isDetecting ? const SizedBox(width: 12, height: 12,
              child: CircularProgressIndicator(strokeWidth: 2, color: Colors.white))
              : const Icon(Icons.search, size: 14),
            label: const Text('АВТО', style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(
              backgroundColor: Colors.cyan, foregroundColor: Colors.white,
              padding: const EdgeInsets.symmetric(vertical: 6)))),
          const SizedBox(width: 4),
          Expanded(child: ElevatedButton.icon(
            onPressed: (_isDetecting || _isUnlocking || !conn) ? null : _tryUnlock,
            icon: _isUnlocking ? const SizedBox(width: 12, height: 12,
              child: CircularProgressIndicator(strokeWidth: 2, color: Colors.white))
              : Icon(_isUnlocked ? Icons.lock_open : Icons.lock, size: 14),
            label: Text(_isUnlocked ? 'UNLOCKED' : 'UNLOCK',
              style: const TextStyle(fontSize: 10, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(
              backgroundColor: _isUnlocked ? Colors.green : Colors.red,
              foregroundColor: Colors.white,
              padding: const EdgeInsets.symmetric(vertical: 6)))),
        ]),
        if (_detectLog.isNotEmpty) Container(
          margin: const EdgeInsets.only(top: 4),
          padding: const EdgeInsets.all(4),
          constraints: const BoxConstraints(maxHeight: 100),
          decoration: BoxDecoration(color: Colors.black, borderRadius: BorderRadius.circular(4)),
          child: SingleChildScrollView(reverse: true,
            child: Text(_detectLog.join('\\n'),
              style: const TextStyle(fontFamily: 'monospace', fontSize: 9, color: Colors.cyan)))),
      ]));
  }

  Widget _mapsTab() {
    return ListView.builder(padding: const EdgeInsets.all(8), itemCount: _standardMaps.length,
      itemBuilder: (c, i) => _mapCard(_standardMaps[i]));
  }

  Widget _mapCard(CustomMapDef def) {
    final progress = _readProgress[def.addressHex];
    final result = _readResults[def.addressHex];
    return Card(color: const Color(0xFF16213E), margin: const EdgeInsets.symmetric(vertical: 3),
      child: Padding(padding: const EdgeInsets.all(10), child: Column(
        crossAxisAlignment: CrossAxisAlignment.start, children: [
          Row(children: [
            Expanded(child: Text(def.name,
              style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 13))),
            Container(padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
              decoration: BoxDecoration(color: const Color(0xFF0F3460),
                borderRadius: BorderRadius.circular(4)),
              child: Text(def.addressHex, style: const TextStyle(
                fontFamily: 'monospace', fontSize: 10, color: Colors.cyan))),
          ]),
          Text('${def.rows}x${def.cols} ' + (def.isUInt16 ? 'UInt16' : 'UInt8'),
            style: const TextStyle(color: Colors.white70, fontSize: 10)),
          const SizedBox(height: 6),
          if (progress != null) Column(children: [
            LinearProgressIndicator(value: progress / 100, color: Colors.cyan),
            Text(progress.toInt().toString() + '%', style: const TextStyle(color: Colors.cyan, fontSize: 11)),
          ])
          else if (result != null) Column(children: [
            Text('Прочитано: ' + DateFormat('HH:mm:ss').format(result.readAt),
              style: const TextStyle(color: Colors.green, fontSize: 11)),
            const SizedBox(height: 6),
            Row(children: [
              Expanded(child: ElevatedButton.icon(onPressed: () => _viewReadMap(result),
                icon: const Icon(Icons.visibility, size: 14), label: const Text('Просмотр'),
                style: ElevatedButton.styleFrom(
                  backgroundColor: Colors.blue, foregroundColor: Colors.white))),
              const SizedBox(width: 4),
              Expanded(child: ElevatedButton.icon(onPressed: () => _saveAsDefault(result),
                icon: const Icon(Icons.save, size: 14), label: const Text('В дефолт'),
                style: ElevatedButton.styleFrom(
                  backgroundColor: Colors.orange, foregroundColor: Colors.white))),
            ]),
          ])
          else SizedBox(width: double.infinity, child: ElevatedButton.icon(
            onPressed: _detectedCommand == null ? null : () => _readMap(def),
            icon: const Icon(Icons.download, size: 16),
            label: Text(_detectedCommand == null ? 'Сначала АВТО' : 'ЧИТАТЬ',
              style: const TextStyle(fontSize: 12, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(backgroundColor: Colors.cyan, foregroundColor: Colors.white))),
        ])));
  }

  Widget _termTab() {
    return Column(children: [
      Flexible(flex: 3, child: Container(color: const Color(0xFF16213E),
        child: SingleChildScrollView(padding: const EdgeInsets.all(6), child: Column(children: [
          const Text('Extended Session:', style: TextStyle(color: Colors.white70, fontSize: 10)),
          Wrap(spacing: 3, runSpacing: 3, children: [
            _qBtn('1081', 'Default'), _qBtn('1085', 'ECU Prog'),
            _qBtn('1090', 'Ext90'), _qBtn('1092', 'Ext92'),
          ]),
          const SizedBox(height: 6),
          const Text('Security:', style: TextStyle(color: Colors.white70, fontSize: 10)),
          Wrap(spacing: 3, runSpacing: 3, children: [
            _qBtn('2701', 'Seed L1'), _qBtn('2703', 'Seed L3'),
          ]),
          const SizedBox(height: 6),
          const Text('Чтение памяти:', style: TextStyle(color: Colors.white70, fontSize: 10)),
          Wrap(spacing: 3, runSpacing: 3, children: [
            _qBtn('237C3C04', 'KWP'), _qBtn('217C3C', 'Consult'),
            _qBtn('D207C3C04', 'D2'),
          ]),
          const SizedBox(height: 6),
          const Text('Полезные:', style: TextStyle(color: Colors.white70, fontSize: 10)),
          Wrap(spacing: 3, runSpacing: 3, children: [
            _qBtn('1A81', 'ECU ID'), _qBtn('1180', 'Reset'),
            _qBtn('3E00', 'TestPres'),
          ]),
        ])))),
      Flexible(flex: 4, child: Container(color: Colors.black, padding: const EdgeInsets.all(8),
        child: SingleChildScrollView(controller: _termScroll,
          child: SelectableText(_termLog.join('\\n'),
            style: const TextStyle(fontFamily: 'monospace', fontSize: 11, color: Colors.green))))),
      Container(color: const Color(0xFF16213E), padding: const EdgeInsets.all(6),
        child: Row(children: [
          Expanded(child: TextField(controller: _termCmd,
            style: const TextStyle(fontFamily: 'monospace'),
            decoration: const InputDecoration(hintText: 'Команда...',
              border: OutlineInputBorder(),
              contentPadding: EdgeInsets.symmetric(horizontal: 8, vertical: 6), isDense: true),
            textCapitalization: TextCapitalization.characters, onSubmitted: (_) => _sendTermCmd())),
          IconButton(icon: const Icon(Icons.send, color: Colors.green), onPressed: _sendTermCmd),
          IconButton(icon: const Icon(Icons.clear_all, color: Colors.white54),
            onPressed: () => setState(() => _termLog.clear())),
        ])),
    ]);
  }

  Widget _qBtn(String cmd, String label) {
    return ElevatedButton(onPressed: () => _quick(cmd),
      style: ElevatedButton.styleFrom(
        backgroundColor: const Color(0xFF0F3460), foregroundColor: Colors.white,
        padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 3)),
      child: Column(mainAxisSize: MainAxisSize.min, children: [
        Text(cmd, style: const TextStyle(fontFamily: 'monospace', fontSize: 10)),
        Text(label, style: const TextStyle(fontSize: 8, color: Colors.white70)),
      ]));
  }
}
''')
print("✅ ecu_read_screen.dart (с АВТО + UNLOCK + терминал)")

# ============ service_screen.dart (тесты + repeater + агрессивный STOP + обучения) ============
with open('lib/screens/service_screen.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'package:flutter/material.dart';
import '../services/obd_service.dart';
import '../widgets/fps_indicator.dart';

class ActiveTestDef {
  final String name;
  final String cmdBase;
  final String unit;
  final double min;
  final double max;
  final double step;
  final bool isBool;
  final String category;
  final String Function(double) formatValue;
  final int Function(double) toRawByte;
  bool isSupported;
  bool isActive;
  ActiveTestDef({
    required this.name, required this.cmdBase, required this.unit,
    required this.min, required this.max, required this.step, this.isBool = false,
    required this.category, required this.formatValue, required this.toRawByte,
    this.isSupported = true, this.isActive = false,
  });
  String startCmd(double value) {
    final raw = toRawByte(value);
    return cmdBase + raw.toRadixString(16).padLeft(2, '0').toUpperCase() + '00';
  }
  String get stopCmd => cmdBase + '0000';
}

class LearningDef {
  final String name;
  final String cmd;
  final String description;
  final bool isDangerous;
  LearningDef({required this.name, required this.cmd, required this.description, this.isDangerous = false});
}

class ServiceScreen extends StatefulWidget {
  final OBDService obdService;
  const ServiceScreen({super.key, required this.obdService});
  @override
  State<ServiceScreen> createState() => _ServiceScreenState();
}

class _ServiceScreenState extends State<ServiceScreen>
    with SingleTickerProviderStateMixin {
  late TabController _tab;
  String _cat = 'all';
  bool _busy = false;
  bool _scanning = false;
  String _result = '';
  int _supported = 0;
  bool _scanned = false;
  final Map<String, Timer> _activeTimers = {};

  final List<ActiveTestDef> _tests = [
    ActiveTestDef(name: 'Симуляция темп. ОЖ', cmdBase: '3001', unit: '°C',
      min: 0, max: 125, step: 1, category: 'temp',
      formatValue: (v) => v.toStringAsFixed(0) + '°C',
      toRawByte: (v) => (v + 50).toInt().clamp(0, 255)),
    ActiveTestDef(name: 'Симуляция темп. топлива', cmdBase: '300A', unit: '°C',
      min: -50, max: 110, step: 1, category: 'temp',
      formatValue: (v) => v.toStringAsFixed(0) + '°C',
      toRawByte: (v) => (v + 50).toInt().clamp(0, 255)),
    ActiveTestDef(name: 'Коррекция впрыска', cmdBase: '3002', unit: '%',
      min: 75, max: 125, step: 1, category: 'fuel',
      formatValue: (v) => v.toStringAsFixed(0) + '%',
      toRawByte: (v) => v.toInt().clamp(0, 255)),
    ActiveTestDef(name: 'Угол зажигания', cmdBase: '3003', unit: '°',
      min: -10, max: 0, step: 1, category: 'ignition',
      formatValue: (v) => v.toStringAsFixed(0) + '°',
      toRawByte: (v) => v.toInt().clamp(-128, 127) & 0xFF),
    ActiveTestDef(name: 'Клапан ХХ %', cmdBase: '3005', unit: '%',
      min: 0, max: 127, step: 1, category: 'idle',
      formatValue: (v) => v.toStringAsFixed(0) + '%',
      toRawByte: (v) => (v + 50).toInt().clamp(0, 255)),
    ActiveTestDef(name: 'Клапан ХХ шаг', cmdBase: '3006', unit: 'шаг',
      min: 0, max: 120, step: 0.5, category: 'idle',
      formatValue: (v) => v.toStringAsFixed(1),
      toRawByte: (v) => (v * 2).toInt().clamp(0, 255)),
    ActiveTestDef(name: 'Продувка EVAP', cmdBase: '3009', unit: '%',
      min: 0, max: 100, step: 0.5, category: 'evap',
      formatValue: (v) => v.toStringAsFixed(1) + '%',
      toRawByte: (v) => (v * 2).toInt().clamp(0, 255)),
    ActiveTestDef(name: 'Клапан EGR', cmdBase: '300B', unit: 'шаг',
      min: 0, max: 100, step: 0.5, category: 'egr',
      formatValue: (v) => v.toStringAsFixed(1),
      toRawByte: (v) => (v * 2).toInt().clamp(0, 255)),
    ActiveTestDef(name: 'VTC Intake угол', cmdBase: '3019', unit: '°',
      min: -64, max: 63, step: 0.5, category: 'vtc',
      formatValue: (v) => v.toStringAsFixed(1) + '°',
      toRawByte: (v) => (v * 2).toInt().clamp(-128, 127) & 0xFF),
    ActiveTestDef(name: 'Целевые об. вентилятора', cmdBase: '301C', unit: 'RPM',
      min: 0, max: 3187, step: 12.5, category: 'fan',
      formatValue: (v) => v.toStringAsFixed(0) + ' RPM',
      toRawByte: (v) => (v / 12.5).toInt().clamp(0, 255)),
    ActiveTestDef(name: 'FAN DUTY', cmdBase: '304E', unit: '%',
      min: 0, max: 100, step: 1, category: 'fan',
      formatValue: (v) => v.toStringAsFixed(0) + '%',
      toRawByte: (v) => v.toInt().clamp(0, 255)),
    ActiveTestDef(name: 'Генератор duty', cmdBase: '304F', unit: '%',
      min: 0, max: 127, step: 0.5, category: 'electric',
      formatValue: (v) => v.toStringAsFixed(1) + '%',
      toRawByte: (v) => (v * 2).toInt().clamp(0, 255)),
    ActiveTestDef(name: 'Откл. цилиндра 1', cmdBase: '300C', unit: '',
      min: 0, max: 1, step: 1, isBool: true, category: 'cylinder',
      formatValue: (v) => v > 0 ? 'ОТКЛ' : 'НОРМ', toRawByte: (v) => v > 0 ? 0x01 : 0x00),
    ActiveTestDef(name: 'Откл. цилиндра 2', cmdBase: '300C', unit: '',
      min: 0, max: 1, step: 1, isBool: true, category: 'cylinder',
      formatValue: (v) => v > 0 ? 'ОТКЛ' : 'НОРМ', toRawByte: (v) => v > 0 ? 0x02 : 0x00),
    ActiveTestDef(name: 'Откл. цилиндра 3', cmdBase: '300C', unit: '',
      min: 0, max: 1, step: 1, isBool: true, category: 'cylinder',
      formatValue: (v) => v > 0 ? 'ОТКЛ' : 'НОРМ', toRawByte: (v) => v > 0 ? 0x04 : 0x00),
    ActiveTestDef(name: 'Откл. цилиндра 4', cmdBase: '300C', unit: '',
      min: 0, max: 1, step: 1, isBool: true, category: 'cylinder',
      formatValue: (v) => v > 0 ? 'ОТКЛ' : 'НОРМ', toRawByte: (v) => v > 0 ? 0x08 : 0x00),
    ActiveTestDef(name: 'Вентилятор ОЖ HIGH', cmdBase: '300D', unit: '',
      min: 0, max: 1, step: 1, isBool: true, category: 'fan',
      formatValue: (v) => v > 0 ? 'ВКЛ' : 'ВЫКЛ', toRawByte: (v) => v > 0 ? 0x01 : 0x00),
    ActiveTestDef(name: 'Вентилятор ОЖ LOW', cmdBase: '300F', unit: '',
      min: 0, max: 1, step: 1, isBool: true, category: 'fan',
      formatValue: (v) => v > 0 ? 'ВКЛ' : 'ВЫКЛ', toRawByte: (v) => v > 0 ? 0x01 : 0x00),
    ActiveTestDef(name: 'Реле бензонасоса', cmdBase: '302D', unit: '',
      min: 0, max: 1, step: 1, isBool: true, category: 'fuel',
      formatValue: (v) => v > 0 ? 'ВКЛ' : 'ВЫКЛ', toRawByte: (v) => v > 0 ? 0x01 : 0x00),
    ActiveTestDef(name: 'VALVE TIMING SOL', cmdBase: '3031', unit: '',
      min: 0, max: 1, step: 1, isBool: true, category: 'vtc',
      formatValue: (v) => v > 0 ? 'ВКЛ' : 'ВЫКЛ', toRawByte: (v) => v > 0 ? 0x01 : 0x00),
    ActiveTestDef(name: 'EGRC Соленоид', cmdBase: '3021', unit: '',
      min: 0, max: 1, step: 1, isBool: true, category: 'egr',
      formatValue: (v) => v > 0 ? 'ВКЛ' : 'ВЫКЛ', toRawByte: (v) => v > 0 ? 0x01 : 0x00),
    ActiveTestDef(name: 'Реле кондиционера', cmdBase: '3045', unit: '',
      min: 0, max: 1, step: 1, isBool: true, category: 'other',
      formatValue: (v) => v > 0 ? 'ВКЛ' : 'ВЫКЛ', toRawByte: (v) => v > 0 ? 0x01 : 0x00),
  ];

  static final List<LearningDef> _learnings = [
    LearningDef(name: 'Обучение подачи воздуха ХХ', cmd: '3103',
      description: 'Двигатель прогрет, потребители выключены!', isDangerous: true),
    LearningDef(name: 'Обучение дросселя', cmd: '3104',
      description: 'После чистки дросселя!', isDangerous: true),
    LearningDef(name: 'СБРОС АДАПТАЦИЙ ЭБУ', cmd: '3106',
      description: 'Сбросить ВСЕ адаптации!', isDangerous: true),
    LearningDef(name: 'Обучение VTC', cmd: '310A',
      description: 'Обучение фаз газораспределения', isDangerous: true),
    LearningDef(name: 'Начальное обучение A/F', cmd: '3114',
      description: 'После замены лямбды!', isDangerous: true),
    LearningDef(name: 'Калибровка G-сенсора', cmd: '3116',
      description: 'Машина на ровной поверхности!'),
    LearningDef(name: 'Обучение нейтрали МКПП', cmd: '3110',
      description: 'Нейтральное положение МКПП'),
  ];

  @override
  void initState() { super.initState(); _tab = TabController(length: 2, vsync: this); }
  @override
  void dispose() {
    for (var t in _activeTimers.values) { t.cancel(); }
    _activeTimers.clear(); _tab.dispose(); super.dispose();
  }

  Future<void> _scanTests() async {
    if (!widget.obdService.isConnected || !widget.obdService.ecuResponds) {
      _snack('Подключитесь!', Colors.red); return;
    }
    setState(() { _scanning = true; _supported = 0; });
    for (var t in _tests) {
      final r = await widget.obdService.sendCommand(t.stopCmd, timeout: 2000);
      final rc = r.replaceAll(' ', '').toUpperCase();
      final testId = t.cmdBase.substring(2, 4);
      final expectedPositive = '70' + testId;
      t.isSupported = rc.contains(expectedPositive) && !rc.contains('7F30');
      if (t.isSupported) _supported++;
      await Future.delayed(const Duration(milliseconds: 80));
    }
    setState(() { _scanning = false; _scanned = true; });
    _snack('Найдено: ' + _supported.toString() + '/' + _tests.length.toString(), Colors.green);
  }

  Future<void> _startTest(ActiveTestDef t, double value) async {
    if (!widget.obdService.isConnected) return;
    setState(() => _busy = true);
    final cmd = t.startCmd(value);
    String r = '';
    for (int i = 0; i < 3; i++) {
      r = await widget.obdService.sendCommand(cmd, timeout: 3000);
      if (r.isNotEmpty && !r.contains('NO DATA')) break;
      await Future.delayed(const Duration(milliseconds: 300));
    }
    setState(() { _busy = false; _result = '>>> ' + cmd + ' <<< ' + r; t.isActive = true; });
    _activeTimers[t.cmdBase]?.cancel();
    _activeTimers[t.cmdBase] = Timer.periodic(const Duration(seconds: 2), (_) async {
      if (t.isActive && widget.obdService.isConnected) {
        await widget.obdService.sendCommand(cmd, timeout: 1500);
      }
    });
    _snack(t.name + ' = ' + t.formatValue(value), Colors.green);
  }

  Future<void> _stopTest(ActiveTestDef t) async {
    if (!widget.obdService.isConnected) return;
    _activeTimers[t.cmdBase]?.cancel();
    _activeTimers.remove(t.cmdBase);
    setState(() => _busy = true);
    final buf = StringBuffer();
    final r1 = await widget.obdService.sendCommand(t.stopCmd, timeout: 3000);
    buf.writeln('>>> ' + t.stopCmd + ' <<< ' + r1);
    await Future.delayed(const Duration(milliseconds: 150));
    final r2 = await widget.obdService.sendCommand('30000000', timeout: 2000);
    buf.writeln('>>> 30000000 <<< ' + r2);
    await Future.delayed(const Duration(milliseconds: 150));
    final r3 = await widget.obdService.sendCommand('30FF0000', timeout: 2000);
    buf.writeln('>>> 30FF0000 <<< ' + r3);
    await Future.delayed(const Duration(milliseconds: 150));
    await widget.obdService.sendCommand('1081', timeout: 2000);
    await Future.delayed(const Duration(milliseconds: 150));
    await widget.obdService.sendCommand('ATSH8110FC', timeout: 1000);
    await widget.obdService.sendCommand('ATFI', timeout: 3000);
    setState(() { _busy = false; _result = buf.toString(); t.isActive = false; });
    _snack(t.name + ' СТОП', Colors.green);
  }

  Future<void> _stopAll() async {
    setState(() => _busy = true);
    for (var t in _tests.where((t) => t.isActive)) {
      _activeTimers[t.cmdBase]?.cancel();
      _activeTimers.remove(t.cmdBase);
      await widget.obdService.sendCommand(t.stopCmd, timeout: 1500);
      t.isActive = false;
      await Future.delayed(const Duration(milliseconds: 80));
    }
    try {
      await widget.obdService.sendCommand('30000000', timeout: 2000);
      await widget.obdService.sendCommand('1081', timeout: 2000);
      await widget.obdService.sendCommand('ATSH8110FC', timeout: 1000);
      await widget.obdService.sendCommand('ATFI', timeout: 3000);
    } catch (e) {}
    setState(() => _busy = false);
    _snack('Все тесты сброшены', Colors.green);
  }

  Future<void> _runLearning(LearningDef l) async {
    if (!widget.obdService.isConnected) return;
    if (l.isDangerous) {
      final ok = await showDialog<bool>(context: context, builder: (c) => AlertDialog(
        backgroundColor: const Color(0xFF16213E),
        title: Text(l.name),
        content: Text(l.description + '\\n\\nУверены?', style: const TextStyle(color: Colors.white70)),
        actions: [
          TextButton(onPressed: () => Navigator.pop(c, false),
            style: TextButton.styleFrom(foregroundColor: Colors.white),
            child: const Text('Отмена')),
          TextButton(onPressed: () => Navigator.pop(c, true),
            style: TextButton.styleFrom(foregroundColor: Colors.orange),
            child: const Text('ВЫПОЛНИТЬ')),
        ]));
      if (ok != true) return;
    }
    setState(() => _busy = true);
    final r = await widget.obdService.sendCommand(l.cmd, timeout: 5000);
    setState(() { _busy = false; _result = '>>> ' + l.cmd + ' <<< ' + r; });
    if (r.contains('71')) _snack(l.name + ' УСПЕХ', Colors.green);
    else _snack(l.name + ': ' + r, Colors.orange);
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(
      content: Text(m, style: const TextStyle(color: Colors.white)),
      backgroundColor: c, duration: const Duration(seconds: 3)));
  }

  @override
  Widget build(BuildContext context) {
    final hasActive = _tests.any((t) => t.isActive);
    return Scaffold(
      appBar: AppBar(title: const Text('Сервис'), backgroundColor: const Color(0xFF16213E),
        actions: [
          FpsIndicator(obdService: widget.obdService),
          if (hasActive) IconButton(icon: const Icon(Icons.stop_circle, color: Colors.red),
            onPressed: _busy ? null : _stopAll),
        ],
        bottom: TabBar(controller: _tab, tabs: const [
          Tab(icon: Icon(Icons.build), text: 'Тесты'),
          Tab(icon: Icon(Icons.school), text: 'Обучения'),
        ])),
      body: Column(children: [
        if (!widget.obdService.ecuResponds) Container(
          width: double.infinity, padding: const EdgeInsets.all(8),
          color: Colors.red.withOpacity(0.3),
          child: const Text('⚠️ ЭБУ не подключен!', textAlign: TextAlign.center,
            style: TextStyle(color: Colors.white, fontWeight: FontWeight.bold))),
        if (_busy || _scanning) const LinearProgressIndicator(color: Color(0xFFE94560)),
        if (_result.isNotEmpty) Container(
          width: double.infinity, padding: const EdgeInsets.all(6),
          color: const Color(0xFF0F3460),
          child: Text(_result, style: const TextStyle(
            fontFamily: 'monospace', fontSize: 10, color: Colors.cyan))),
        if (hasActive) Container(
          width: double.infinity, padding: const EdgeInsets.all(6),
          color: Colors.orange.withOpacity(0.3),
          child: Row(children: [
            const Icon(Icons.warning, color: Colors.orange, size: 18),
            const SizedBox(width: 6),
            Expanded(child: Text('Активных: ' + _tests.where((t) => t.isActive).length.toString(),
              style: const TextStyle(color: Colors.white, fontWeight: FontWeight.bold, fontSize: 12))),
            ElevatedButton(onPressed: _busy ? null : _stopAll,
              style: ElevatedButton.styleFrom(
                backgroundColor: Colors.red, foregroundColor: Colors.white,
                padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 4)),
              child: const Text('СТОП ВСЕ', style: TextStyle(fontSize: 11))),
          ])),
        Expanded(child: TabBarView(controller: _tab, children: [_testsTab(), _learningsTab()])),
      ]));
  }

  Widget _testsTab() {
    var list = _cat == 'all' ? _tests : _tests.where((t) => t.category == _cat).toList();
    if (_scanned) list = list.where((t) => t.isSupported).toList();
    return Column(children: [
      if (!_scanned) Padding(padding: const EdgeInsets.all(8),
        child: ElevatedButton.icon(
          onPressed: _scanning ? null : _scanTests,
          icon: _scanning ? const SizedBox(width: 16, height: 16,
            child: CircularProgressIndicator(strokeWidth: 2, color: Colors.white))
            : const Icon(Icons.search),
          label: Text(_scanning ? 'Сканирование...' : 'СКАНИРОВАТЬ ТЕСТЫ'),
          style: ElevatedButton.styleFrom(
            backgroundColor: Colors.cyan, foregroundColor: Colors.white,
            minimumSize: const Size.fromHeight(44))))
      else Padding(padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 4),
        child: Row(children: [
          Text('Найдено: ' + _supported.toString() + '/' + _tests.length.toString(),
            style: const TextStyle(color: Colors.green, fontSize: 12, fontWeight: FontWeight.bold)),
          const Spacer(),
          TextButton(onPressed: _scanning ? null : _scanTests,
            style: TextButton.styleFrom(foregroundColor: Colors.white),
            child: const Text('Пересканировать', style: TextStyle(fontSize: 11))),
        ])),
      Expanded(child: list.isEmpty
        ? Center(child: Text(_scanned ? 'Нет тестов' : 'Нажмите СКАНИРОВАТЬ',
            style: const TextStyle(color: Colors.white54)))
        : ListView.builder(padding: const EdgeInsets.all(8), itemCount: list.length,
            itemBuilder: (c, i) => _testCard(list[i]))),
    ]);
  }

  Widget _testCard(ActiveTestDef t) {
    return Card(color: t.isActive ? Colors.green.withOpacity(0.2) : const Color(0xFF16213E),
      margin: const EdgeInsets.symmetric(vertical: 3),
      child: Padding(padding: const EdgeInsets.all(10), child: Column(
        crossAxisAlignment: CrossAxisAlignment.start, children: [
          Row(children: [
            if (t.isActive) Container(width: 10, height: 10,
              margin: const EdgeInsets.only(right: 6),
              decoration: const BoxDecoration(color: Colors.green, shape: BoxShape.circle)),
            Expanded(child: Text(t.name,
              style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 13))),
            Text(t.cmdBase, style: const TextStyle(
              fontFamily: 'monospace', fontSize: 9, color: Colors.cyan)),
          ]),
          const SizedBox(height: 8),
          if (t.isBool) Row(children: [
            Expanded(child: ElevatedButton.icon(
              onPressed: _busy ? null : () => _startTest(t, 1),
              icon: const Icon(Icons.play_arrow, size: 18),
              label: const Text('СТАРТ', style: TextStyle(fontWeight: FontWeight.bold)),
              style: ElevatedButton.styleFrom(
                backgroundColor: Colors.green, foregroundColor: Colors.white,
                minimumSize: const Size.fromHeight(42)))),
            const SizedBox(width: 10),
            Expanded(child: ElevatedButton.icon(
              onPressed: _busy ? null : () => _stopTest(t),
              icon: const Icon(Icons.stop, size: 18),
              label: const Text('СТОП', style: TextStyle(fontWeight: FontWeight.bold)),
              style: ElevatedButton.styleFrom(
                backgroundColor: Colors.red, foregroundColor: Colors.white,
                minimumSize: const Size.fromHeight(42)))),
          ])
          else _Slider(test: t, onStart: _busy ? null : (v) => _startTest(t, v),
            onStop: _busy ? null : () => _stopTest(t)),
        ])));
  }

  Widget _learningsTab() {
    return ListView.builder(padding: const EdgeInsets.all(8), itemCount: _learnings.length,
      itemBuilder: (c, i) {
        final l = _learnings[i];
        return Card(color: l.isDangerous ? Colors.orange.withOpacity(0.15) : const Color(0xFF16213E),
          margin: const EdgeInsets.symmetric(vertical: 3),
          child: ListTile(
            leading: Icon(l.isDangerous ? Icons.warning : Icons.school,
              color: l.isDangerous ? Colors.orange : Colors.cyan),
            title: Text(l.name, style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 13)),
            subtitle: Text(l.description, style: const TextStyle(color: Colors.white70, fontSize: 11)),
            trailing: ElevatedButton(onPressed: _busy ? null : () => _runLearning(l),
              style: ElevatedButton.styleFrom(
                backgroundColor: l.isDangerous ? Colors.orange : Colors.cyan,
                foregroundColor: Colors.white),
              child: Text(l.isDangerous ? 'ВЫПОЛН.' : 'СТАРТ', style: const TextStyle(fontSize: 11)))));
      });
  }
}

class _Slider extends StatefulWidget {
  final ActiveTestDef test;
  final void Function(double)? onStart;
  final void Function()? onStop;
  const _Slider({required this.test, this.onStart, this.onStop});
  @override
  State<_Slider> createState() => _SliderState();
}

class _SliderState extends State<_Slider> {
  late double _val;
  @override
  void initState() { super.initState(); _val = (widget.test.min + widget.test.max) / 2; }
  @override
  Widget build(BuildContext context) {
    return Column(children: [
      Text(widget.test.formatValue(_val),
        style: const TextStyle(color: Colors.yellow, fontWeight: FontWeight.bold, fontSize: 18)),
      Slider(value: _val.clamp(widget.test.min, widget.test.max),
        min: widget.test.min, max: widget.test.max,
        divisions: ((widget.test.max - widget.test.min) / widget.test.step).round(),
        onChanged: (v) => setState(() => _val = v)),
      Row(children: [
        Expanded(child: ElevatedButton.icon(
          onPressed: widget.onStart != null ? () => widget.onStart!(_val) : null,
          icon: const Icon(Icons.play_arrow, size: 18),
          label: const Text('СТАРТ', style: TextStyle(fontWeight: FontWeight.bold)),
          style: ElevatedButton.styleFrom(
            backgroundColor: Colors.green, foregroundColor: Colors.white,
            minimumSize: const Size.fromHeight(44)))),
        const SizedBox(width: 10),
        Expanded(child: ElevatedButton.icon(
          onPressed: widget.onStop,
          icon: const Icon(Icons.stop, size: 18),
          label: const Text('СТОП', style: TextStyle(fontWeight: FontWeight.bold)),
          style: ElevatedButton.styleFrom(
            backgroundColor: Colors.red, foregroundColor: Colors.white,
            minimumSize: const Size.fromHeight(44)))),
      ]),
    ]);
  }
}
''')
print("✅ service_screen.dart (тесты + repeater + агрессивный STOP + обучения)")

# ============ performance_screen.dart ============
with open('lib/screens/performance_screen.dart', 'w') as f:
    f.write('''import 'package:flutter/material.dart';
import '../services/obd_service.dart';
import '../services/performance_service.dart';
import '../widgets/fps_indicator.dart';

class PerformanceScreen extends StatefulWidget {
  final OBDService obdService;
  final PerformanceService performanceService;
  const PerformanceScreen({super.key, required this.obdService, required this.performanceService});
  @override
  State<PerformanceScreen> createState() => _PerformanceScreenState();
}

class _PerformanceScreenState extends State<PerformanceScreen> {
  @override
  void initState() {
    super.initState();
    widget.performanceService.runStream.listen((_) { if (mounted) setState(() {}); });
  }
  void _toggle() {
    if (widget.performanceService.isRunning || widget.performanceService.isWaiting) {
      widget.performanceService.stop();
    } else {
      if (!widget.obdService.isConnected) return;
      widget.performanceService.startWaiting();
    }
    setState(() {});
  }
  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Замер разгона'), backgroundColor: const Color(0xFF16213E),
        actions: [FpsIndicator(obdService: widget.obdService)]),
      body: Padding(padding: const EdgeInsets.all(16), child: Column(children: [
        Card(color: widget.performanceService.isRunning ? Colors.red.withOpacity(0.3)
            : widget.performanceService.isWaiting ? Colors.orange.withOpacity(0.3)
            : const Color(0xFF16213E),
          child: Padding(padding: const EdgeInsets.all(20), child: Column(children: [
            Icon(widget.performanceService.isRunning ? Icons.speed
                : widget.performanceService.isWaiting ? Icons.hourglass_bottom : Icons.timer, size: 60,
              color: widget.performanceService.isRunning ? Colors.red
                : widget.performanceService.isWaiting ? Colors.orange : Colors.blue),
            const SizedBox(height: 12),
            Text(widget.performanceService.isRunning ? 'ЗАМЕР!'
                : widget.performanceService.isWaiting ? 'ЖДУ СТАРТА...' : 'ГОТОВ',
              style: const TextStyle(fontSize: 20, fontWeight: FontWeight.bold)),
            const SizedBox(height: 16),
            SizedBox(width: double.infinity, height: 60,
              child: ElevatedButton.icon(onPressed: _toggle,
                icon: Icon(widget.performanceService.isRunning || widget.performanceService.isWaiting
                  ? Icons.stop : Icons.play_arrow, size: 32),
                label: Text(widget.performanceService.isRunning || widget.performanceService.isWaiting
                  ? 'СТОП' : 'НАЧАТЬ',
                  style: const TextStyle(fontSize: 18, fontWeight: FontWeight.bold)),
                style: ElevatedButton.styleFrom(
                  backgroundColor: widget.performanceService.isRunning || widget.performanceService.isWaiting
                    ? Colors.red : Colors.green,
                  foregroundColor: Colors.white))),
          ]))),
        const SizedBox(height: 16),
        Expanded(child: GridView.count(crossAxisCount: 2, childAspectRatio: 1.5,
          crossAxisSpacing: 8, mainAxisSpacing: 8, children: [
            _card('0-60', _fmt(widget.performanceService.time0to60), 'сек', Colors.green),
            _card('0-100', _fmt(widget.performanceService.time0to100), 'сек', Colors.blue),
            _card('400м', _fmt(widget.performanceService.time400m), 'сек', Colors.orange),
            _card('Макс скор', widget.performanceService.maxSpeed.toStringAsFixed(0), 'км/ч', Colors.purple),
            _card('Макс RPM', widget.performanceService.maxRPM.toString(), 'об', Colors.red),
            _card('Макс HP', widget.performanceService.maxHP.toStringAsFixed(0), 'л.с.', Colors.yellow),
          ])),
      ])));
  }
  Widget _card(String l, String v, String u, Color c) {
    return Card(color: const Color(0xFF16213E),
      child: Padding(padding: const EdgeInsets.all(12),
        child: Column(mainAxisAlignment: MainAxisAlignment.center, children: [
          Text(l, style: const TextStyle(color: Colors.white70, fontSize: 12)),
          const SizedBox(height: 8),
          FittedBox(child: Text(v, style: TextStyle(color: c, fontSize: 28, fontWeight: FontWeight.bold))),
          Text(u, style: TextStyle(color: c.withOpacity(0.7), fontSize: 11)),
        ])));
  }
  String _fmt(double s) => s == 0 ? '--' : s.toStringAsFixed(2);
}
''')
print("✅ performance_screen.dart")

# ============ export_screen.dart ============
with open('lib/screens/export_screen.dart', 'w') as f:
    f.write('''import 'dart:io';
import 'package:flutter/material.dart';
import 'package:path_provider/path_provider.dart';
import 'package:share_plus/share_plus.dart';
import '../services/tuning_service.dart';
import '../services/export_service.dart';

class ExportScreen extends StatefulWidget {
  const ExportScreen({super.key});
  @override
  State<ExportScreen> createState() => _ExportScreenState();
}

class _ExportScreenState extends State<ExportScreen> {
  final _t = TuningService();
  final _e = ExportService();
  List<FileSystemEntity> _files = [];

  @override
  void initState() { super.initState(); _load(); }

  Future<void> _load() async {
    try {
      final dir = await getApplicationDocumentsDirectory();
      final files = dir.listSync().where((f) => f.path.endsWith('.ols')
        || f.path.endsWith('.hex') || f.path.endsWith('.txt') || f.path.endsWith('.json')).toList();
      files.sort((a, b) => b.path.compareTo(a.path));
      setState(() => _files = files);
    } catch (e) {}
  }

  Future<void> _exp(String type, String fmt) async {
    try {
      var m = type == 'Spark' ? await _t.getSparkAdvanceMap() :
              type == 'Fuel' ? await _t.getFuelMap() :
              type == 'VTC' ? await _t.getVTCMap() : await _t.getEngineTorqueMap();
      String p = '';
      if (fmt == 'winols') p = await _e.exportToWinOLS(m);
      else if (fmt == 'ecuedit') p = await _e.exportToEcuEdit(m);
      else if (fmt == 'hex') p = await _e.exportHexPatch(m);
      ScaffoldMessenger.of(context).showSnackBar(SnackBar(
        content: Text('Сохранено: ' + p.split('/').last, style: const TextStyle(color: Colors.white)),
        backgroundColor: Colors.green));
      await _load();
    } catch (e) {}
  }

  Future<void> _share(String p) async {
    try { await Share.shareXFiles([XFile(p)]); } catch (e) {}
  }

  Future<void> _del(String p) async {
    await File(p).delete();
    await _load();
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Экспорт карт'), backgroundColor: const Color(0xFF16213E),
        actions: [IconButton(icon: const Icon(Icons.refresh), onPressed: _load)]),
      body: Column(children: [
        Padding(padding: const EdgeInsets.all(12), child: Card(color: const Color(0xFF16213E),
          child: Padding(padding: const EdgeInsets.all(12), child: Column(
            crossAxisAlignment: CrossAxisAlignment.start, children: [
              const Text('ЭКСПОРТ КАРТ', style: TextStyle(
                color: Colors.white70, fontSize: 12, fontWeight: FontWeight.bold)),
              const SizedBox(height: 8),
              ..._row('Зажигание', 'Spark'),
              const Divider(color: Colors.white24),
              ..._row('Топливо', 'Fuel'),
              const Divider(color: Colors.white24),
              ..._row('VTC', 'VTC'),
              const Divider(color: Colors.white24),
              ..._row('Момент', 'Torque'),
            ])))),
        Expanded(child: _files.isEmpty
          ? const Center(child: Text('Нет файлов', style: TextStyle(color: Colors.white54)))
          : ListView.builder(padding: const EdgeInsets.all(8), itemCount: _files.length,
              itemBuilder: (c, i) {
                final f = _files[i];
                final n = f.path.split('/').last;
                return Card(color: const Color(0xFF16213E), child: ListTile(
                  title: Text(n, style: const TextStyle(fontSize: 12)),
                  trailing: Row(mainAxisSize: MainAxisSize.min, children: [
                    IconButton(icon: const Icon(Icons.share, color: Colors.blue), onPressed: () => _share(f.path)),
                    IconButton(icon: const Icon(Icons.delete, color: Colors.red), onPressed: () => _del(f.path)),
                  ])));
              })),
      ]));
  }

  List<Widget> _row(String l, String t) {
    return [Padding(padding: const EdgeInsets.symmetric(vertical: 4),
      child: Row(children: [
        Expanded(child: Text(l)),
        IconButton(icon: const Icon(Icons.file_download, size: 20),
          onPressed: () => _exp(t, 'winols'), color: Colors.blue),
        IconButton(icon: const Icon(Icons.description, size: 20),
          onPressed: () => _exp(t, 'ecuedit'), color: Colors.purple),
        IconButton(icon: const Icon(Icons.memory, size: 20),
          onPressed: () => _exp(t, 'hex'), color: Colors.orange),
      ]))];
  }
}
''')
print("✅ export_screen.dart")

# ============ custom_pid_screen.dart ============
with open('lib/screens/custom_pid_screen.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import '../services/obd_service.dart';
import '../widgets/fps_indicator.dart';

class CustomPIDScreen extends StatefulWidget {
  final OBDService? obdService;
  const CustomPIDScreen({super.key, this.obdService});
  @override
  State<CustomPIDScreen> createState() => _CustomPIDScreenState();
}

class _CustomPIDScreenState extends State<CustomPIDScreen> {
  Timer? _t;
  String _cat = 'all';
  @override
  void initState() {
    super.initState();
    _t = Timer.periodic(const Duration(milliseconds: 500), (_) { if (mounted) setState(() {}); });
  }
  @override
  void dispose() { _t?.cancel(); super.dispose(); }

  Future<void> _copy() async {
    if (widget.obdService == null) return;
    final sb = StringBuffer();
    sb.writeln('ECU: ' + widget.obdService!.ecuId);
    for (var p in widget.obdService!.activePids) {
      final v = widget.obdService!.nissanValues[p.name] ?? 0;
      sb.writeln(p.cmd + ' [' + p.desc + '] = ' + v.toStringAsFixed(2) + ' ' + p.unit);
    }
    await Clipboard.setData(ClipboardData(text: sb.toString()));
  }

  @override
  Widget build(BuildContext context) {
    final obd = widget.obdService;
    final pids = obd?.activePids ?? [];
    final filtered = _cat == 'all' ? pids : pids.where((p) => p.category == _cat).toList();
    final cats = ['all', 'engine', 'ignition', 'fuel', 'air', 'temp', 'vtc', 'throttle', 'idle', 'electric', 'other'];
    return Scaffold(
      appBar: AppBar(title: const Text('Nissan PID'), backgroundColor: const Color(0xFF16213E),
        actions: [
          if (obd != null) FpsIndicator(obdService: obd),
          IconButton(icon: const Icon(Icons.copy), onPressed: _copy),
        ]),
      body: Column(children: [
        if (obd?.ecuId.isNotEmpty ?? false) Card(color: const Color(0xFF16213E), margin: const EdgeInsets.all(8),
          child: Padding(padding: const EdgeInsets.all(10),
            child: Text('ECU: ' + (obd?.ecuId ?? '') + ' | PID: ' + pids.length.toString(),
              style: const TextStyle(color: Colors.green)))),
        SizedBox(height: 40, child: ListView.builder(scrollDirection: Axis.horizontal,
          padding: const EdgeInsets.symmetric(horizontal: 8), itemCount: cats.length,
          itemBuilder: (c, i) => Padding(padding: const EdgeInsets.only(right: 4),
            child: FilterChip(label: Text(cats[i], style: const TextStyle(fontSize: 11)),
              selected: _cat == cats[i],
              onSelected: (_) => setState(() => _cat = cats[i]),
              backgroundColor: const Color(0xFF0F3460),
              selectedColor: const Color(0xFFE94560).withOpacity(0.5))))),
        Expanded(child: ListView.builder(padding: const EdgeInsets.all(8),
          itemCount: filtered.length, itemBuilder: (c, i) {
            final p = filtered[i];
            final v = obd?.nissanValues[p.name] ?? 0;
            return Card(color: const Color(0xFF16213E),
              child: ListTile(dense: true,
                leading: CircleAvatar(radius: 14,
                  backgroundColor: p.priority == 1 ? Colors.red
                    : p.priority == 2 ? Colors.orange : Colors.grey,
                  child: Text('P' + p.priority.toString(),
                    style: const TextStyle(fontSize: 9, color: Colors.white))),
                title: Text(p.desc, style: const TextStyle(fontSize: 12, fontWeight: FontWeight.bold)),
                subtitle: Text(p.cmd, style: const TextStyle(
                  fontFamily: 'monospace', fontSize: 10, color: Colors.cyan)),
                trailing: Text(v.toStringAsFixed(2) + ' ' + p.unit,
                  style: const TextStyle(color: Colors.yellow, fontWeight: FontWeight.bold, fontSize: 13))));
          })),
      ]));
  }
}
''')
print("✅ custom_pid_screen.dart")

# ============ profile_screen.dart ============
with open('lib/screens/profile_screen.dart', 'w') as f:
    f.write('''import 'package:flutter/material.dart';
import 'package:intl/intl.dart';
import '../services/profile_service.dart';
import '../models/vehicle_profile.dart';

class ProfileScreen extends StatefulWidget {
  final ProfileService profileService;
  const ProfileScreen({super.key, required this.profileService});
  @override
  State<ProfileScreen> createState() => _ProfileScreenState();
}

class _ProfileScreenState extends State<ProfileScreen> {
  List<VehicleProfile> _profiles = [];
  VehicleProfile? _active;

  @override
  void initState() { super.initState(); _load(); }

  Future<void> _load() async {
    final p = await widget.profileService.getAll();
    final a = await widget.profileService.getActive();
    setState(() { _profiles = p; _active = a; });
  }

  void _add() {
    final name = TextEditingController(text: 'Мой X-Trail');
    final make = TextEditingController(text: 'Nissan');
    final model = TextEditingController(text: 'X-Trail T30');
    final year = TextEditingController(text: '2004');
    final engine = TextEditingController(text: 'QR20DE');
    final disp = TextEditingController(text: '2.0');
    final ecu = TextEditingController(text: '1EQ010');
    showDialog(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: const Text('Новый профиль'),
      content: SingleChildScrollView(child: Column(mainAxisSize: MainAxisSize.min, children: [
        TextField(controller: name, decoration: const InputDecoration(labelText: 'Название')),
        TextField(controller: make, decoration: const InputDecoration(labelText: 'Марка')),
        TextField(controller: model, decoration: const InputDecoration(labelText: 'Модель')),
        TextField(controller: year, decoration: const InputDecoration(labelText: 'Год')),
        TextField(controller: engine, decoration: const InputDecoration(labelText: 'Двигатель')),
        TextField(controller: disp, decoration: const InputDecoration(labelText: 'Объём (л)'),
          keyboardType: TextInputType.number),
        TextField(controller: ecu, decoration: const InputDecoration(labelText: 'ECU')),
      ])),
      actions: [
        TextButton(onPressed: () => Navigator.pop(c),
          style: TextButton.styleFrom(foregroundColor: Colors.white),
          child: const Text('Отмена')),
        TextButton(onPressed: () async {
          if (name.text.isNotEmpty) {
            await widget.profileService.create(
              name: name.text, make: make.text, model: model.text,
              year: year.text, engine: engine.text,
              displacement: double.tryParse(disp.text) ?? 2.0,
              ecuFirmware: ecu.text);
            Navigator.pop(c);
            await _load();
          }
        },
        style: TextButton.styleFrom(foregroundColor: Colors.green),
        child: const Text('Создать')),
      ]));
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Профили'), backgroundColor: const Color(0xFF16213E),
        actions: [IconButton(icon: const Icon(Icons.add), onPressed: _add)]),
      body: _profiles.isEmpty
        ? Center(child: Column(mainAxisAlignment: MainAxisAlignment.center, children: [
            const Icon(Icons.directions_car, size: 80, color: Colors.white24),
            const SizedBox(height: 16),
            const Text('Нет профилей', style: TextStyle(color: Colors.white54, fontSize: 18)),
            const SizedBox(height: 8),
            ElevatedButton.icon(onPressed: _add,
              icon: const Icon(Icons.add), label: const Text('Создать'),
              style: ElevatedButton.styleFrom(foregroundColor: Colors.white)),
          ]))
        : ListView.builder(padding: const EdgeInsets.all(12), itemCount: _profiles.length,
            itemBuilder: (c, i) {
              final p = _profiles[i];
              final active = _active?.id == p.id;
              return Card(color: active ? Colors.green.withOpacity(0.2) : const Color(0xFF16213E),
                child: ListTile(
                  leading: CircleAvatar(backgroundColor: active ? Colors.green : const Color(0xFF0F3460),
                    child: Icon(Icons.directions_car, color: active ? Colors.white : Colors.white70)),
                  title: Text(p.name, style: const TextStyle(fontWeight: FontWeight.bold)),
                  subtitle: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                    Text(p.make + ' ' + p.model + ' ' + p.year),
                    Text(p.engine + ' | ' + p.displacement.toStringAsFixed(1) + 'L | ' + p.ecuFirmware,
                      style: const TextStyle(fontSize: 11)),
                    Text(DateFormat('dd.MM.yyyy').format(p.createdAt),
                      style: const TextStyle(fontSize: 10, color: Colors.white54)),
                  ]),
                  trailing: Row(mainAxisSize: MainAxisSize.min, children: [
                    if (active) const Icon(Icons.check_circle, color: Colors.green)
                    else IconButton(icon: const Icon(Icons.check, color: Colors.blue),
                      onPressed: () async {
                        await widget.profileService.setActive(p.id);
                        await _load();
                      }),
                    IconButton(icon: const Icon(Icons.delete, color: Colors.red),
                      onPressed: () async {
                        await widget.profileService.delete(p.id);
                        await _load();
                      }),
                  ])));
            }));
  }
}
''')
print("✅ profile_screen.dart")

# ============ terminal_screen.dart ============
with open('lib/screens/terminal_screen.dart', 'w') as f:
    f.write('''import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import '../services/obd_service.dart';
import '../widgets/fps_indicator.dart';

class TerminalScreen extends StatefulWidget {
  final OBDService obdService;
  const TerminalScreen({super.key, required this.obdService});
  @override
  State<TerminalScreen> createState() => _TerminalScreenState();
}

class _TerminalScreenState extends State<TerminalScreen> {
  final List<String> _logs = [];
  final _cmd = TextEditingController();
  final _scroll = ScrollController();

  @override
  void initState() {
    super.initState();
    widget.obdService.logStream.listen((m) {
      if (mounted) {
        setState(() {
          _logs.add(m);
          if (_logs.length > 500) _logs.removeAt(0);
        });
      }
    });
  }

  Future<void> _send() async {
    final c = _cmd.text.trim().toUpperCase();
    if (c.isEmpty) return;
    if (!widget.obdService.isConnected) return;
    setState(() => _logs.add('>>> ' + c));
    final r = await widget.obdService.sendCommand(c);
    setState(() => _logs.add('<<< ' + r));
    _cmd.clear();
  }

  void _quick(String c) { _cmd.text = c; _send(); }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Терминал'), backgroundColor: const Color(0xFF16213E),
        actions: [
          FpsIndicator(obdService: widget.obdService),
          IconButton(icon: const Icon(Icons.clear_all), onPressed: () => setState(() => _logs.clear())),
          IconButton(icon: const Icon(Icons.copy), onPressed: () {
            Clipboard.setData(ClipboardData(text: _logs.join('\\n')));
          }),
        ]),
      body: Column(children: [
        Container(padding: const EdgeInsets.all(8), color: const Color(0xFF16213E),
          child: Wrap(spacing: 6, runSpacing: 6, children: [
            _qBtn('ATZ', 'Reset'), _qBtn('ATI', 'Info'), _qBtn('ATRV', 'Volt'),
            _qBtn('0100', 'Sup'), _qBtn('010C', 'RPM'), _qBtn('010D', 'Sp'),
            _qBtn('03', 'DTC'),
          ])),
        Expanded(child: Container(color: Colors.black, padding: const EdgeInsets.all(8),
          child: SingleChildScrollView(controller: _scroll,
            child: SelectableText(_logs.join('\\n'),
              style: const TextStyle(fontFamily: 'monospace', fontSize: 12, color: Colors.green))))),
        Container(color: const Color(0xFF16213E), padding: const EdgeInsets.all(8),
          child: Row(children: [
            Expanded(child: TextField(controller: _cmd,
              style: const TextStyle(fontFamily: 'monospace'),
              decoration: const InputDecoration(hintText: 'Команда...',
                border: OutlineInputBorder(),
                contentPadding: EdgeInsets.symmetric(horizontal: 12, vertical: 8)),
              onSubmitted: (_) => _send(),
              textCapitalization: TextCapitalization.characters)),
            const SizedBox(width: 8),
            IconButton(icon: const Icon(Icons.send, color: Colors.green), onPressed: _send),
          ])),
      ]));
  }

  Widget _qBtn(String c, String l) {
    return ElevatedButton(onPressed: () => _quick(c),
      style: ElevatedButton.styleFrom(
        backgroundColor: const Color(0xFF0F3460), foregroundColor: Colors.white,
        padding: const EdgeInsets.symmetric(horizontal: 10, vertical: 4)),
      child: Column(mainAxisSize: MainAxisSize.min, children: [
        Text(c, style: const TextStyle(fontFamily: 'monospace', fontSize: 11)),
        Text(l, style: const TextStyle(fontSize: 9, color: Colors.white60)),
      ]));
  }
}
''')
print("✅ terminal_screen.dart")

print()
print("=" * 60)
print("✅ ВСЕ 15 ЭКРАНОВ ГОТОВЫ!")
print("=" * 60)
!ls lib/screens/

✅ main.dart
✅ fps_indicator.dart
✅ map_table_view.dart
✅ home_screen.dart
✅ dashboard_screen.dart

Часть A завершена. Файлы:
fps_indicator.dart  map_table_view.dart
dashboard_screen.dart
home_screen.dart
✅ settings_screen.dart (полный v4 + MAF/Speed + trip fuel)
✅ graph_screen.dart
✅ logging_screen.dart
✅ events_screen.dart
✅ dtc_screen.dart
✅ log_graph_screen.dart
✅ analyzer_screen.dart

Часть B (первые 8 экранов) готова. Осталось часть C.
✅ ecu_read_screen.dart (с АВТО + UNLOCK + терминал)
✅ service_screen.dart (тесты + repeater + агрессивный STOP + обучения)
✅ performance_screen.dart
✅ export_screen.dart
✅ custom_pid_screen.dart
✅ profile_screen.dart
✅ terminal_screen.dart

✅ ВСЕ 15 ЭКРАНОВ ГОТОВЫ!
analyzer_screen.dart	export_screen.dart	 profile_screen.dart
custom_pid_screen.dart	graph_screen.dart	 service_screen.dart
dashboard_screen.dart	home_screen.dart	 settings_screen.dart
dtc_screen.dart		logging_screen.dart	 terminal_screen.dart
ecu_read_screen.dart	log_graph_screen.dart
eve

In [5]:
# @title 🔨 Ячейка 5/5.1: БЕЗ Сборки APK v5
import os
import glob

os.chdir('/content/nissan_logger_pro_v5')

# КРИТИЧНО: Java 17 (Java 21 несовместима с compileSdk 36)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ.get('PATH', '')
os.environ['PATH'] = '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:' + os.environ['PATH']
os.environ['PUB_CACHE'] = '/content/.pub-cache'
os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['PATH'] = '/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:' + os.environ['PATH']
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/ninja'

print("=" * 60)
print("Java version:")
print("=" * 60)
!java -version 2>&1 | head -1

print()
print("=" * 60)
print("📦 ЭТАП 1: pub get")
print("=" * 60)

!flutter clean
!rm -rf /content/nissan_logger_pro_v5/build
!rm -f pubspec.lock

pub_result = !flutter pub get 2>&1
for line in pub_result[-10:]:
    print(line)

if any('failed' in l.lower() or 'error' in l.lower() for l in pub_result[-5:]):
    print("\n⚠️ pub get с проблемами, но пробуем сборку")
else:
    print("\n✅ pub get OK")

print()
print("=" * 60)
print("🔧 ЭТАП 2: Патч Bluetooth плагина")
print("=" * 60)

plugin_dirs = glob.glob('/content/.pub-cache/hosted/pub.dev/flutter_bluetooth_serial-*')
if not plugin_dirs:
    plugin_dirs = glob.glob('/root/.pub-cache/hosted/pub.dev/flutter_bluetooth_serial-*')

for plugin_dir in plugin_dirs:
    with open(f'{plugin_dir}/android/build.gradle', 'w') as f:
        f.write('''group 'io.github.edufolly.flutterbluetoothserial'
version '1.0-SNAPSHOT'

buildscript {
    repositories {
        google()
        mavenCentral()
    }
    dependencies {
        classpath 'com.android.tools.build:gradle:8.6.0'
    }
}

allprojects {
    repositories {
        google()
        mavenCentral()
    }
}

apply plugin: 'com.android.library'

android {
    namespace 'io.github.edufolly.flutterbluetoothserial'
    compileSdk 36

    compileOptions {
        sourceCompatibility JavaVersion.VERSION_17
        targetCompatibility JavaVersion.VERSION_17
    }

    defaultConfig {
        minSdk 21
    }

    lintOptions {
        disable 'InvalidPackage'
        checkReleaseBuilds false
        abortOnError false
    }
}

dependencies {
    implementation 'androidx.core:core:1.13.1'
}
''')
    with open(f'{plugin_dir}/android/src/main/AndroidManifest.xml', 'w') as f:
        f.write('''<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-feature android:name="android.hardware.bluetooth" android:required="true" />
    <uses-permission android:name="android.permission.BLUETOOTH" />
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" />
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT" />
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" />
    <uses-permission android:name="android.permission.ACCESS_FINE_LOCATION" />
    <uses-permission android:name="android.permission.ACCESS_COARSE_LOCATION" />
</manifest>
''')
    print(f"✅ Bluetooth плагин пропатчен")

Java version:
openjdk version "17.0.19" 2026-04-21

📦 ЭТАП 1: pub get
Deleting .dart_tool...                                               0ms
Deleting ephemeral...                                                0ms
Deleting Generated.xcconfig...                                       0ms
Deleting flutter_export_environment.sh...                            0ms
Deleting ephemeral...                                                0ms
+ vibration_platform_interface 0.0.3 (0.1.2 available)
+ vm_service 15.2.0
+ web 1.1.1
+ win32 5.15.0 (6.4.0 available)
+ win32_registry 2.1.0 (3.0.3 available)
+ xdg_directories 1.1.0
+ yaml 3.1.3
Changed 92 dependencies!
23 packages have newer versions incompatible with dependency constraints.
Try `flutter pub outdated` for more information.

✅ pub get OK

🔧 ЭТАП 2: Патч Bluetooth плагина
✅ Bluetooth плагин пропатчен


In [ ]:
# @title 🔨 Ячейка 5/5: Сборка APK v5 (~10-15 мин)
import os
import glob

os.chdir('/content/nissan_logger_pro_v5')

# КРИТИЧНО: Java 17 (Java 21 несовместима с compileSdk 36)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ.get('PATH', '')
os.environ['PATH'] = '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:' + os.environ['PATH']
os.environ['PUB_CACHE'] = '/content/.pub-cache'
os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['PATH'] = '/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:' + os.environ['PATH']
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/ninja'

print("=" * 60)
print("Java version:")
print("=" * 60)
!java -version 2>&1 | head -1

print()
print("=" * 60)
print("📦 ЭТАП 1: pub get")
print("=" * 60)

!flutter clean
!rm -rf /content/nissan_logger_pro_v5/build
!rm -f pubspec.lock

pub_result = !flutter pub get 2>&1
for line in pub_result[-10:]:
    print(line)

if any('failed' in l.lower() or 'error' in l.lower() for l in pub_result[-5:]):
    print("\n⚠️ pub get с проблемами, но пробуем сборку")
else:
    print("\n✅ pub get OK")

print()
print("=" * 60)
print("🔧 ЭТАП 2: Патч Bluetooth плагина")
print("=" * 60)

plugin_dirs = glob.glob('/content/.pub-cache/hosted/pub.dev/flutter_bluetooth_serial-*')
if not plugin_dirs:
    plugin_dirs = glob.glob('/root/.pub-cache/hosted/pub.dev/flutter_bluetooth_serial-*')

for plugin_dir in plugin_dirs:
    with open(f'{plugin_dir}/android/build.gradle', 'w') as f:
        f.write('''group 'io.github.edufolly.flutterbluetoothserial'
version '1.0-SNAPSHOT'

buildscript {
    repositories {
        google()
        mavenCentral()
    }
    dependencies {
        classpath 'com.android.tools.build:gradle:8.6.0'
    }
}

allprojects {
    repositories {
        google()
        mavenCentral()
    }
}

apply plugin: 'com.android.library'

android {
    namespace 'io.github.edufolly.flutterbluetoothserial'
    compileSdk 36

    compileOptions {
        sourceCompatibility JavaVersion.VERSION_17
        targetCompatibility JavaVersion.VERSION_17
    }

    defaultConfig {
        minSdk 21
    }

    lintOptions {
        disable 'InvalidPackage'
        checkReleaseBuilds false
        abortOnError false
    }
}

dependencies {
    implementation 'androidx.core:core:1.13.1'
}
''')
    with open(f'{plugin_dir}/android/src/main/AndroidManifest.xml', 'w') as f:
        f.write('''<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-feature android:name="android.hardware.bluetooth" android:required="true" />
    <uses-permission android:name="android.permission.BLUETOOTH" />
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" />
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT" />
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" />
    <uses-permission android:name="android.permission.ACCESS_FINE_LOCATION" />
    <uses-permission android:name="android.permission.ACCESS_COARSE_LOCATION" />
</manifest>
''')
    print(f"✅ Bluetooth плагин пропатчен")

if not plugin_dirs:
    print("⚠️ Bluetooth плагин не найден")

print()
print("=" * 60)
print("🔨 ЭТАП 3: СБОРКА APK v5 (~10-15 минут)")
print("=" * 60)

result = !JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64 CMAKE_MAKE_PROGRAM=/usr/bin/ninja flutter build apk --release --no-tree-shake-icons --android-skip-build-dependency-validation 2>&1

important = []
for line in result:
    ll = line.lower()
    if any(k in ll for k in ['error', 'failed', 'built', 'app-release.apk', 'exception']):
        important.append(line)

print("\n📋 Ключевые события:")
for line in important[-40:]:
    print(line)

print()
print("=" * 60)
print("📱 РЕЗУЛЬТАТ")
print("=" * 60)

apk_path = '/content/nissan_logger_pro_v5/build/app/outputs/flutter-apk/app-release.apk'
if os.path.exists(apk_path):
    size_mb = os.path.getsize(apk_path) / (1024 * 1024)
    print(f"\n🎉🎉🎉 APK v5 СОБРАН! 🎉🎉🎉")
    print(f"📁 Путь: {apk_path}")
    print(f"📏 Размер: {size_mb:.1f} MB")

    from google.colab import files
    !cp {apk_path} /content/NissanLoggerPro_v5.0_FULL.apk
    print(f"\n📥 Скачиваю APK...")
    files.download('/content/NissanLoggerPro_v5.0_FULL.apk')

    print()
    print("=" * 60)
    print("✅ ЧТО В APK:")
    print("=" * 60)
    print()
    print("📱 15 вкладок с полным функционалом:")
    print("  1. Приборы (Dashboard - все параметры + расход + мощность)")
    print("  2. Графики (4 профиля с реальными данными)")
    print("  3. ЛогГраф (загрузка CSV + отображение)")
    print("  4. Лог (start/stop/автолог/список файлов/share)")
    print("  5. События (алерты с snapshot + explanation)")
    print("  6. DTC (сканирование + очистка + русские описания)")
    print("  7. Анализ (загрузка CSV + анализ карт + просмотр в 3D)")
    print("  8. ЭБУ Карты (АВТО + UNLOCK 12 алгоритмов + чтение + терминал)")
    print("  9. Сервис (тесты с repeater + агрессивный STOP + обучения)")
    print("  10. Замер (0-60, 0-100, 400м)")
    print("  11. Экспорт (WinOLS/ecuEdit/HEX + share)")
    print("  12. PID (все активные + фильтр по категориям)")
    print("  13. Авто (профили автомобилей)")
    print("  14. Терминал (общий OBD терминал)")
    print("  15. Настройки (ИНИЦИАЛИЗАЦИЯ ЭБУ + MAF + Speed + всё)")
    print()
    print("🔧 Все критичные фиксы:")
    print("  ✅ Java 17 + NDK 28 + compileSdk 36")
    print("  ✅ Pause polling во время sendCommand (не залипают ответы)")
    print("  ✅ Точный скан тестов через '70+testId'")
    print("  ✅ Агрессивный _stopTest (6 команд сброса)")
    print("  ✅ Repeater 2 сек для активных тестов")
    print("  ✅ MAF slider 0.01-5.0 + 5 пресетов")
    print("  ✅ Speed multiplier 0.80-1.30 + 5 пресетов")
    print("  ✅ Alert snapshot + explanation")
    print("  ✅ Trip fuel integrator")
    print("  ✅ Кнопка ИНИЦИАЛИЗАЦИЯ ЭБУ в Настройках")
    print("  ✅ MapStorage интегрирован в TuningService")
    print("  ✅ MapTableView с Save/Reset правок")
    print("  ✅ Parser() в FormulaEvaluator (правильно для 2.5.0)")
    print()
    print("🚗 УСТАНОВКА:")
    print("1. NissanLoggerPro_v5.0_FULL.apk из загрузок")
    print("2. Установи, разреши все разрешения")
    print("3. Настройки → CONNECT твой ELM327")
    print("4. Нажми ИНИЦИАЛИЗАЦИЯ ЭБУ (большая deepOrange кнопка)")
    print("5. Настрой MAF множитель (пресет 0.5-2.0 обычно подходит)")
    print("6. Настрой Speed multiplier если приборка врёт")
    print()
    print("💡 Всё должно работать как в твоём рабочем v5!")
else:
    print("\n❌ APK НЕ СОБРАН")
    print()
    print("Последние 80 строк лога:")
    print("=" * 60)
    for line in result[-80:]:
        print(line)
    print()
    print("Скинь ошибки — быстро пофикшу.")

Java version:
openjdk version "17.0.19" 2026-04-21

📦 ЭТАП 1: pub get
Deleting .dart_tool...                                               0ms
Deleting ephemeral...                                                0ms
Deleting Generated.xcconfig...                                       0ms
Deleting flutter_export_environment.sh...                            0ms
Deleting ephemeral...                                                0ms
+ vibration_platform_interface 0.0.3 (0.1.2 available)
+ vm_service 15.2.0
+ web 1.1.1
+ win32 5.15.0 (6.4.0 available)
+ win32_registry 2.1.0 (3.0.3 available)
+ xdg_directories 1.1.0
+ yaml 3.1.3
Changed 92 dependencies!
23 packages have newer versions incompatible with dependency constraints.
Try `flutter pub outdated` for more information.

✅ pub get OK

🔧 ЭТАП 2: Патч Bluetooth плагина
✅ Bluetooth плагин пропатчен

🔨 ЭТАП 3: СБОРКА APK v5 (~10-15 минут)

📋 Ключевые события:
✓ Built build/app/outputs/flutter-apk/app-release.apk (58.6MB)

📱 РЕЗУЛЬТАТ

🎉🎉🎉 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ ЧТО В APK:

📱 15 вкладок с полным функционалом:
  1. Приборы (Dashboard - все параметры + расход + мощность)
  2. Графики (4 профиля с реальными данными)
  3. ЛогГраф (загрузка CSV + отображение)
  4. Лог (start/stop/автолог/список файлов/share)
  5. События (алерты с snapshot + explanation)
  6. DTC (сканирование + очистка + русские описания)
  7. Анализ (загрузка CSV + анализ карт + просмотр в 3D)
  8. ЭБУ Карты (АВТО + UNLOCK 12 алгоритмов + чтение + терминал)
  9. Сервис (тесты с repeater + агрессивный STOP + обучения)
  10. Замер (0-60, 0-100, 400м)
  11. Экспорт (WinOLS/ecuEdit/HEX + share)
  12. PID (все активные + фильтр по категориям)
  13. Авто (профили автомобилей)
  14. Терминал (общий OBD терминал)
  15. Настройки (ИНИЦИАЛИЗАЦИЯ ЭБУ + MAF + Speed + всё)

🔧 Все критичные фиксы:
  ✅ Java 17 + NDK 28 + compileSdk 36
  ✅ Pause polling во время sendCommand (не залипают ответы)
  ✅ Точный скан тестов через '70+testId'
  ✅ Агрессивный _stopTest (6 команд сброса)
  ✅ Repeater 2

In [6]:
# @title 🔓 UNLOCK v2: Ручной ввод формул + расширенный перебор + логирование
import os
os.chdir('/content/nissan_logger_pro_v5')

with open('lib/screens/ecu_read_screen.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'dart:io';
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import 'package:intl/intl.dart';
import 'package:path_provider/path_provider.dart';
import 'package:share_plus/share_plus.dart';
import 'package:math_expressions/math_expressions.dart';
import '../models/custom_map_def.dart';
import '../models/tuning_map.dart';
import '../services/obd_service.dart';
import '../services/ecu_map_reader.dart';
import '../services/map_storage_service.dart';
import '../services/settings_service.dart';
import '../widgets/fps_indicator.dart';
import '../widgets/map_table_view.dart';

class EcuReadScreen extends StatefulWidget {
  final OBDService obdService;
  const EcuReadScreen({super.key, required this.obdService});
  @override
  State<EcuReadScreen> createState() => _EcuReadScreenState();
}

class _EcuReadScreenState extends State<EcuReadScreen>
    with SingleTickerProviderStateMixin {
  late TabController _tab;
  late EcuMapReaderService _reader;
  bool _isDetecting = false;
  MemReadCommand? _detectedCommand;
  final List<String> _detectLog = [];
  Map<String, double> _readProgress = {};
  Map<String, EcuMapReadResult> _readResults = {};
  final TextEditingController _termCmd = TextEditingController();
  final ScrollController _termScroll = ScrollController();
  final List<String> _termLog = [];
  bool _isUnlocking = false;
  bool _isUnlocked = false;

  // v2: сохранённые пары seed→key для реверса
  final List<Map<String, String>> _seedKeyPairs = [];

  static final List<CustomMapDef> _standardMaps = [
    CustomMapDef(id: 'std_torque', name: 'Engine Torque',
      address: 0x7C3C, rows: 16, cols: 16, isUInt16: true,
      formula: '(X-32768)/10.24', units: 'Nm', createdAt: DateTime(2025)),
    CustomMapDef(id: 'std_force', name: 'Powertrain Force',
      address: 0xA2BC, rows: 16, cols: 16, isUInt16: true,
      formula: 'X-32768', units: 'N', createdAt: DateTime(2025)),
    CustomMapDef(id: 'std_vtc', name: 'VTC Intake Cam',
      address: 0x6BF1, rows: 8, cols: 8, isUInt16: false,
      formula: '(X-128)/2', units: 'deg', createdAt: DateTime(2025)),
    CustomMapDef(id: 'std_spark', name: 'Spark Advance WOT',
      address: 0x6EBC, rows: 16, cols: 16, isUInt16: true,
      formula: '(X-16384)/128', units: 'deg', createdAt: DateTime(2025)),
    CustomMapDef(id: 'std_ve', name: 'Fresh Air Rate / VE',
      address: 0xA754, rows: 16, cols: 15, isUInt16: true,
      formula: 'X/256', units: '%', createdAt: DateTime(2025)),
    CustomMapDef(id: 'std_lambda', name: 'Enrichment Lambda',
      address: 0x6521, rows: 8, cols: 8, isUInt16: false,
      formula: 'X/128', units: 'Lambda', createdAt: DateTime(2025)),
    CustomMapDef(id: 'std_idle', name: 'Desired Idle Speed',
      address: 0x8A7B, rows: 1, cols: 16, isUInt16: false,
      formula: 'X*10', units: 'RPM', createdAt: DateTime(2025)),
  ];

  @override
  void initState() {
    super.initState();
    _tab = TabController(length: 3, vsync: this);
    _reader = EcuMapReaderService(widget.obdService);
    _detectedCommand = _reader.getSavedCommand();
  }
  @override
  void dispose() {
    _tab.dispose(); _termCmd.dispose(); _termScroll.dispose(); super.dispose();
  }

  Future<void> _autoDetect() async {
    if (!widget.obdService.isConnected || !widget.obdService.ecuResponds) {
      _snack('Подключитесь к ЭБУ!', Colors.red); return;
    }
    setState(() { _isDetecting = true; _detectLog.clear(); });
    final cmd = await _reader.autoDetectCommand(
      onProgress: (msg) => setState(() => _detectLog.add(msg)));
    setState(() { _isDetecting = false; _detectedCommand = cmd; });
    if (cmd != null) _snack('OK ' + cmd.description, Colors.green);
    else _snack('Не найдено', Colors.red);
  }

  // v2: получить свежий seed
  Future<String?> _requestFreshSeed() async {
    final resp = await widget.obdService.sendCommand('2701', timeout: 3000);
    final clean = resp.replaceAll(' ', '').toUpperCase();
    if (!clean.startsWith('6701') || clean.length < 12) return null;
    return clean.substring(4, 12);
  }

  // v2: попробовать key на текущем seed
  Future<String> _sendKey(String keyHex) async {
    final resp = await widget.obdService.sendCommand('2702' + keyHex, timeout: 3000);
    return resp.replaceAll(' ', '').toUpperCase();
  }

  // v2: расширенный перебор с фиксом регенерации seed
  Future<void> _tryUnlockAuto() async {
    if (!widget.obdService.isConnected || !widget.obdService.ecuResponds) {
      _snack('Подключитесь!', Colors.red); return;
    }
    setState(() {
      _isUnlocking = true;
      _detectLog.clear();
      _detectLog.add('=== ПЕРЕБОР 30+ АЛГОРИТМОВ ===');
    });

    // ВАЖНО: seed для КАЖДОЙ попытки запрашиваем свежий
    // потому что после 7F 27 35 (invalid) ЭБУ регенерирует seed

    for (int attempt = 0; attempt < 30; attempt++) {
      final seedHex = await _requestFreshSeed();
      if (seedHex == null) {
        _detectLog.add('❌ Не получен seed на попытке ' + attempt.toString());
        setState(() { _isUnlocking = false; });
        _snack('Seed не получен', Colors.red);
        return;
      }

      final seed = <int>[];
      for (int i = 0; i < 8; i += 2) {
        seed.add(int.parse(seedHex.substring(i, i + 2), radix: 16));
      }
      final s = (seed[0] << 24) | (seed[1] << 16) | (seed[2] << 8) | seed[3];

      int keyInt = 0;
      String algName = '';

      // Выбираем алгоритм по номеру попытки
      switch (attempt) {
        case 0: keyInt = ~s & 0xFFFFFFFF; algName = 'NOT'; break;
        case 1: keyInt = s ^ 0xFFFFFFFF; algName = 'XOR FFFFFFFF'; break;
        case 2: keyInt = (s + 0x12345678) & 0xFFFFFFFF; algName = '+ 12345678'; break;
        case 3: keyInt = (s - 0x12345678) & 0xFFFFFFFF; algName = '- 12345678'; break;
        case 4: keyInt = (((s << 3) | (s >> 29)) ^ s) & 0xFFFFFFFF; algName = 'ROT3 XOR'; break;
        case 5: keyInt = (((s << 7) | (s >> 25)) ^ s) & 0xFFFFFFFF; algName = 'ROT7 XOR'; break;
        case 6: keyInt = (((s << 13) | (s >> 19)) ^ s) & 0xFFFFFFFF; algName = 'ROT13 XOR'; break;
        case 7: keyInt = (s + 0x1971) & 0xFFFFFFFF; algName = '+ 1971'; break;
        case 8: keyInt = s ^ 0x5A5A5A5A; algName = 'XOR 5A5A5A5A'; break;
        case 9: keyInt = s ^ 0xA5A5A5A5; algName = 'XOR A5A5A5A5'; break;
        case 10: keyInt = ((seed[3] << 24) | (seed[2] << 16) | (seed[1] << 8) | seed[0]); algName = 'REVERSE'; break;
        case 11: keyInt = ((s * 0x91) + 0x1971) & 0xFFFFFFFF; algName = 'Nissan cl'; break;
        case 12: keyInt = ((s ^ 0xAAAAAAAA) + 0x1971) & 0xFFFFFFFF; algName = 'XOR AAAA + 1971'; break;
        case 13: keyInt = ((s << 1) | (s >> 31)) & 0xFFFFFFFF; algName = 'ROT1'; break;
        case 14: keyInt = ((s << 5) ^ (s >> 3)) & 0xFFFFFFFF; algName = 'ROT5 XOR3'; break;
        case 15: keyInt = (s * 0x1EF3) & 0xFFFFFFFF; algName = '* 1EF3'; break;
        case 16: keyInt = (s + 0xDEADBEEF) & 0xFFFFFFFF; algName = '+ DEADBEEF'; break;
        case 17: keyInt = (s ^ 0xDEADBEEF); algName = 'XOR DEADBEEF'; break;
        case 18: keyInt = ((~s) + 1) & 0xFFFFFFFF; algName = 'NEG'; break;
        case 19: keyInt = (s ^ 0x1971A5CE) & 0xFFFFFFFF; algName = 'XOR 1971A5CE'; break;
        case 20: keyInt = ((s >> 8) | (s << 24)) & 0xFFFFFFFF; algName = 'ROT8'; break;
        case 21: keyInt = ((s >> 16) | (s << 16)) & 0xFFFFFFFF; algName = 'ROT16'; break;
        case 22: keyInt = ((seed[1] << 24) | (seed[0] << 16) | (seed[3] << 8) | seed[2]); algName = 'SWAP16'; break;
        case 23: keyInt = ((seed[0] ^ 0xFF) << 24) | ((seed[1] ^ 0xFF) << 16) | ((seed[2] ^ 0xFF) << 8) | (seed[3] ^ 0xFF); algName = 'NOT bytes'; break;
        case 24: keyInt = (s + 0xABCD1234) & 0xFFFFFFFF; algName = '+ ABCD1234'; break;
        case 25: keyInt = (s * 3 + 0x1971) & 0xFFFFFFFF; algName = '*3 + 1971'; break;
        case 26: keyInt = ((s << 4) ^ (s >> 4)) & 0xFFFFFFFF; algName = 'ROT4 XOR-4'; break;
        case 27: keyInt = (s + 0xF0F0F0F0) & 0xFFFFFFFF; algName = '+ F0F0F0F0'; break;
        case 28: keyInt = (~(s ^ 0x5A5A5A5A)) & 0xFFFFFFFF; algName = 'NOT XOR 5A'; break;
        case 29: keyInt = ((s + 0x1971) ^ 0x5A5A5A5A) & 0xFFFFFFFF; algName = '+ 1971 XOR 5A'; break;
      }

      final keyHex = keyInt.toRadixString(16).padLeft(8, '0').toUpperCase();
      _detectLog.add('#' + attempt.toString() + ' seed=' + seedHex + ' алг=' + algName);
      _detectLog.add('   key=' + keyHex);
      setState(() {});

      // Отправляем этот key на СВЕЖИЙ seed (мы только что запросили)
      final resp = await _sendKey(keyHex);

      // Сохраняем пару для реверса
      _seedKeyPairs.add({
        'seed': seedHex,
        'key_tried': keyHex,
        'algorithm': algName,
        'response': resp,
        'timestamp': DateTime.now().toIso8601String(),
      });

      if (resp.startsWith('6702')) {
        _detectLog.add('   ✅✅✅ УСПЕХ! Алгоритм: ' + algName);
        setState(() { _isUnlocking = false; _isUnlocked = true; });
        _snack('РАЗБЛОКИРОВАНО!', Colors.green);
        return;
      } else if (resp.startsWith('7F2736')) {
        _detectLog.add('   ⏸ Превышено число попыток, жди 10 сек...');
        setState(() {});
        await Future.delayed(const Duration(seconds: 11));
        _detectLog.add('   Продолжаем...');
      } else if (resp.startsWith('7F2735')) {
        _detectLog.add('   ❌ Invalid key');
      } else {
        _detectLog.add('   ? ' + resp);
      }
      setState(() {});
      await Future.delayed(const Duration(milliseconds: 200));
    }

    _detectLog.add('');
    _detectLog.add('❌ Стандартные алгоритмы не подошли');
    _detectLog.add('Используй "СВОЯ ФОРМУЛА" или "ЛОГ пар"');
    setState(() { _isUnlocking = false; });
    _snack('Стандартные не подошли', Colors.orange);
  }

  // v2: ручной ввод формулы key
  Future<void> _customFormulaDialog() async {
    final formulaCtrl = TextEditingController();
    final seedCtrl = TextEditingController();
    String? currentSeed;
    String? computedKey;
    String? responseText;

    // Пробуем получить свежий seed для отображения
    if (widget.obdService.isConnected && widget.obdService.ecuResponds) {
      currentSeed = await _requestFreshSeed();
      if (currentSeed != null) seedCtrl.text = currentSeed;
    }

    if (!mounted) return;

    await showDialog(context: context, builder: (c) {
      return StatefulBuilder(builder: (c, setD) {
        return AlertDialog(
          backgroundColor: const Color(0xFF16213E),
          title: const Text('Своя формула Key', style: TextStyle(fontSize: 16)),
          content: SingleChildScrollView(child: Column(
            crossAxisAlignment: CrossAxisAlignment.start,
            mainAxisSize: MainAxisSize.min,
            children: [
              const Text('Переменная: SEED (32-битное число)',
                style: TextStyle(color: Colors.white70, fontSize: 11)),
              const SizedBox(height: 8),
              TextField(controller: seedCtrl,
                decoration: const InputDecoration(
                  labelText: 'Seed (hex, 8 символов)',
                  hintText: '8652D32D',
                  border: OutlineInputBorder(),
                  isDense: true),
                style: const TextStyle(fontFamily: 'monospace')),
              const SizedBox(height: 8),
              TextField(controller: formulaCtrl,
                decoration: const InputDecoration(
                  labelText: 'Формула key от SEED',
                  hintText: '(SEED + 4660) ^ 23130',
                  helperText: 'Только десятичные числа! HEX не работает',
                  border: OutlineInputBorder(),
                  isDense: true),
                style: const TextStyle(fontFamily: 'monospace')),
              const SizedBox(height: 8),
              const Text('Примеры формул:',
                style: TextStyle(color: Colors.orange, fontSize: 11, fontWeight: FontWeight.bold)),
              const Text('  SEED + 4660           (то же что SEED + 0x1234)',
                style: TextStyle(color: Colors.white70, fontSize: 10, fontFamily: 'monospace')),
              const Text('  SEED * 145 + 6513     (SEED * 0x91 + 0x1971)',
                style: TextStyle(color: Colors.white70, fontSize: 10, fontFamily: 'monospace')),
              const Text('  (SEED + 6513) ^ 90    (сложение потом XOR)',
                style: TextStyle(color: Colors.white70, fontSize: 10, fontFamily: 'monospace')),
              const Text('  4294967295 - SEED     (то же что NOT для 32-бит)',
                style: TextStyle(color: Colors.white70, fontSize: 10, fontFamily: 'monospace')),
              const SizedBox(height: 12),
              if (computedKey != null) Container(
                padding: const EdgeInsets.all(6),
                decoration: BoxDecoration(color: Colors.green.withOpacity(0.2),
                  borderRadius: BorderRadius.circular(4)),
                child: Text('Key = ' + computedKey!,
                  style: const TextStyle(color: Colors.green, fontFamily: 'monospace', fontSize: 13))),
              if (responseText != null) Padding(padding: const EdgeInsets.only(top: 6),
                child: Text('Ответ ЭБУ: ' + responseText!,
                  style: TextStyle(
                    color: responseText!.startsWith('6702') ? Colors.green : Colors.red,
                    fontFamily: 'monospace', fontSize: 12))),
            ])),
          actions: [
            TextButton(onPressed: () => Navigator.pop(c),
              style: TextButton.styleFrom(foregroundColor: Colors.white),
              child: const Text('Закрыть')),
            // Кнопка "Вычислить" (без отправки)
            TextButton(onPressed: () {
              try {
                final seedStr = seedCtrl.text.trim();
                if (seedStr.length != 8) {
                  setD(() { responseText = 'Seed должен быть 8 hex символов'; });
                  return;
                }
                final seedVal = int.parse(seedStr, radix: 16);
                final parser = Parser();
                final expr = parser.parse(formulaCtrl.text.replaceAll('SEED', 'seed'));
                final cm = ContextModel();
                cm.bindVariable(Variable('seed'), Number(seedVal));
                final result = expr.evaluate(EvaluationType.REAL, cm);
                final keyInt = (result as num).toInt() & 0xFFFFFFFF;
                setD(() {
                  computedKey = keyInt.toRadixString(16).padLeft(8, '0').toUpperCase();
                  responseText = null;
                });
              } catch (e) {
                setD(() { responseText = 'Ошибка формулы: ' + e.toString(); });
              }
            },
              style: TextButton.styleFrom(foregroundColor: Colors.cyan),
              child: const Text('ВЫЧИСЛИТЬ')),
            // Кнопка "Отправить" (запрашивает свежий seed + отправляет)
            TextButton(onPressed: () async {
              if (!widget.obdService.isConnected || !widget.obdService.ecuResponds) {
                setD(() { responseText = 'Нет подключения к ЭБУ'; });
                return;
              }
              try {
                // Запрашиваем СВЕЖИЙ seed
                final freshSeed = await _requestFreshSeed();
                if (freshSeed == null) {
                  setD(() { responseText = 'Не получен свежий seed'; });
                  return;
                }
                setD(() { seedCtrl.text = freshSeed; });

                final seedVal = int.parse(freshSeed, radix: 16);
                final parser = Parser();
                final expr = parser.parse(formulaCtrl.text.replaceAll('SEED', 'seed'));
                final cm = ContextModel();
                cm.bindVariable(Variable('seed'), Number(seedVal));
                final result = expr.evaluate(EvaluationType.REAL, cm);
                final keyInt = (result as num).toInt() & 0xFFFFFFFF;
                final keyHex = keyInt.toRadixString(16).padLeft(8, '0').toUpperCase();

                setD(() { computedKey = keyHex; });

                final resp = await _sendKey(keyHex);

                // Сохраняем пару
                _seedKeyPairs.add({
                  'seed': freshSeed,
                  'key_tried': keyHex,
                  'algorithm': 'CUSTOM: ' + formulaCtrl.text,
                  'response': resp,
                  'timestamp': DateTime.now().toIso8601String(),
                });

                setD(() { responseText = resp; });

                if (resp.startsWith('6702')) {
                  setState(() => _isUnlocked = true);
                  await Future.delayed(const Duration(seconds: 2));
                  if (mounted) Navigator.pop(c);
                  _snack('РАЗБЛОКИРОВАНО!', Colors.green);
                }
              } catch (e) {
                setD(() { responseText = 'Ошибка: ' + e.toString(); });
              }
            },
              style: TextButton.styleFrom(foregroundColor: Colors.green),
              child: const Text('ОТПРАВИТЬ')),
          ]);
      });
    });
  }

  // v2: прямой ввод key (без формулы)
  Future<void> _directKeyDialog() async {
    final keyCtrl = TextEditingController();
    String? currentSeed;
    String? responseText;

    if (widget.obdService.isConnected && widget.obdService.ecuResponds) {
      currentSeed = await _requestFreshSeed();
    }

    if (!mounted) return;

    await showDialog(context: context, builder: (c) {
      return StatefulBuilder(builder: (c, setD) {
        return AlertDialog(
          backgroundColor: const Color(0xFF16213E),
          title: const Text('Прямой Key (без формулы)', style: TextStyle(fontSize: 16)),
          content: Column(mainAxisSize: MainAxisSize.min,
            crossAxisAlignment: CrossAxisAlignment.start, children: [
              if (currentSeed != null) Text('Свежий Seed: ' + currentSeed!,
                style: const TextStyle(color: Colors.cyan, fontFamily: 'monospace')),
              const SizedBox(height: 8),
              TextField(controller: keyCtrl,
                decoration: const InputDecoration(
                  labelText: 'Key (hex, 8 символов)',
                  hintText: '12345678',
                  border: OutlineInputBorder(),
                  isDense: true),
                style: const TextStyle(fontFamily: 'monospace')),
              const SizedBox(height: 8),
              const Text('Если ты знаешь готовый key для этого seed —\\nвведи и жми ОТПРАВИТЬ',
                style: TextStyle(color: Colors.white70, fontSize: 11)),
              const SizedBox(height: 8),
              if (responseText != null) Text('Ответ: ' + responseText!,
                style: TextStyle(
                  color: responseText!.startsWith('6702') ? Colors.green : Colors.red,
                  fontFamily: 'monospace', fontSize: 12)),
            ]),
          actions: [
            TextButton(onPressed: () => Navigator.pop(c),
              style: TextButton.styleFrom(foregroundColor: Colors.white),
              child: const Text('Закрыть')),
            TextButton(onPressed: () async {
              final key = keyCtrl.text.trim().toUpperCase();
              if (key.length != 8) {
                setD(() { responseText = 'Key должен быть 8 hex символов'; });
                return;
              }
              final resp = await _sendKey(key);
              _seedKeyPairs.add({
                'seed': currentSeed ?? '',
                'key_tried': key,
                'algorithm': 'DIRECT',
                'response': resp,
                'timestamp': DateTime.now().toIso8601String(),
              });
              setD(() { responseText = resp; });
              if (resp.startsWith('6702')) {
                setState(() => _isUnlocked = true);
                await Future.delayed(const Duration(seconds: 2));
                if (mounted) Navigator.pop(c);
                _snack('РАЗБЛОКИРОВАНО!', Colors.green);
              }
            },
              style: TextButton.styleFrom(foregroundColor: Colors.green),
              child: const Text('ОТПРАВИТЬ')),
          ]);
      });
    });
  }

  // v2: показать лог пар seed→key для реверса
  Future<void> _showPairsLog() async {
    await showDialog(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: Row(children: [
        Expanded(child: Text('Пары seed→key (' + _seedKeyPairs.length.toString() + ')',
          style: const TextStyle(fontSize: 14))),
        IconButton(icon: const Icon(Icons.copy, size: 18),
          onPressed: () async {
            final text = _pairsAsText();
            await Clipboard.setData(ClipboardData(text: text));
            _snack('Скопировано', Colors.green);
          }),
        IconButton(icon: const Icon(Icons.share, size: 18),
          onPressed: () async {
            try {
              final dir = await getApplicationDocumentsDirectory();
              final ts = DateFormat('yyyyMMdd_HHmmss').format(DateTime.now());
              final file = File(dir.path + '/seedkey_pairs_' + ts + '.txt');
              await file.writeAsString(_pairsAsText());
              await Share.shareXFiles([XFile(file.path)]);
            } catch (e) {}
          }),
      ]),
      content: SizedBox(width: double.maxFinite, child: _seedKeyPairs.isEmpty
        ? const Text('Пока нет попыток', style: TextStyle(color: Colors.white70))
        : SingleChildScrollView(child: Column(
            crossAxisAlignment: CrossAxisAlignment.start,
            children: _seedKeyPairs.reversed.map((p) {
              final isSuccess = (p['response'] ?? '').startsWith('6702');
              return Container(
                margin: const EdgeInsets.symmetric(vertical: 2),
                padding: const EdgeInsets.all(6),
                decoration: BoxDecoration(
                  color: isSuccess ? Colors.green.withOpacity(0.2) : Colors.black26,
                  borderRadius: BorderRadius.circular(4)),
                child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                  Text('seed: ' + (p['seed'] ?? ''),
                    style: const TextStyle(color: Colors.cyan, fontFamily: 'monospace', fontSize: 10)),
                  Text('key:  ' + (p['key_tried'] ?? ''),
                    style: const TextStyle(color: Colors.yellow, fontFamily: 'monospace', fontSize: 10)),
                  Text('alg:  ' + (p['algorithm'] ?? ''),
                    style: const TextStyle(color: Colors.white70, fontSize: 9)),
                  Text('resp: ' + (p['response'] ?? ''),
                    style: TextStyle(
                      color: isSuccess ? Colors.green : Colors.red,
                      fontFamily: 'monospace', fontSize: 10)),
                ]));
            }).toList()))),
      actions: [
        TextButton(onPressed: () { _seedKeyPairs.clear(); Navigator.pop(c); },
          style: TextButton.styleFrom(foregroundColor: Colors.red),
          child: const Text('ОЧИСТИТЬ')),
        TextButton(onPressed: () => Navigator.pop(c),
          style: TextButton.styleFrom(foregroundColor: Colors.white),
          child: const Text('Закрыть')),
      ]));
  }

  String _pairsAsText() {
    final sb = StringBuffer();
    sb.writeln('Nissan 1EQ010 seed/key pairs log');
    sb.writeln('Generated: ' + DateTime.now().toIso8601String());
    sb.writeln('ECU: ' + widget.obdService.ecuId);
    sb.writeln('=' * 60);
    for (var i = 0; i < _seedKeyPairs.length; i++) {
      final p = _seedKeyPairs[i];
      sb.writeln('#' + (i+1).toString());
      sb.writeln('  seed:     ' + (p['seed'] ?? ''));
      sb.writeln('  key:      ' + (p['key_tried'] ?? ''));
      sb.writeln('  alg:      ' + (p['algorithm'] ?? ''));
      sb.writeln('  response: ' + (p['response'] ?? ''));
      sb.writeln('  time:     ' + (p['timestamp'] ?? ''));
      sb.writeln('');
    }
    return sb.toString();
  }

  Future<void> _readMap(CustomMapDef def) async {
    if (_detectedCommand == null) { _snack('Сначала АВТО', Colors.orange); return; }
    setState(() => _readProgress[def.addressHex] = 0.0);
    try {
      final result = await _reader.readMap(def: def, command: _detectedCommand,
        onProgress: (done, total) => setState(() => _readProgress[def.addressHex] = done / total * 100));
      if (result != null) {
        setState(() {
          _readResults[def.addressHex] = result;
          _readProgress.remove(def.addressHex);
        });
      } else {
        setState(() => _readProgress.remove(def.addressHex));
      }
    } catch (e) {
      setState(() => _readProgress.remove(def.addressHex));
    }
  }

  Future<void> _saveAsDefault(EcuMapReadResult r) async {
    final m = TuningMap(name: r.def.name, address: r.def.addressHex,
      rows: r.def.rows, cols: r.def.cols,
      rpmAxis: r.def.yAxis ?? List.generate(r.def.rows, (i) => i.toDouble()),
      loadAxis: r.def.xAxis ?? List.generate(r.def.cols, (i) => i.toDouble()),
      data: r.data, units: r.def.units);
    await MapStorageService.saveMap(m);
    _snack('Сохранено', Colors.green);
  }

  void _viewReadMap(EcuMapReadResult r) {
    final m = TuningMap(name: r.def.name + ' (из ЭБУ)', address: r.def.addressHex,
      rows: r.def.rows, cols: r.def.cols,
      rpmAxis: r.def.yAxis ?? List.generate(r.def.rows, (i) => i * 400.0),
      loadAxis: r.def.xAxis ?? List.generate(r.def.cols, (i) => i * 10.0),
      data: r.data, units: r.def.units);
    Navigator.push(context, MaterialPageRoute(
      builder: (c) => MapFullscreenView(originalMap: m, updatedMap: null, changes: const [])));
  }

  Future<void> _sendTermCmd() async {
    final cmd = _termCmd.text.trim().toUpperCase();
    if (cmd.isEmpty) return;
    if (!widget.obdService.isConnected) { _snack('Нет подключения', Colors.red); return; }
    setState(() => _termLog.add('>>> ' + cmd));
    final r = await _reader.testRawCommand(cmd);
    setState(() => _termLog.add('<<< ' + r));
    _termCmd.clear();
  }

  void _quick(String c) { _termCmd.text = c; _sendTermCmd(); }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(
      content: Text(m, style: const TextStyle(color: Colors.white)), backgroundColor: c));
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('ЭБУ Карты'), backgroundColor: const Color(0xFF16213E),
        actions: [FpsIndicator(obdService: widget.obdService)],
        bottom: TabBar(controller: _tab, tabs: const [
          Tab(icon: Icon(Icons.list), text: 'Карты'),
          Tab(icon: Icon(Icons.add_box), text: 'Свои'),
          Tab(icon: Icon(Icons.terminal), text: 'Терминал'),
        ])),
      body: Column(children: [
        _statusPanel(),
        Expanded(child: TabBarView(controller: _tab, children: [
          _mapsTab(),
          const Center(child: Text('Кастомные карты (в разработке)', style: TextStyle(color: Colors.white54))),
          _termTab(),
        ])),
      ]));
  }

  Widget _statusPanel() {
    final conn = widget.obdService.ecuResponds;
    return Container(padding: const EdgeInsets.all(6), color: const Color(0xFF16213E),
      child: Column(children: [
        Row(children: [
          Icon(conn ? Icons.check_circle : Icons.error,
            color: conn ? Colors.green : Colors.red, size: 16),
          const SizedBox(width: 4),
          Expanded(child: Text(conn ? 'ЭБУ подключен' : 'ЭБУ не отвечает',
            style: TextStyle(color: conn ? Colors.green : Colors.red, fontSize: 12))),
          if (_isUnlocked) Container(margin: const EdgeInsets.only(left: 4),
            padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
            decoration: BoxDecoration(color: Colors.green, borderRadius: BorderRadius.circular(4)),
            child: const Text('UNLOCKED', style: TextStyle(
              color: Colors.white, fontSize: 9, fontWeight: FontWeight.bold))),
        ]),
        const SizedBox(height: 6),
        Row(children: [
          Expanded(child: ElevatedButton.icon(
            onPressed: (_isDetecting || _isUnlocking || !conn) ? null : _autoDetect,
            icon: _isDetecting ? const SizedBox(width: 12, height: 12,
              child: CircularProgressIndicator(strokeWidth: 2, color: Colors.white))
              : const Icon(Icons.search, size: 14),
            label: const Text('АВТО', style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(
              backgroundColor: Colors.cyan, foregroundColor: Colors.white,
              padding: const EdgeInsets.symmetric(vertical: 6)))),
          const SizedBox(width: 3),
          Expanded(child: ElevatedButton.icon(
            onPressed: (_isDetecting || _isUnlocking || !conn) ? null : _tryUnlockAuto,
            icon: _isUnlocking ? const SizedBox(width: 12, height: 12,
              child: CircularProgressIndicator(strokeWidth: 2, color: Colors.white))
              : Icon(_isUnlocked ? Icons.lock_open : Icons.autorenew, size: 14),
            label: Text(_isUnlocked ? 'UNLOCKED' : 'АВТО-KEY',
              style: const TextStyle(fontSize: 10, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(
              backgroundColor: _isUnlocked ? Colors.green : Colors.red,
              foregroundColor: Colors.white,
              padding: const EdgeInsets.symmetric(vertical: 6)))),
        ]),
        const SizedBox(height: 4),
        Row(children: [
          Expanded(child: ElevatedButton.icon(
            onPressed: !conn || _isUnlocking ? null : _customFormulaDialog,
            icon: const Icon(Icons.calculate, size: 12),
            label: const Text('ФОРМУЛА', style: TextStyle(fontSize: 9, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(
              backgroundColor: Colors.deepPurple, foregroundColor: Colors.white,
              padding: const EdgeInsets.symmetric(vertical: 5)))),
          const SizedBox(width: 3),
          Expanded(child: ElevatedButton.icon(
            onPressed: !conn || _isUnlocking ? null : _directKeyDialog,
            icon: const Icon(Icons.key, size: 12),
            label: const Text('HEX KEY', style: TextStyle(fontSize: 9, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(
              backgroundColor: Colors.indigo, foregroundColor: Colors.white,
              padding: const EdgeInsets.symmetric(vertical: 5)))),
          const SizedBox(width: 3),
          Expanded(child: ElevatedButton.icon(
            onPressed: _showPairsLog,
            icon: const Icon(Icons.list_alt, size: 12),
            label: Text('ЛОГ (' + _seedKeyPairs.length.toString() + ')',
              style: const TextStyle(fontSize: 9, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(
              backgroundColor: Colors.brown, foregroundColor: Colors.white,
              padding: const EdgeInsets.symmetric(vertical: 5)))),
        ]),
        if (_detectLog.isNotEmpty) Container(
          margin: const EdgeInsets.only(top: 4),
          padding: const EdgeInsets.all(4),
          constraints: const BoxConstraints(maxHeight: 120),
          decoration: BoxDecoration(color: Colors.black, borderRadius: BorderRadius.circular(4)),
          child: SingleChildScrollView(reverse: true,
            child: Text(_detectLog.join('\\n'),
              style: const TextStyle(fontFamily: 'monospace', fontSize: 9, color: Colors.cyan)))),
      ]));
  }

  Widget _mapsTab() {
    return ListView.builder(padding: const EdgeInsets.all(8), itemCount: _standardMaps.length,
      itemBuilder: (c, i) => _mapCard(_standardMaps[i]));
  }

  Widget _mapCard(CustomMapDef def) {
    final progress = _readProgress[def.addressHex];
    final result = _readResults[def.addressHex];
    return Card(color: const Color(0xFF16213E), margin: const EdgeInsets.symmetric(vertical: 3),
      child: Padding(padding: const EdgeInsets.all(10), child: Column(
        crossAxisAlignment: CrossAxisAlignment.start, children: [
          Row(children: [
            Expanded(child: Text(def.name,
              style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 13))),
            Container(padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
              decoration: BoxDecoration(color: const Color(0xFF0F3460),
                borderRadius: BorderRadius.circular(4)),
              child: Text(def.addressHex, style: const TextStyle(
                fontFamily: 'monospace', fontSize: 10, color: Colors.cyan))),
          ]),
          Text('${def.rows}x${def.cols} ' + (def.isUInt16 ? 'UInt16' : 'UInt8'),
            style: const TextStyle(color: Colors.white70, fontSize: 10)),
          const SizedBox(height: 6),
          if (progress != null) Column(children: [
            LinearProgressIndicator(value: progress / 100, color: Colors.cyan),
            Text(progress.toInt().toString() + '%', style: const TextStyle(color: Colors.cyan, fontSize: 11)),
          ])
          else if (result != null) Column(children: [
            Text('Прочитано: ' + DateFormat('HH:mm:ss').format(result.readAt),
              style: const TextStyle(color: Colors.green, fontSize: 11)),
            const SizedBox(height: 6),
            Row(children: [
              Expanded(child: ElevatedButton.icon(onPressed: () => _viewReadMap(result),
                icon: const Icon(Icons.visibility, size: 14), label: const Text('Просмотр'),
                style: ElevatedButton.styleFrom(
                  backgroundColor: Colors.blue, foregroundColor: Colors.white))),
              const SizedBox(width: 4),
              Expanded(child: ElevatedButton.icon(onPressed: () => _saveAsDefault(result),
                icon: const Icon(Icons.save, size: 14), label: const Text('В дефолт'),
                style: ElevatedButton.styleFrom(
                  backgroundColor: Colors.orange, foregroundColor: Colors.white))),
            ]),
          ])
          else SizedBox(width: double.infinity, child: ElevatedButton.icon(
            onPressed: _detectedCommand == null ? null : () => _readMap(def),
            icon: const Icon(Icons.download, size: 16),
            label: Text(_detectedCommand == null ? 'Сначала АВТО' : 'ЧИТАТЬ',
              style: const TextStyle(fontSize: 12, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(backgroundColor: Colors.cyan, foregroundColor: Colors.white))),
        ])));
  }

  Widget _termTab() {
    return Column(children: [
      Flexible(flex: 3, child: Container(color: const Color(0xFF16213E),
        child: SingleChildScrollView(padding: const EdgeInsets.all(6), child: Column(children: [
          const Text('Extended Session:', style: TextStyle(color: Colors.white70, fontSize: 10)),
          Wrap(spacing: 3, runSpacing: 3, children: [
            _qBtn('1081', 'Default'), _qBtn('1085', 'ECU Prog'),
            _qBtn('1090', 'Ext90'), _qBtn('1092', 'Ext92'),
          ]),
          const SizedBox(height: 6),
          const Text('Security:', style: TextStyle(color: Colors.white70, fontSize: 10)),
          Wrap(spacing: 3, runSpacing: 3, children: [
            _qBtn('2701', 'Seed L1'), _qBtn('2703', 'Seed L3'),
          ]),
          const SizedBox(height: 6),
          const Text('Чтение памяти:', style: TextStyle(color: Colors.white70, fontSize: 10)),
          Wrap(spacing: 3, runSpacing: 3, children: [
            _qBtn('237C3C04', 'KWP'), _qBtn('217C3C', 'Consult'),
            _qBtn('D207C3C04', 'D2'),
          ]),
          const SizedBox(height: 6),
          const Text('Полезные:', style: TextStyle(color: Colors.white70, fontSize: 10)),
          Wrap(spacing: 3, runSpacing: 3, children: [
            _qBtn('1A81', 'ECU ID'), _qBtn('1180', 'Reset'),
            _qBtn('3E00', 'TestPres'),
          ]),
        ])))),
      Flexible(flex: 4, child: Container(color: Colors.black, padding: const EdgeInsets.all(8),
        child: SingleChildScrollView(controller: _termScroll,
          child: SelectableText(_termLog.join('\\n'),
            style: const TextStyle(fontFamily: 'monospace', fontSize: 11, color: Colors.green))))),
      Container(color: const Color(0xFF16213E), padding: const EdgeInsets.all(6),
        child: Row(children: [
          Expanded(child: TextField(controller: _termCmd,
            style: const TextStyle(fontFamily: 'monospace'),
            decoration: const InputDecoration(hintText: 'Команда...',
              border: OutlineInputBorder(),
              contentPadding: EdgeInsets.symmetric(horizontal: 8, vertical: 6), isDense: true),
            textCapitalization: TextCapitalization.characters, onSubmitted: (_) => _sendTermCmd())),
          IconButton(icon: const Icon(Icons.send, color: Colors.green), onPressed: _sendTermCmd),
          IconButton(icon: const Icon(Icons.clear_all, color: Colors.white54),
            onPressed: () => setState(() => _termLog.clear())),
        ])),
    ]);
  }

  Widget _qBtn(String cmd, String label) {
    return ElevatedButton(onPressed: () => _quick(cmd),
      style: ElevatedButton.styleFrom(
        backgroundColor: const Color(0xFF0F3460), foregroundColor: Colors.white,
        padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 3)),
      child: Column(mainAxisSize: MainAxisSize.min, children: [
        Text(cmd, style: const TextStyle(fontFamily: 'monospace', fontSize: 10)),
        Text(label, style: const TextStyle(fontSize: 8, color: Colors.white70)),
      ]));
  }
}
''')
print("✅ ecu_read_screen.dart v2 обновлён")



✅ ecu_read_screen.dart v2 обновлён


In [7]:
# @title 🔧 FULL: LogGraph + Analyzer со всем функционалом v4
import os
os.chdir('/content/nissan_logger_pro_v5')

# ============ log_graph_screen.dart (ПОЛНЫЙ — тач-курсор + статистика + zoom + фильтры) ============
with open('lib/screens/log_graph_screen.dart', 'w') as f:
    f.write('''import 'dart:math';
import 'package:flutter/material.dart';
import 'package:fl_chart/fl_chart.dart';
import 'package:file_picker/file_picker.dart';
import '../models/obd_data.dart';
import '../services/analyzer_service.dart';
import '../services/settings_service.dart';

class LogGraphScreen extends StatefulWidget {
  const LogGraphScreen({super.key});
  @override
  State<LogGraphScreen> createState() => _LogGraphScreenState();
}

class _ParamInfo {
  final String key;
  final String label;
  final Color color;
  final String unit;
  final double Function(OBDData) getValue;
  final int digits;
  _ParamInfo({required this.key, required this.label, required this.color,
    required this.unit, required this.getValue, this.digits = 1});
}

class _LogGraphScreenState extends State<LogGraphScreen> {
  final AnalyzerService _analyzer = AnalyzerService();
  List<OBDData>? _log;
  String? _fileName;
  bool _isLoading = false;
  Set<String> _selected = {};
  double _rStart = 0.0;
  double _rEnd = 1.0;
  int? _touchIdx;

  static final List<_ParamInfo> _params = [
    _ParamInfo(key: 'RPM', label: 'RPM', color: Colors.blue, unit: 'об/мин',
      getValue: (d) => d.rpm.toDouble(), digits: 0),
    _ParamInfo(key: 'Speed', label: 'Скор.', color: Colors.cyan, unit: 'км/ч',
      getValue: (d) => d.speed.toDouble(), digits: 0),
    _ParamInfo(key: 'ECT', label: 'ОЖ', color: Colors.red, unit: '°C',
      getValue: (d) => d.coolantTemp.toDouble(), digits: 0),
    _ParamInfo(key: 'IAT', label: 'Впуск', color: Colors.orange, unit: '°C',
      getValue: (d) => d.intakeTemp.toDouble(), digits: 0),
    _ParamInfo(key: 'MAF', label: 'MAF', color: Colors.purple, unit: 'g/s',
      getValue: (d) => d.maf, digits: 2),
    _ParamInfo(key: 'TPS', label: 'Дроссель', color: Colors.green, unit: '%',
      getValue: (d) => d.throttlePos, digits: 1),
    _ParamInfo(key: 'Timing', label: 'УОЗ', color: Colors.lightGreen, unit: '°',
      getValue: (d) => d.actualIgnition, digits: 1),
    _ParamInfo(key: 'Knock', label: 'Knock', color: Colors.deepOrange, unit: '°',
      getValue: (d) => d.knockRetard, digits: 1),
    _ParamInfo(key: 'VTC', label: 'VTC', color: Colors.pink, unit: '°',
      getValue: (d) => d.vtcActualAngle, digits: 1),
    _ParamInfo(key: 'Load', label: 'Нагрузка', color: Colors.amber, unit: '%',
      getValue: (d) => d.engineLoad, digits: 1),
    _ParamInfo(key: 'STFT', label: 'STFT', color: Colors.lime, unit: '%',
      getValue: (d) => d.shortFuelTrim, digits: 1),
    _ParamInfo(key: 'LTFT', label: 'LTFT', color: Colors.teal, unit: '%',
      getValue: (d) => d.longFuelTrim, digits: 1),
    _ParamInfo(key: 'AFR', label: 'AFR', color: Colors.yellow, unit: '',
      getValue: (d) => d.afr, digits: 2),
    _ParamInfo(key: 'O2', label: 'O2', color: Colors.indigo, unit: 'V',
      getValue: (d) => d.o2Voltage, digits: 3),
    _ParamInfo(key: 'Injector', label: 'Впрыск', color: Colors.deepPurple, unit: 'ms',
      getValue: (d) => d.injectorPulseWidth, digits: 2),
    _ParamInfo(key: 'HP', label: 'Мощность', color: Colors.yellowAccent, unit: 'л.с.',
      getValue: (d) => d.calculatedHP, digits: 1),
    _ParamInfo(key: 'Batt', label: 'Батарея', color: Colors.lightBlueAccent, unit: 'V',
      getValue: (d) => d.batteryVoltage, digits: 2),
  ];

  _ParamInfo _paramByKey(String k) => _params.firstWhere((p) => p.key == k);

  @override
  void initState() {
    super.initState();
    _selected = SettingsService.selectedLogParams.toSet();
    if (_selected.isEmpty) _selected = {'RPM'};
  }

  Future<void> _loadLog() async {
    try {
      final r = await FilePicker.platform.pickFiles(
        type: FileType.custom, allowedExtensions: ['csv']);
      if (r == null) return;
      setState(() => _isLoading = true);
      _log = await _analyzer.loadLogFromCSV(r.files.single.path!);
      _fileName = r.files.single.name;
      _rStart = 0.0; _rEnd = 1.0; _touchIdx = null;
      setState(() => _isLoading = false);
      if (mounted) ScaffoldMessenger.of(context).showSnackBar(SnackBar(
        content: Text('Загружено: ' + _log!.length.toString(),
          style: const TextStyle(color: Colors.white)),
        backgroundColor: Colors.green));
    } catch (e) { setState(() => _isLoading = false); }
  }

  Future<void> _saveSelected() async {
    await SettingsService.setSelectedLogParams(_selected.toList());
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Графики лога'),
        backgroundColor: const Color(0xFF16213E),
        actions: [
          if (_log != null) IconButton(icon: const Icon(Icons.zoom_out_map),
            onPressed: () => setState(() { _rStart = 0.0; _rEnd = 1.0; _touchIdx = null; }),
            tooltip: 'Сбросить zoom'),
        ]),
      body: Column(children: [
        Card(color: const Color(0xFF16213E), margin: const EdgeInsets.all(8),
          child: Padding(padding: const EdgeInsets.all(10), child: Column(children: [
            ElevatedButton.icon(
              onPressed: _isLoading ? null : _loadLog,
              icon: const Icon(Icons.folder_open),
              label: const Text('Загрузить CSV'),
              style: ElevatedButton.styleFrom(
                minimumSize: const Size.fromHeight(45),
                foregroundColor: Colors.white)),
            if (_fileName != null) ...[
              const SizedBox(height: 6),
              Text(_fileName!, style: const TextStyle(color: Colors.white70, fontSize: 11)),
              Text('Записей: ' + (_log?.length ?? 0).toString(),
                style: const TextStyle(color: Colors.green, fontWeight: FontWeight.bold)),
            ],
          ]))),
        if (_log != null && _log!.isNotEmpty)
          Card(color: const Color(0xFF16213E),
            margin: const EdgeInsets.symmetric(horizontal: 8),
            child: Padding(padding: const EdgeInsets.all(8),
              child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                Row(mainAxisAlignment: MainAxisAlignment.spaceBetween, children: [
                  Text('ПАРАМЕТРЫ (' + _selected.length.toString() + '):',
                    style: const TextStyle(color: Colors.white70, fontSize: 11, fontWeight: FontWeight.bold)),
                  TextButton(onPressed: () async {
                    setState(() { _selected.clear(); _selected.add('RPM'); });
                    await _saveSelected();
                  },
                    style: TextButton.styleFrom(foregroundColor: Colors.white),
                    child: const Text('Сброс', style: TextStyle(fontSize: 11))),
                ]),
                Wrap(spacing: 4, runSpacing: 4, children: _params.map((p) {
                  final sel = _selected.contains(p.key);
                  return FilterChip(
                    label: Text(p.label, style: TextStyle(fontSize: 11,
                      color: sel ? Colors.white : Colors.white70,
                      fontWeight: sel ? FontWeight.bold : FontWeight.normal)),
                    selected: sel,
                    onSelected: (s) async {
                      setState(() {
                        if (s) _selected.add(p.key);
                        else if (_selected.length > 1) _selected.remove(p.key);
                      });
                      await _saveSelected();
                    },
                    selectedColor: p.color.withOpacity(0.5),
                    backgroundColor: const Color(0xFF0F3460),
                    side: BorderSide(color: sel ? p.color : Colors.white24),
                    materialTapTargetSize: MaterialTapTargetSize.shrinkWrap);
                }).toList()),
              ]))),
        const SizedBox(height: 6),
        Expanded(child: _log == null || _log!.isEmpty
          ? const Center(child: Padding(padding: EdgeInsets.all(30),
              child: Text('Загрузите CSV для просмотра графиков',
                style: TextStyle(color: Colors.white54, fontSize: 14),
                textAlign: TextAlign.center)))
          : SingleChildScrollView(padding: const EdgeInsets.all(8),
              child: Column(children: [
                if (_touchIdx != null) _buildTouchInfo(),
                _buildStats(),
                const SizedBox(height: 8),
                Card(color: const Color(0xFF16213E),
                  child: Padding(padding: const EdgeInsets.all(10), child: Column(
                    crossAxisAlignment: CrossAxisAlignment.start, children: [
                      const Text('СОВМЕЩЁННЫЙ ГРАФИК (0-100%)',
                        style: TextStyle(color: Colors.white70, fontSize: 11, fontWeight: FontWeight.bold)),
                      const Text('Тап по графику - курсор с реальными значениями',
                        style: TextStyle(color: Colors.cyan, fontSize: 9)),
                      const SizedBox(height: 8),
                      SizedBox(height: 300, child: _buildCombined()),
                    ]))),
                const SizedBox(height: 8),
                _buildLegend(),
                const SizedBox(height: 8),
                ..._selected.map((k) {
                  final p = _paramByKey(k);
                  return Padding(padding: const EdgeInsets.only(bottom: 8),
                    child: Card(color: const Color(0xFF16213E),
                      child: Padding(padding: const EdgeInsets.all(10), child: Column(
                        crossAxisAlignment: CrossAxisAlignment.start, children: [
                          Row(children: [
                            Container(width: 12, height: 12,
                              decoration: BoxDecoration(color: p.color,
                                borderRadius: BorderRadius.circular(2))),
                            const SizedBox(width: 6),
                            Text(p.label + ' (' + p.unit + ')',
                              style: TextStyle(color: p.color, fontSize: 13, fontWeight: FontWeight.bold)),
                          ]),
                          const SizedBox(height: 6),
                          SizedBox(height: 200, child: _buildSingle(p)),
                        ]))));
                }).toList(),
                _buildSlider(),
                const SizedBox(height: 20),
              ]))),
      ]));
  }

  Widget _buildTouchInfo() {
    if (_touchIdx == null || _log == null || _touchIdx! >= _log!.length) return const SizedBox();
    final d = _log![_touchIdx!];
    return Card(color: Colors.cyan.withOpacity(0.2),
      child: Padding(padding: const EdgeInsets.all(8), child: Column(
        crossAxisAlignment: CrossAxisAlignment.start, children: [
          Row(children: [
            const Icon(Icons.touch_app, color: Colors.cyan, size: 16),
            const SizedBox(width: 4),
            Text('Точка #' + _touchIdx!.toString(),
              style: const TextStyle(color: Colors.cyan, fontWeight: FontWeight.bold)),
            const Spacer(),
            IconButton(icon: const Icon(Icons.close, size: 16),
              onPressed: () => setState(() => _touchIdx = null),
              padding: EdgeInsets.zero, constraints: const BoxConstraints()),
          ]),
          const SizedBox(height: 4),
          Wrap(spacing: 10, runSpacing: 4, children: _selected.map((k) {
            final p = _paramByKey(k);
            return Row(mainAxisSize: MainAxisSize.min, children: [
              Container(width: 8, height: 8, color: p.color),
              const SizedBox(width: 3),
              Text(p.label + ': ' + p.getValue(d).toStringAsFixed(p.digits) + ' ' + p.unit,
                style: TextStyle(color: p.color, fontSize: 11, fontWeight: FontWeight.bold)),
            ]);
          }).toList()),
        ])));
  }

  Widget _buildStats() {
    if (_log == null || _log!.isEmpty) return const SizedBox();
    return Card(color: const Color(0xFF16213E),
      child: Padding(padding: const EdgeInsets.all(8), child: Column(
        crossAxisAlignment: CrossAxisAlignment.start, children: [
          const Text('СТАТИСТИКА:',
            style: TextStyle(color: Colors.white70, fontSize: 11, fontWeight: FontWeight.bold)),
          const SizedBox(height: 6),
          Table(border: TableBorder.all(color: Colors.white12, width: 0.5),
            columnWidths: const {0: FlexColumnWidth(2), 1: FlexColumnWidth(1.5),
              2: FlexColumnWidth(1.5), 3: FlexColumnWidth(1.5)},
            children: [
              const TableRow(decoration: BoxDecoration(color: Color(0xFF0F3460)),
                children: [
                  Padding(padding: EdgeInsets.all(4), child: Text('Параметр',
                    style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold))),
                  Padding(padding: EdgeInsets.all(4), child: Text('Мин',
                    style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold), textAlign: TextAlign.center)),
                  Padding(padding: EdgeInsets.all(4), child: Text('Сред',
                    style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold), textAlign: TextAlign.center)),
                  Padding(padding: EdgeInsets.all(4), child: Text('Макс',
                    style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold), textAlign: TextAlign.center)),
                ]),
              ..._selected.map((k) {
                final p = _paramByKey(k);
                final vals = _log!.map((d) => p.getValue(d)).toList();
                double minV = vals.reduce(min);
                double maxV = vals.reduce(max);
                double avgV = vals.reduce((a, b) => a + b) / vals.length;
                return TableRow(children: [
                  Padding(padding: const EdgeInsets.all(4),
                    child: Row(children: [
                      Container(width: 8, height: 8, color: p.color),
                      const SizedBox(width: 4),
                      Expanded(child: Text(p.label, style: const TextStyle(fontSize: 11))),
                    ])),
                  Padding(padding: const EdgeInsets.all(4),
                    child: Text(minV.toStringAsFixed(p.digits),
                      style: const TextStyle(fontSize: 11, fontFamily: 'monospace'),
                      textAlign: TextAlign.center)),
                  Padding(padding: const EdgeInsets.all(4),
                    child: Text(avgV.toStringAsFixed(p.digits),
                      style: const TextStyle(fontSize: 11, fontFamily: 'monospace'),
                      textAlign: TextAlign.center)),
                  Padding(padding: const EdgeInsets.all(4),
                    child: Text(maxV.toStringAsFixed(p.digits),
                      style: const TextStyle(fontSize: 11, fontFamily: 'monospace'),
                      textAlign: TextAlign.center)),
                ]);
              }).toList(),
            ]),
        ])));
  }

  Widget _buildLegend() {
    return Card(color: const Color(0xFF16213E), child: Padding(
      padding: const EdgeInsets.all(8),
      child: Wrap(spacing: 12, runSpacing: 6, children: _selected.map((k) {
        final p = _paramByKey(k);
        return Row(mainAxisSize: MainAxisSize.min, children: [
          Container(width: 14, height: 3, color: p.color),
          const SizedBox(width: 4),
          Text(p.label, style: TextStyle(color: p.color, fontSize: 11)),
        ]);
      }).toList())));
  }

  Widget _buildSlider() {
    if (_log == null || _log!.length < 10) return const SizedBox();
    return Card(color: const Color(0xFF16213E),
      child: Padding(padding: const EdgeInsets.all(8), child: Column(children: [
        Text('ZOOM: ' + (_rStart * _log!.length).toInt().toString() +
          ' — ' + (_rEnd * _log!.length).toInt().toString() +
          ' / ' + _log!.length.toString(),
          style: const TextStyle(color: Colors.white70, fontSize: 11)),
        RangeSlider(values: RangeValues(_rStart, _rEnd),
          min: 0.0, max: 1.0, divisions: 100,
          activeColor: const Color(0xFFE94560),
          inactiveColor: Colors.white24,
          onChanged: (v) => setState(() {
            if (v.end - v.start >= 0.02) { _rStart = v.start; _rEnd = v.end; _touchIdx = null; }
          })),
      ])));
  }

  Widget _buildCombined() {
    if (_log == null || _log!.isEmpty) return const SizedBox();
    final total = _log!.length;
    final si = (_rStart * total).floor();
    final ei = (_rEnd * total).ceil().clamp(si + 1, total);
    final range = _log!.sublist(si, ei);
    int step = (range.length / 300).ceil();
    if (step < 1) step = 1;

    List<LineChartBarData> lines = [];
    for (var k in _selected) {
      final p = _paramByKey(k);
      final vals = range.map((d) => p.getValue(d)).toList();
      double minV = vals.reduce(min);
      double maxV = vals.reduce(max);
      double r = maxV - minV;
      if (r < 0.001) r = 1;
      List<FlSpot> spots = [];
      for (int i = 0; i < range.length; i += step) {
        double val = p.getValue(range[i]);
        spots.add(FlSpot(i.toDouble(), (val - minV) / r * 100));
      }
      lines.add(LineChartBarData(spots: spots, isCurved: false, color: p.color,
        barWidth: 1.5, dotData: const FlDotData(show: false),
        belowBarData: BarAreaData(show: false)));
    }

    return LineChart(LineChartData(
      gridData: FlGridData(show: true, drawVerticalLine: false, horizontalInterval: 25,
        getDrawingHorizontalLine: (v) => FlLine(color: Colors.white.withOpacity(0.1), strokeWidth: 1)),
      titlesData: FlTitlesData(
        rightTitles: const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        topTitles: const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        bottomTitles: const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        leftTitles: AxisTitles(sideTitles: SideTitles(showTitles: true, reservedSize: 32,
          interval: 25, getTitlesWidget: (v, m) => Text(v.toInt().toString() + '%',
            style: const TextStyle(color: Colors.white54, fontSize: 9))))),
      borderData: FlBorderData(show: true, border: Border.all(color: Colors.white.withOpacity(0.1))),
      minX: 0, maxX: range.length.toDouble(), minY: 0, maxY: 100,
      lineBarsData: lines,
      lineTouchData: LineTouchData(enabled: true,
        touchCallback: (event, response) {
          if (response?.lineBarSpots != null && response!.lineBarSpots!.isNotEmpty) {
            final idx = response.lineBarSpots!.first.x.toInt();
            if (idx >= 0 && idx < range.length) {
              setState(() => _touchIdx = si + idx);
            }
          }
        },
        touchTooltipData: LineTouchTooltipData(
          getTooltipColor: (_) => Colors.black87,
          getTooltipItems: (spots) => spots.map((spot) {
            final k = _selected.elementAt(spot.barIndex);
            final p = _paramByKey(k);
            final idx = spot.x.toInt().clamp(0, range.length - 1);
            return LineTooltipItem(
              p.label + ': ' + p.getValue(range[idx]).toStringAsFixed(p.digits) + ' ' + p.unit,
              TextStyle(color: p.color, fontSize: 10, fontWeight: FontWeight.bold));
          }).toList())),
    ));
  }

  Widget _buildSingle(_ParamInfo p) {
    if (_log == null || _log!.isEmpty) return const SizedBox();
    final total = _log!.length;
    final si = (_rStart * total).floor();
    final ei = (_rEnd * total).ceil().clamp(si + 1, total);
    final range = _log!.sublist(si, ei);
    int step = (range.length / 300).ceil();
    if (step < 1) step = 1;
    List<FlSpot> spots = [];
    double minY = double.infinity;
    double maxY = -double.infinity;
    for (int i = 0; i < range.length; i += step) {
      double val = p.getValue(range[i]);
      spots.add(FlSpot(i.toDouble(), val));
      if (val < minY) minY = val;
      if (val > maxY) maxY = val;
    }
    if (spots.isEmpty) return const SizedBox();
    if (minY == maxY) { minY -= 1; maxY += 1; }
    double margin = (maxY - minY) * 0.05;
    minY -= margin;
    maxY += margin;
    return LineChart(LineChartData(
      gridData: FlGridData(show: true, drawVerticalLine: false,
        horizontalInterval: (maxY - minY) / 4,
        getDrawingHorizontalLine: (v) => FlLine(color: Colors.white.withOpacity(0.1), strokeWidth: 1)),
      titlesData: FlTitlesData(
        rightTitles: const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        topTitles: const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        bottomTitles: const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        leftTitles: AxisTitles(sideTitles: SideTitles(showTitles: true, reservedSize: 48,
          interval: (maxY - minY) / 4,
          getTitlesWidget: (v, m) => Text(v.toStringAsFixed(p.digits),
            style: const TextStyle(color: Colors.white54, fontSize: 9))))),
      borderData: FlBorderData(show: true, border: Border.all(color: Colors.white.withOpacity(0.1))),
      minX: 0, maxX: range.length.toDouble(), minY: minY, maxY: maxY,
      lineBarsData: [LineChartBarData(spots: spots, isCurved: false, color: p.color,
        barWidth: 1.5, dotData: const FlDotData(show: false),
        belowBarData: BarAreaData(show: true, color: p.color.withOpacity(0.15)))],
      lineTouchData: LineTouchData(enabled: true,
        touchTooltipData: LineTouchTooltipData(
          getTooltipColor: (_) => Colors.black87,
          getTooltipItems: (spots) => spots.map((spot) => LineTooltipItem(
            p.label + ': ' + spot.y.toStringAsFixed(p.digits) + ' ' + p.unit,
            TextStyle(color: p.color, fontSize: 11, fontWeight: FontWeight.bold))).toList())),
    ));
  }
}
''')
print("✅ log_graph_screen.dart — ПОЛНАЯ версия (тач-курсор, статистика, zoom, фильтры, совмещённый + отдельные графики)")

# ============ analyzer_screen.dart (ПОЛНЫЙ — Онлайн + Из лога + карты в 3D + экспорт) ============
with open('lib/screens/analyzer_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'package:flutter/material.dart';
import 'package:file_picker/file_picker.dart';
import '../models/obd_data.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';
import '../services/analyzer_service.dart';
import '../services/tuning_service.dart';
import '../services/export_service.dart';
import '../services/obd_service.dart';
import '../widgets/fps_indicator.dart';
import '../widgets/map_table_view.dart';

class AnalyzerScreen extends StatefulWidget {
  final OBDService? obdService;
  const AnalyzerScreen({super.key, this.obdService});
  @override
  State<AnalyzerScreen> createState() => _AnalyzerScreenState();
}

class _AnalyzerScreenState extends State<AnalyzerScreen>
    with SingleTickerProviderStateMixin {
  final AnalyzerService _analyzer = AnalyzerService();
  final TuningService _tuning = TuningService();
  final ExportService _export = ExportService();
  late TabController _tab;

  // Общее состояние
  String _map = 'Spark Advance';
  TuningMap? _orig;
  TuningMap? _upd;

  // Из лога
  List<OBDData>? _log;
  String? _logPath;
  AnalysisResult? _result;
  bool _isAnalyzing = false;

  // Онлайн
  bool _isRecording = false;
  final List<OBDData> _buf = [];
  AnalysisResult? _onlineResult;
  StreamSubscription? _dataSub;
  Timer? _autoTimer;
  int _records = 0;

  @override
  void initState() {
    super.initState();
    _tab = TabController(length: 2, vsync: this);
  }

  @override
  void dispose() {
    _tab.dispose();
    _dataSub?.cancel();
    _autoTimer?.cancel();
    super.dispose();
  }

  void _startRec() {
    if (widget.obdService == null || !widget.obdService!.isConnected) {
      _snack('Нет подключения', Colors.red); return;
    }
    setState(() {
      _isRecording = true;
      _buf.clear();
      _records = 0;
      _onlineResult = null;
    });
    _dataSub = widget.obdService!.dataStream.listen((data) {
      if (_isRecording) {
        _buf.add(data);
        _records = _buf.length;
        if (_buf.length > 5000) _buf.removeRange(0, 1000);
      }
    });
    _autoTimer = Timer.periodic(const Duration(seconds: 5), (_) async {
      if (_buf.length >= 20 && mounted) await _runOnline();
      if (mounted) setState(() {});
    });
    _snack('Запись начата', Colors.green);
  }

  void _stopRec() {
    setState(() => _isRecording = false);
    _dataSub?.cancel();
    _autoTimer?.cancel();
    _snack('Запись остановлена', Colors.orange);
  }

  Future<void> _runOnline() async {
    if (_buf.length < 20) return;
    try {
      final r = await _analyzeMap(_buf);
      if (mounted && r != null) {
        setState(() {
          _onlineResult = r.$1;
          _orig = r.$2;
          _upd = r.$3;
        });
      }
    } catch (e) {}
  }

  Future<(AnalysisResult, TuningMap, TuningMap)?> _analyzeMap(List<OBDData> data) async {
    TuningMap m;
    AnalysisResult r;
    switch (_map) {
      case 'Spark Advance':
        m = await _tuning.getSparkAdvanceMap();
        r = await _analyzer.analyzeSparkMap(data, m);
        break;
      case 'Fuel Map / VE':
        m = await _tuning.getFuelMap();
        r = await _analyzer.analyzeFuelMap(data, m);
        break;
      case 'VTC / Intake Cam':
        m = await _tuning.getVTCMap();
        r = await _analyzer.analyzeVTCMap(data, m);
        break;
      default:
        m = await _tuning.getEngineTorqueMap();
        r = await _analyzer.analyzeTorqueMap(data, m);
    }
    TuningMap u = m.copy();
    for (var c in r.changes) {
      u.data[c.rpmIndex][c.loadIndex] = c.suggestedValue;
    }
    return (r, m, u);
  }

  Future<void> _load() async {
    try {
      final r = await FilePicker.platform.pickFiles(
        type: FileType.custom, allowedExtensions: ['csv']);
      if (r != null) {
        setState(() => _isAnalyzing = true);
        _log = await _analyzer.loadLogFromCSV(r.files.single.path!);
        _logPath = r.files.single.name;
        setState(() => _isAnalyzing = false);
        _snack('Загружено: ' + _log!.length.toString(), Colors.green);
      }
    } catch (e) { setState(() => _isAnalyzing = false); }
  }

  Future<void> _analyzeLog() async {
    if (_log == null || _log!.isEmpty) {
      _snack('Загрузите лог', Colors.orange); return;
    }
    setState(() => _isAnalyzing = true);
    try {
      final r = await _analyzeMap(_log!);
      if (r != null) {
        setState(() {
          _result = r.$1;
          _orig = r.$2;
          _upd = r.$3;
          _isAnalyzing = false;
        });
      }
    } catch (e) { setState(() => _isAnalyzing = false); }
  }

  Future<void> _exp(String fmt, AnalysisResult r) async {
    if (_upd == null || _orig == null) return;
    try {
      String path;
      switch (fmt) {
        case 'winols': path = await _export.exportToWinOLS(_upd!); break;
        case 'ecuedit': path = await _export.exportToEcuEdit(_upd!); break;
        case 'json': path = await _export.exportToJson(r, _orig!, _upd!); break;
        case 'hex': path = await _export.exportHexPatch(_upd!); break;
        default: return;
      }
      _snack('Сохранено: ' + path.split('/').last, Colors.green);
    } catch (e) {}
  }

  void _openFullscreen() {
    if (_orig == null) return;
    Navigator.push(context, MaterialPageRoute(
      builder: (c) => MapFullscreenView(
        originalMap: _orig!,
        updatedMap: _upd,
        changes: (_result?.changes ?? _onlineResult?.changes) ?? [],
        onSaved: () async { if (_log != null) await _analyzeLog(); })));
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(
      content: Text(m, style: const TextStyle(color: Colors.white)),
      backgroundColor: c));
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Анализатор'),
        backgroundColor: const Color(0xFF16213E),
        actions: [if (widget.obdService != null) FpsIndicator(obdService: widget.obdService!)],
        bottom: TabBar(controller: _tab, tabs: const [
          Tab(icon: Icon(Icons.wifi), text: 'Онлайн'),
          Tab(icon: Icon(Icons.folder_open), text: 'Из лога'),
        ])),
      body: TabBarView(controller: _tab, children: [_online(), _fromLog()]));
  }

  Widget _mapSel() {
    return Card(color: const Color(0xFF16213E),
      child: Padding(padding: const EdgeInsets.all(8), child: Row(children: [
        const Text('Карта:', style: TextStyle(color: Colors.white70, fontSize: 12)),
        const SizedBox(width: 8),
        Expanded(child: DropdownButtonFormField<String>(
          value: _map, dropdownColor: const Color(0xFF16213E), isDense: true,
          items: const [
            DropdownMenuItem(value: 'Spark Advance', child: Text('Зажигание')),
            DropdownMenuItem(value: 'Fuel Map / VE', child: Text('Топливо/VE')),
            DropdownMenuItem(value: 'VTC / Intake Cam', child: Text('VTC')),
            DropdownMenuItem(value: 'Engine Torque', child: Text('Момент')),
          ],
          onChanged: (v) => setState(() => _map = v!))),
      ])));
  }

  Widget _online() {
    final conn = widget.obdService?.isConnected ?? false;
    final ecu = widget.obdService?.ecuResponds ?? false;
    return Padding(padding: const EdgeInsets.all(6), child: Column(children: [
      Card(color: _isRecording ? Colors.green.withOpacity(0.2) : const Color(0xFF16213E),
        child: Padding(padding: const EdgeInsets.all(8), child: Column(children: [
          Row(children: [
            Icon(conn && ecu ? Icons.check_circle : Icons.error,
              color: conn && ecu ? Colors.green : Colors.red, size: 18),
            const SizedBox(width: 6),
            Expanded(child: Text(conn && ecu ? 'ЭБУ готов' : 'Нет подключения',
              style: const TextStyle(fontSize: 12))),
            if (_isRecording) Container(
              padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
              decoration: BoxDecoration(color: Colors.red, borderRadius: BorderRadius.circular(4)),
              child: const Text('REC', style: TextStyle(color: Colors.white,
                fontWeight: FontWeight.bold, fontSize: 10))),
          ]),
          const SizedBox(height: 4),
          Row(children: [
            _stat('Записей', _records.toString(), Colors.blue),
            const SizedBox(width: 4),
            _stat('FPS', widget.obdService?.pollFps.toString() ?? '0', Colors.green),
            const SizedBox(width: 4),
            _stat('Правок', _onlineResult?.changes.length.toString() ?? '0', Colors.orange),
          ]),
        ]))),
      const SizedBox(height: 4),
      _mapSel(),
      const SizedBox(height: 4),
      Row(children: [
        Expanded(child: ElevatedButton.icon(
          onPressed: _isRecording ? _stopRec : (conn && ecu ? _startRec : null),
          icon: Icon(_isRecording ? Icons.stop : Icons.play_arrow, size: 18),
          label: Text(_isRecording ? 'СТОП' : 'ЗАПИСЬ', style: const TextStyle(fontSize: 12)),
          style: ElevatedButton.styleFrom(
            backgroundColor: _isRecording ? Colors.red : Colors.green,
            foregroundColor: Colors.white,
            minimumSize: const Size.fromHeight(40)))),
        const SizedBox(width: 4),
        Expanded(child: ElevatedButton.icon(
          onPressed: _buf.length >= 20 ? _runOnline : null,
          icon: const Icon(Icons.refresh, size: 18),
          label: const Text('АНАЛИЗ', style: TextStyle(fontSize: 12)),
          style: ElevatedButton.styleFrom(
            foregroundColor: Colors.white,
            minimumSize: const Size.fromHeight(40)))),
      ]),
      const SizedBox(height: 4),
      if (_onlineResult != null) Expanded(child: _results(_onlineResult!))
      else const Expanded(child: Center(child: Padding(padding: EdgeInsets.all(20),
        child: Text('Нажми ЗАПИСЬ → покатайся 2-5 мин →\nправки появятся автоматически',
          style: TextStyle(color: Colors.white54, fontSize: 12), textAlign: TextAlign.center)))),
    ]));
  }

  Widget _stat(String l, String v, Color c) {
    return Expanded(child: Container(
      padding: const EdgeInsets.all(3),
      decoration: BoxDecoration(color: const Color(0xFF0F3460),
        borderRadius: BorderRadius.circular(4)),
      child: Column(children: [
        Text(l, style: const TextStyle(color: Colors.white70, fontSize: 9)),
        Text(v, style: TextStyle(color: c, fontSize: 13, fontWeight: FontWeight.bold)),
      ])));
  }

  Widget _fromLog() {
    return Column(children: [
      Padding(padding: const EdgeInsets.all(6), child: Column(children: [
        Card(color: const Color(0xFF16213E), child: Padding(
          padding: const EdgeInsets.all(8), child: Column(children: [
            ElevatedButton.icon(
              onPressed: _isAnalyzing ? null : _load,
              icon: const Icon(Icons.folder_open, size: 18),
              label: const Text('Загрузить CSV'),
              style: ElevatedButton.styleFrom(
                foregroundColor: Colors.white,
                minimumSize: const Size.fromHeight(38))),
            if (_logPath != null) ...[
              const SizedBox(height: 4),
              Text(_logPath!, style: const TextStyle(color: Colors.white70, fontSize: 11)),
              Text('Записей: ' + (_log?.length ?? 0).toString(),
                style: const TextStyle(color: Colors.green, fontWeight: FontWeight.bold)),
            ],
          ]))),
        const SizedBox(height: 4),
        _mapSel(),
        const SizedBox(height: 4),
        ElevatedButton.icon(
          onPressed: _isAnalyzing || _log == null ? null : _analyzeLog,
          icon: const Icon(Icons.analytics, size: 18),
          label: const Text('АНАЛИЗИРОВАТЬ'),
          style: ElevatedButton.styleFrom(
            backgroundColor: Colors.green, foregroundColor: Colors.white,
            minimumSize: const Size.fromHeight(40))),
      ])),
      if (_result != null) Expanded(child: _results(_result!)),
    ]);
  }

  Widget _results(AnalysisResult r) {
    return Card(color: const Color(0xFF16213E),
      margin: const EdgeInsets.symmetric(horizontal: 6),
      child: Padding(padding: const EdgeInsets.all(6), child: Column(
        crossAxisAlignment: CrossAxisAlignment.start, children: [
          Row(children: [
            const Icon(Icons.assessment, color: Colors.green, size: 18),
            const SizedBox(width: 4),
            Expanded(child: Text(r.mapName,
              style: const TextStyle(fontSize: 13, fontWeight: FontWeight.bold))),
            Text(r.changes.length.toString() + ' правок',
              style: const TextStyle(color: Colors.orange, fontSize: 12)),
          ]),
          Text(r.summary, style: const TextStyle(color: Colors.white70, fontSize: 10)),
          const SizedBox(height: 4),
          if (_orig != null) SizedBox(width: double.infinity,
            child: ElevatedButton.icon(
              onPressed: _openFullscreen,
              icon: const Icon(Icons.grid_on, size: 20),
              label: const Text('ОТКРЫТЬ КАРТУ',
                style: TextStyle(fontSize: 13, fontWeight: FontWeight.bold)),
              style: ElevatedButton.styleFrom(
                backgroundColor: Colors.cyan.shade700, foregroundColor: Colors.white,
                minimumSize: const Size.fromHeight(42)))),
          const SizedBox(height: 6),
          Expanded(child: r.changes.isEmpty
            ? const Center(child: Text('Правок не требуется',
                style: TextStyle(color: Colors.green, fontSize: 14)))
            : ListView.builder(itemCount: r.changes.length,
                itemBuilder: (c, i) => _change(r.changes[i]))),
          const SizedBox(height: 4),
          Wrap(spacing: 4, runSpacing: 4, children: [
            _expBtn('WinOLS', 'winols', Colors.blue, r),
            _expBtn('ecuEdit', 'ecuedit', Colors.purple, r),
            _expBtn('JSON', 'json', Colors.green, r),
            _expBtn('HEX', 'hex', Colors.orange, r),
          ]),
        ])));
  }

  Widget _expBtn(String l, String f, Color c, AnalysisResult r) {
    return ElevatedButton.icon(
      onPressed: () => _exp(f, r),
      icon: const Icon(Icons.download, size: 12),
      label: Text(l, style: const TextStyle(fontSize: 10)),
      style: ElevatedButton.styleFrom(
        backgroundColor: c, foregroundColor: Colors.white,
        padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 4)));
  }

  Widget _change(MapCell c) {
    Color dc = c.delta > 0 ? Colors.green : Colors.orange;
    return Card(color: const Color(0xFF0F3460),
      margin: const EdgeInsets.symmetric(vertical: 2),
      child: ListTile(dense: true,
        title: Text('RPM ' + c.rpm.toInt().toString() + ' | Load ' + c.load.toInt().toString() + '%',
          style: const TextStyle(fontSize: 11, fontWeight: FontWeight.bold)),
        subtitle: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
          Row(children: [
            Text(c.currentValue.toStringAsFixed(1) + ' → ' + c.suggestedValue.toStringAsFixed(1),
              style: TextStyle(color: dc, fontWeight: FontWeight.bold, fontSize: 11)),
            Text(' (' + (c.delta > 0 ? '+' : '') + c.delta.toStringAsFixed(1) + ')',
              style: TextStyle(color: dc, fontSize: 9)),
          ]),
          Text(c.reason, style: const TextStyle(color: Colors.white70, fontSize: 9)),
        ]),
        trailing: Text((c.confidence * 100).toInt().toString() + '%',
          style: TextStyle(color: c.confidence > 0.8 ? Colors.green : Colors.orange,
            fontSize: 11, fontWeight: FontWeight.bold))));
  }
}
''')
print("✅ analyzer_screen.dart — ПОЛНАЯ версия (2 вкладки Онлайн/Из лога, автоанализ каждые 5 сек, экспорт 4 формата)")



✅ log_graph_screen.dart — ПОЛНАЯ версия (тач-курсор, статистика, zoom, фильтры, совмещённый + отдельные графики)
✅ analyzer_screen.dart — ПОЛНАЯ версия (2 вкладки Онлайн/Из лога, автоанализ каждые 5 сек, экспорт 4 формата)


In [12]:
# @title 🔧 ПАТЧ: Полный analyzer_service.dart со всеми алгоритмами анализа
import os
os.chdir('/content/nissan_logger_pro_v5')

with open('lib/services/analyzer_service.dart', 'w') as f:
    f.write('''import 'dart:io';
import 'dart:math';
import 'package:csv/csv.dart';
import '../models/obd_data.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';

class AnalyzerService {
  static const int MIN_SAMPLES = 2;
  static const double MIN_CONFIDENCE = 0.4;

  // ============================================================
  // SPARK ADVANCE — полный анализ
  // ============================================================
  Future<AnalysisResult> analyzeSparkMap(List<OBDData> logData, TuningMap map) async {
    List<MapCell> changes = [];
    final valid = logData.where((d) => d.rpm > 400 && d.rpm < 7500 && d.engineLoad > 0).toList();
    if (valid.length < 10) {
      return AnalysisResult(mapName: 'Spark Advance', analyzedAt: DateTime.now(),
        totalSamples: logData.length, changes: [],
        summary: 'Мало данных: ' + valid.length.toString());
    }
    Map<String, List<OBDData>> cellData = _groupByCells(valid, map);
    for (var entry in cellData.entries) {
      final samples = entry.value;
      if (samples.length < MIN_SAMPLES) continue;
      final parts = entry.key.split(',');
      int rpmIdx = int.parse(parts[0]);
      int loadIdx = int.parse(parts[1]);
      double currentValue = map.data[rpmIdx][loadIdx];
      double avgTiming = _average(samples.map((d) => d.actualIgnition));
      double avgKnock = _average(samples.map((d) => d.knockRetard));
      double avgAFR = _average(samples.map((d) => d.afr));
      double suggested = currentValue;
      String reason = '';
      double confidence = 0;
      if (avgKnock > 2.0) {
        suggested = currentValue - min(avgKnock, 3.0);
        reason = 'Детонация ' + avgKnock.toStringAsFixed(1) + '°';
        confidence = 0.9;
      } else if (avgKnock > 0.5) {
        suggested = currentValue - 1;
        reason = 'Лёгкая детонация';
        confidence = 0.7;
      } else if (avgTiming != 0 && (avgTiming - currentValue).abs() > 2) {
        suggested = avgTiming;
        reason = 'ЭБУ ставит ' + avgTiming.toStringAsFixed(1) + '°';
        confidence = 0.6;
      } else if (avgKnock < 0.1 && avgAFR > 13.0 && avgAFR < 14.5) {
        suggested = currentValue + 1.0;
        reason = 'Стабильно, +1° УОЗ';
        confidence = 0.5;
      }
      suggested = suggested.clamp(-5, 45);
      if ((suggested - currentValue).abs() >= 0.5 && confidence >= MIN_CONFIDENCE) {
        changes.add(MapCell(rpmIndex: rpmIdx, loadIndex: loadIdx,
          rpm: map.rpmAxis[rpmIdx], load: map.loadAxis[loadIdx],
          currentValue: currentValue, suggestedValue: suggested,
          confidence: confidence, sampleCount: samples.length, reason: reason));
      }
    }
    return AnalysisResult(mapName: 'Spark Advance', analyzedAt: DateTime.now(),
      totalSamples: logData.length, changes: changes,
      summary: _summary('Spark', changes, valid.length, cellData.length));
  }

  // ============================================================
  // FUEL MAP / VE — полный анализ
  // ============================================================
  Future<AnalysisResult> analyzeFuelMap(List<OBDData> logData, TuningMap map) async {
    List<MapCell> changes = [];
    final valid = logData.where((d) =>
      d.rpm > 400 && d.rpm < 7500 && d.engineLoad > 0 &&
      d.longFuelTrim.abs() < 30).toList();
    if (valid.length < 10) {
      return AnalysisResult(mapName: 'Fuel Map / VE', analyzedAt: DateTime.now(),
        totalSamples: logData.length, changes: [],
        summary: 'Мало данных: ' + valid.length.toString());
    }
    Map<String, List<OBDData>> cellData = _groupByCells(valid, map);
    for (var entry in cellData.entries) {
      final samples = entry.value;
      if (samples.length < MIN_SAMPLES) continue;
      final parts = entry.key.split(',');
      int rpmIdx = int.parse(parts[0]);
      int loadIdx = int.parse(parts[1]);
      double currentValue = map.data[rpmIdx][loadIdx];
      double avgSTFT = _average(samples.map((d) => d.shortFuelTrim));
      double avgLTFT = _average(samples.map((d) => d.longFuelTrim));
      double totalTrim = avgSTFT + avgLTFT;
      double avgAFR = _average(samples.map((d) => d.afr));
      double suggested = currentValue;
      String reason = '';
      double confidence = 0;
      if (totalTrim > 3) {
        suggested = currentValue * (1 + totalTrim / 100.0);
        reason = 'Trim +' + totalTrim.toStringAsFixed(1) + '% (бедно)';
        confidence = min(0.9, totalTrim.abs() / 10);
      } else if (totalTrim < -3) {
        suggested = currentValue * (1 + totalTrim / 100.0);
        reason = 'Trim ' + totalTrim.toStringAsFixed(1) + '% (богато)';
        confidence = min(0.9, totalTrim.abs() / 10);
      } else if (avgAFR > 15.5 && samples.first.engineLoad > 50) {
        suggested = currentValue * 1.05;
        reason = 'AFR ' + avgAFR.toStringAsFixed(1) + ' бедно';
        confidence = 0.6;
      } else if (avgAFR < 12.0 && samples.first.engineLoad > 50) {
        suggested = currentValue * 0.95;
        reason = 'AFR ' + avgAFR.toStringAsFixed(1) + ' богато';
        confidence = 0.6;
      }
      double maxChange = currentValue * 15 / 100;
      double delta = suggested - currentValue;
      if (delta.abs() > maxChange) {
        suggested = currentValue + (delta > 0 ? maxChange : -maxChange);
      }
      double changePercent = currentValue > 0
          ? (suggested - currentValue).abs() / currentValue * 100 : 0;
      if (changePercent >= 1.0 && confidence >= MIN_CONFIDENCE) {
        changes.add(MapCell(rpmIndex: rpmIdx, loadIndex: loadIdx,
          rpm: map.rpmAxis[rpmIdx], load: map.loadAxis[loadIdx],
          currentValue: currentValue, suggestedValue: suggested,
          confidence: confidence, sampleCount: samples.length, reason: reason));
      }
    }
    return AnalysisResult(mapName: 'Fuel Map / VE', analyzedAt: DateTime.now(),
      totalSamples: logData.length, changes: changes,
      summary: _summary('Fuel', changes, valid.length, cellData.length));
  }

  // ============================================================
  // VTC — полный анализ
  // ============================================================
  Future<AnalysisResult> analyzeVTCMap(List<OBDData> logData, TuningMap map) async {
    List<MapCell> changes = [];
    final valid = logData.where((d) => d.rpm > 800 && d.rpm < 7000 && d.engineLoad > 5).toList();
    if (valid.length < 10) {
      return AnalysisResult(mapName: 'VTC Map', analyzedAt: DateTime.now(),
        totalSamples: logData.length, changes: [],
        summary: 'Мало данных');
    }
    Map<String, List<OBDData>> cellData = _groupByCells(valid, map);
    for (var entry in cellData.entries) {
      final samples = entry.value;
      if (samples.length < MIN_SAMPLES) continue;
      final parts = entry.key.split(',');
      int rpmIdx = int.parse(parts[0]);
      int loadIdx = int.parse(parts[1]);
      double currentValue = map.data[rpmIdx][loadIdx];
      double avgActual = _average(samples.map((d) => d.vtcActualAngle));
      double avgKnock = _average(samples.map((d) => d.knockRetard));
      double suggested = currentValue;
      String reason = '';
      double confidence = 0;
      if ((avgActual - currentValue).abs() > 3) {
        suggested = avgActual;
        reason = 'ЭБУ ставит ' + avgActual.toStringAsFixed(1) + '°';
        confidence = 0.7;
      } else if (avgKnock > 1.0 && currentValue > 15) {
        suggested = max(0, currentValue - 5);
        reason = 'Детонация - уменьшить VTC';
        confidence = 0.75;
      }
      suggested = suggested.clamp(0, 45);
      if ((suggested - currentValue).abs() >= 2 && confidence >= MIN_CONFIDENCE) {
        changes.add(MapCell(rpmIndex: rpmIdx, loadIndex: loadIdx,
          rpm: map.rpmAxis[rpmIdx], load: map.loadAxis[loadIdx],
          currentValue: currentValue, suggestedValue: suggested,
          confidence: confidence, sampleCount: samples.length, reason: reason));
      }
    }
    return AnalysisResult(mapName: 'VTC Map', analyzedAt: DateTime.now(),
      totalSamples: logData.length, changes: changes,
      summary: _summary('VTC', changes, valid.length, cellData.length));
  }

  // ============================================================
  // TORQUE — анализ по доступным данным
  // ============================================================
  Future<AnalysisResult> analyzeTorqueMap(List<OBDData> logData, TuningMap map) async {
    List<MapCell> changes = [];
    final valid = logData.where((d) =>
      d.rpm > 400 && d.rpm < 7500 && d.engineLoad > 0).toList();
    if (valid.length < 10) {
      return AnalysisResult(mapName: 'Engine Torque', analyzedAt: DateTime.now(),
        totalSamples: logData.length, changes: [],
        summary: 'Мало данных: ' + valid.length.toString());
    }
    Map<String, List<OBDData>> cellData = _groupByCells(valid, map);
    for (var entry in cellData.entries) {
      final samples = entry.value;
      if (samples.length < MIN_SAMPLES) continue;
      final parts = entry.key.split(',');
      int rpmIdx = int.parse(parts[0]);
      int loadIdx = int.parse(parts[1]);
      double currentValue = map.data[rpmIdx][loadIdx];
      double avgCalcTorque = _average(samples.map((d) => d.calculatedTorque));
      double avgKnock = _average(samples.map((d) => d.knockRetard));
      double avgLoad = _average(samples.map((d) => d.engineLoad));
      double suggested = currentValue;
      String reason = '';
      double confidence = 0;
      // Если расчётный момент значительно отличается от табличного
      if (avgCalcTorque > 0 && (avgCalcTorque - currentValue).abs() > currentValue * 0.15) {
        suggested = avgCalcTorque;
        reason = 'Расчётный момент ' + avgCalcTorque.toStringAsFixed(1) + ' Нм';
        confidence = 0.5;
      }
      // Если есть детонация — момент завышен
      if (avgKnock > 2.0 && currentValue > 50) {
        suggested = currentValue * 0.9;
        reason = 'Детонация ' + avgKnock.toStringAsFixed(1) + '° - снизить момент';
        confidence = 0.7;
      }
      if ((suggested - currentValue).abs() >= 5 && confidence >= MIN_CONFIDENCE) {
        changes.add(MapCell(rpmIndex: rpmIdx, loadIndex: loadIdx,
          rpm: map.rpmAxis[rpmIdx], load: map.loadAxis[loadIdx],
          currentValue: currentValue, suggestedValue: suggested,
          confidence: confidence, sampleCount: samples.length, reason: reason));
      }
    }
    return AnalysisResult(mapName: 'Engine Torque', analyzedAt: DateTime.now(),
      totalSamples: logData.length, changes: changes,
      summary: _summary('Torque', changes, valid.length, cellData.length));
  }

  // ============================================================
  // Вспомогательные методы
  // ============================================================

  Map<String, List<OBDData>> _groupByCells(List<OBDData> data, TuningMap map) {
    Map<String, List<OBDData>> result = {};
    for (var d in data) {
      String key = _getCellKey(d.rpm.toDouble(), d.engineLoad, map);
      result.putIfAbsent(key, () => []).add(d);
    }
    return result;
  }

  String _getCellKey(double rpm, double load, TuningMap map) {
    int rpmIdx = _findClosestIndex(map.rpmAxis, rpm);
    int loadIdx = _findClosestIndex(map.loadAxis, load);
    return rpmIdx.toString() + ',' + loadIdx.toString();
  }

  int _findClosestIndex(List<double> axis, double value) {
    int idx = 0;
    double minDiff = double.infinity;
    for (int i = 0; i < axis.length; i++) {
      double diff = (axis[i] - value).abs();
      if (diff < minDiff) { minDiff = diff; idx = i; }
    }
    return idx;
  }

  double _average(Iterable<num> values) {
    if (values.isEmpty) return 0;
    return values.reduce((a, b) => a + b) / values.length;
  }

  String _summary(String type, List<MapCell> changes, int total, int cells) {
    if (changes.isEmpty) {
      return type + ' карта работает оптимально\\n' +
             'Проанализировано: ' + total.toString() + '\\n' +
             'Клеток: ' + cells.toString();
    }
    return 'Данных: ' + total.toString() + '\\n' +
           'Клеток: ' + cells.toString() + '\\n' +
           'Правок: ' + changes.length.toString();
  }

  Future<List<OBDData>> loadLogFromCSV(String path) async {
    final file = File(path);
    final content = await file.readAsString();
    final rows = const CsvToListConverter().convert(content);
    if (rows.isEmpty) return [];
    List<OBDData> data = [];
    for (int i = 1; i < rows.length; i++) {
      try {
        final row = rows[i];
        data.add(OBDData(
          timestamp: DateTime.fromMillisecondsSinceEpoch(row[0] as int),
          rpm: row[1] as int, speed: row[2] as int,
          engineLoad: double.tryParse(row[3].toString()) ?? 0,
          coolantTemp: row[4] as int, intakeTemp: row[5] as int,
          maf: double.tryParse(row[6].toString()) ?? 0,
          throttlePos: double.tryParse(row[7].toString()) ?? 0,
          ignitionTiming: double.tryParse(row[8].toString()) ?? 0,
          shortFuelTrim: double.tryParse(row[9].toString()) ?? 0,
          longFuelTrim: double.tryParse(row[10].toString()) ?? 0,
          o2Voltage: double.tryParse(row[11].toString()) ?? 0,
          afr: double.tryParse(row[12].toString()) ?? 14.7,
          vtcActualAngle: double.tryParse(row[14].toString()) ?? 0,
          knockRetard: double.tryParse(row[15].toString()) ?? 0,
          actualIgnition: double.tryParse(row[17].toString()) ?? 0,
        ));
      } catch (e) { continue; }
    }
    return data;
  }
}
''')

print("✅ analyzer_service.dart — ПОЛНАЯ версия")
print()
print("Что восстановлено:")
print("  ✅ analyzeSparkMap — анализ зажигания (был рабочий)")
print("  ✅ analyzeFuelMap — анализ топлива/VE по STFT/LTFT/AFR")
print("  ✅ analyzeVTCMap — анализ VTC по фактическому углу и детонации")
print("  ✅ analyzeTorqueMap — анализ момента по расчётному и детонации")
print()
print("Алгоритмы анализа:")
print("  Fuel/VE:")
print("    • STFT+LTFT > 3% → увеличить VE (бедная смесь)")
print("    • STFT+LTFT < -3% → уменьшить VE (богатая смесь)")
print("    • AFR > 15.5 при нагрузке > 50% → увеличить VE")
print("    • AFR < 12.0 при нагрузке > 50% → уменьшить VE")
print("    • Макс изменение 15% за раз")
print()
print("  VTC:")
print("    • Факт VTC отличается от карты > 3° → подстроить")
print("    • Детонация > 1° при VTC > 15° → уменьшить VTC")
print()
print("  Torque:")
print("    • Расчётный момент отличается > 15% → подстроить")
print("    • Детонация > 2° при момент > 50 Нм → снизить на 10%")

print()
print("=" * 60)
print("🔨 ПЕРЕСБОРКА")
print("=" * 60)

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ.get('PATH', '')
os.environ['PATH'] = '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:' + os.environ['PATH']
os.environ['PUB_CACHE'] = '/content/.pub-cache'
os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['PATH'] = '/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:' + os.environ['PATH']
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/ninja'

!rm -rf /content/nissan_logger_pro_v5/build

result = !JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64 CMAKE_MAKE_PROGRAM=/usr/bin/ninja flutter build apk --release --no-tree-shake-icons --android-skip-build-dependency-validation 2>&1

important = [l for l in result if any(k in l.lower() for k in ['error', 'failed', 'built', 'app-release.apk', 'exception'])]
print("\n📋 Ключевые события:")
for line in important[-20:]:
    print(line)

apk_path = '/content/nissan_logger_pro_v5/build/app/outputs/flutter-apk/app-release.apk'
if os.path.exists(apk_path):
    size_mb = os.path.getsize(apk_path) / (1024 * 1024)
    print(f"\n🎉 APK СОБРАН! {size_mb:.1f} MB")
    from google.colab import files
    !cp {apk_path} /content/NissanLoggerPro_v5_analyzer_full.apk
    files.download('/content/NissanLoggerPro_v5_analyzer_full.apk')
else:
    print("\n❌ Не собран, последние 30 строк:")
    for line in result[-30:]:
        print(line)

✅ analyzer_service.dart — ПОЛНАЯ версия

Что восстановлено:
  ✅ analyzeSparkMap — анализ зажигания (был рабочий)
  ✅ analyzeFuelMap — анализ топлива/VE по STFT/LTFT/AFR
  ✅ analyzeVTCMap — анализ VTC по фактическому углу и детонации
  ✅ analyzeTorqueMap — анализ момента по расчётному и детонации

Алгоритмы анализа:
  Fuel/VE:
    • STFT+LTFT > 3% → увеличить VE (бедная смесь)
    • STFT+LTFT < -3% → уменьшить VE (богатая смесь)
    • AFR > 15.5 при нагрузке > 50% → увеличить VE
    • AFR < 12.0 при нагрузке > 50% → уменьшить VE
    • Макс изменение 15% за раз

  VTC:
    • Факт VTC отличается от карты > 3° → подстроить
    • Детонация > 1° при VTC > 15° → уменьшить VTC

  Torque:
    • Расчётный момент отличается > 15% → подстроить
    • Детонация > 2° при момент > 50 Нм → снизить на 10%

🔨 ПЕРЕСБОРКА

📋 Ключевые события:
✓ Built build/app/outputs/flutter-apk/app-release.apk (59.3MB)

🎉 APK СОБРАН! 56.6 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>